# CS781 Phase 7 (PCR) - Notebook 2 (Kaggle account 2)

* **GPU 0 - Run B:** efficient + **LORMA+** (rank 32 on q / v / FFN-down, backbone frozen) - the compulsory LORMA run.
* **GPU 1 - Run F (extra):** efficient + **IA3** (learned rescaling of k / v / FFN-up).
* **Ensemble B + F.**

**Setup:** Accelerator **GPU T4 x2**, Internet **on**, secret `HF` enabled, Phase 1 artifact dataset attached with **phase1_improved_efficient_artifacts.zip**. Then **Save Version -> Save & Run All (Commit)** (~2 h 45 min); when it finishes open the version -> Output -> `SUBMIT/`.

Data: only the official Phase 7 files from `Exploration-Lab/CS-781-Capstone` (re-downloaded with `force_download`). Each split's queries are ranked only against that split's candidate pool. Train/dev queries duplicating a test query (by id or normalised text) are removed (`LEAKAGE CHECK`); the query itself and exact duplicates of it are never returned. The dense retriever trains on train queries only; dev is used for checkpoint selection and to fit the fusion ranker, which is scored by 5-fold cross-fitting on dev (the honest number). No pretrained weights other than the Phase 1 checkpoints are used.


In [ ]:
!nvidia-smi -L
!pip install -q tokenizers huggingface_hub lightgbm
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF')
print('HF token loaded; value intentionally hidden')
found = [os.path.join(f, n) for f, _, ns in os.walk('/kaggle/input', followlinks=True) for n in ns if n.endswith(('.zip', '.pt'))]
print('\n'.join(found))
assert found, 'NOTHING attached under /kaggle/input - Add Input -> your Phase 1 artifact dataset'


In [ ]:
%%writefile /kaggle/working/phase1_pipeline.py
"""Phase 1 training, generation, validation, and submission pipeline."""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


SEED = 230338
SPECIAL = ["<pad>", "<bos>", "<eos>", "<unk>"]
MODEL_NAMES = ("decoder", "encoder_decoder", "hybrid", "efficient")
TRAIN_FILES = (
    "train_acts.json",
    "train_high_courts.json",
    "train_supreme_court.json",
    "train_tribunals.json",
)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class RunConfig:
    vocab_size: int = 4096
    d_model: int = 512
    n_heads: int = 8
    seq_len: int = 256
    batch_size: int = 2
    grad_accum: int = 4
    steps: int = 400
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    train_chars: int = 40_000_000
    train_tokens: int = 12_000_000
    tokenizer_chars: int = 10_000_000
    generation_batch_size: int = 8
    max_generation_tokens: int = 320
    target_words: int = 128


class TokenBlocks(torch.utils.data.Dataset):
    def __init__(self, ids: list[int], block: int, samples: int, seed: int):
        if len(ids) <= block:
            raise ValueError(f"Need more than {block} tokens; received {len(ids)}")
        self.ids = torch.tensor(ids, dtype=torch.long)
        generator = torch.Generator().manual_seed(seed)
        self.starts = torch.randint(0, len(ids) - block, (samples,), generator=generator)
        self.block = block

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, index: int) -> torch.Tensor:
        start = int(self.starts[index])
        return self.ids[start : start + self.block]


def make_generation_validation(texts: list[str], examples: int = 8):
    """Hold out complete documents for the official 512-word to 128-word task."""
    training_texts, prompts, truth = [], {}, {}
    for text in texts:
        words = text.split()
        if len(truth) < examples and len(words) >= 640:
            example_id = f"heldout_{len(truth) + 1}"
            prompts[example_id] = {"text": " ".join(words[:512])}
            truth[example_id] = {"text": " ".join(words[512:640])}
        else:
            training_texts.append(text)
    if len(truth) < examples:
        raise RuntimeError(f"Could construct only {len(truth)} held-out continuation examples")
    return training_texts, prompts, truth


def exclude_generation_validation(texts: list[str], prompts: dict, truth: dict) -> list[str]:
    signatures = {
        example_id: f"{prompts[example_id]['text']} {truth[example_id]['text']}"
        for example_id in truth
    }
    filtered = [
        text for text in texts
        if " ".join(text.split()[:640]) not in signatures.values()
    ]
    if len(filtered) != len(texts) - len(truth):
        raise RuntimeError("Could not exclude every reused held-out document from training")
    return filtered


def train_tokenizer(texts: list[str], path: Path, vocab_size: int):
    """Train a new byte-level BPE on only the selected training corpus."""
    from tokenizers import Tokenizer, decoders, models, normalizers, pre_tokenizers, trainers

    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=SPECIAL, min_frequency=2)
    tokenizer.train_from_iterator(texts, trainer=trainer)
    tokenizer.save(str(path))
    return tokenizer


def load_tokenizer(path: Path):
    from tokenizers import Tokenizer

    return Tokenizer.from_file(str(path))


def collect_training_text(config: RunConfig) -> list[str]:
    """Stream a balanced slice from the four official Phase 1 sources."""
    import ijson
    import requests

    texts, chars = [], 0
    quota = config.train_chars // len(TRAIN_FILES)
    headers = {}
    if os.environ.get("HF_TOKEN"):
        headers["Authorization"] = f"Bearer {os.environ['HF_TOKEN']}"
    for filename in TRAIN_FILES:
        url = (
            "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
            f"Phase_1_Pretraining/Train/{filename}"
        )
        source_chars = 0
        with requests.get(url, headers=headers, stream=True, timeout=(30, 900)) as response:
            response.raise_for_status()
            response.raw.decode_content = True
            for record in ijson.items(response.raw, "item"):
                text = record.get("text", "")
                if not isinstance(text, str) or not text.strip():
                    continue
                remaining = quota - source_chars
                texts.append(text[:remaining])
                taken = min(len(text), remaining)
                chars += taken
                source_chars += taken
                if source_chars >= quota:
                    break
        print(f"{filename}: {source_chars:,} characters")
    if chars < 100_000:
        raise RuntimeError("Official training stream returned too little text")
    print(f"Collected {chars:,} training characters in {len(texts):,} records")
    return texts


def tokenize_training_text(texts: list[str], tokenizer, config: RunConfig) -> list[int]:
    eos = tokenizer.token_to_id("<eos>")
    ids: list[int] = []
    for text in texts:
        ids.extend(tokenizer.encode(text).ids)
        ids.append(eos)
        if len(ids) >= config.train_tokens:
            break
    print(f"Prepared {len(ids):,} training tokens")
    return ids[: config.train_tokens]


def load_test_data() -> dict[str, dict[str, str]]:
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename="Phase_1_Pretraining/Test/test.json",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    with open(path, encoding="utf-8") as stream:
        data = json.load(stream)
    if not isinstance(data, dict) or not data:
        raise ValueError("Official test input is not a non-empty ID mapping")
    return data


def causal_mask(length: int, device: torch.device) -> torch.Tensor:
    return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)


def sinusoidal_positions(length: int, dim: int, device: torch.device) -> torch.Tensor:
    position = torch.arange(length, device=device).float().unsqueeze(1)
    scale = torch.exp(torch.arange(0, dim, 2, device=device).float() * (-math.log(10000.0) / dim))
    output = torch.zeros(length, dim, device=device)
    output[:, 0::2] = torch.sin(position * scale)
    output[:, 1::2] = torch.cos(position * scale)
    return output


class GatedContextFusion(nn.Module):
    """Inject a global or causal summary through a learned content-dependent gate."""

    def __init__(self, dim: int):
        super().__init__()
        self.context = nn.Linear(dim, dim, bias=False)
        self.gate = nn.Linear(2 * dim, dim)

    def forward(self, hidden: torch.Tensor, summary: torch.Tensor) -> torch.Tensor:
        gate = torch.sigmoid(self.gate(torch.cat((hidden, summary), dim=-1)))
        return hidden + gate * self.context(summary)


class DecoderOnlyLM(nn.Module):
    def __init__(self, cfg: RunConfig, layers: int = 16):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        layer = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def logits(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(ids) + sinusoidal_positions(ids.size(1), self.cfg.d_model, ids.device)
        x = self.blocks(x, mask=causal_mask(ids.size(1), ids.device))
        denominator = torch.arange(1, ids.size(1) + 1, device=ids.device).view(1, -1, 1)
        x = self.fusion(x, x.cumsum(dim=1) / denominator)
        return self.head(self.norm(x))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        return F.cross_entropy(self.logits(batch[:, :-1]).flatten(0, 1), batch[:, 1:].flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        context = torch.cat((source, generated), dim=1)[:, -self.cfg.seq_len :]
        return self.logits(context)[:, -1]


class EncoderDecoderLM(nn.Module):
    def __init__(self, cfg: RunConfig, encoder_layers: int = 7, decoder_layers: int = 7):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        encoder = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        decoder = nn.TransformerDecoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.encoder = nn.TransformerEncoder(encoder, encoder_layers, enable_nested_tensor=False)
        self.decoder = nn.TransformerDecoder(decoder, decoder_layers)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def encode(self, source: torch.Tensor) -> torch.Tensor:
        x = self.embedding(source) + sinusoidal_positions(source.size(1), self.cfg.d_model, source.device)
        return self.encoder(x)

    def decode(self, target: torch.Tensor, memory: torch.Tensor) -> torch.Tensor:
        y = self.embedding(target) + sinusoidal_positions(target.size(1), self.cfg.d_model, target.device)
        y = self.decoder(y, memory, tgt_mask=causal_mask(target.size(1), target.device))
        source_summary = memory.mean(dim=1, keepdim=True).expand(-1, y.size(1), -1)
        y = self.fusion(y, source_summary)
        return self.head(self.norm(y))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        split = batch.size(1) // 2
        source, target = batch[:, :split], batch[:, split:]
        decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), dim=1)
        return F.cross_entropy(self.decode(decoder_input, self.encode(source)).flatten(0, 1), target.flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        memory = self.encode(source[:, -self.cfg.seq_len :])
        target = generated[:, -self.cfg.seq_len :]
        if target.size(1) == 0:
            target = torch.full((source.size(0), 1), bos, dtype=torch.long, device=source.device)
        return self.decode(target, memory)[:, -1]


class HybridLM(EncoderDecoderLM):
    """Bidirectional recurrent encoder plus causal Transformer decoder."""

    def __init__(self, cfg: RunConfig, recurrent_layers: int = 5, decoder_layers: int = 11):
        nn.Module.__init__(self)
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.recurrent = nn.GRU(
            cfg.d_model, cfg.d_model // 2, recurrent_layers,
            batch_first=True, bidirectional=True, dropout=0.0,
        )
        decoder = nn.TransformerDecoderLayer(
            cfg.d_model, cfg.n_heads, 4 * cfg.d_model, batch_first=True,
            norm_first=True, activation="gelu", dropout=0.0,
        )
        self.decoder = nn.TransformerDecoder(decoder, decoder_layers)
        self.fusion = GatedContextFusion(cfg.d_model)
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def encode(self, source: torch.Tensor) -> torch.Tensor:
        x = self.embedding(source) + sinusoidal_positions(source.size(1), self.cfg.d_model, source.device)
        return self.recurrent(x)[0]


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope(x: torch.Tensor) -> torch.Tensor:
    # x: batch, heads, time, head_dim
    length, dim = x.size(-2), x.size(-1)
    position = torch.arange(length, device=x.device).float()
    frequency = torch.exp(torch.arange(0, dim, 2, device=x.device).float() * (-math.log(10000.0) / dim))
    angle = torch.outer(position, frequency)
    cos, sin = angle.cos()[None, None], angle.sin()[None, None]
    even, odd = x[..., 0::2], x[..., 1::2]
    return torch.stack((even * cos - odd * sin, even * sin + odd * cos), dim=-1).flatten(-2)


class GroupedQueryAttention(nn.Module):
    def __init__(self, dim: int, heads: int, kv_heads: int = 2, window: int = 64):
        super().__init__()
        self.heads, self.kv_heads, self.head_dim, self.window = heads, kv_heads, dim // heads, window
        self.q = nn.Linear(dim, heads * self.head_dim, bias=False)
        self.k = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.v = nn.Linear(dim, kv_heads * self.head_dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, length, _ = x.shape
        q = rope(self.q(x).view(batch, length, self.heads, self.head_dim).transpose(1, 2))
        k = rope(self.k(x).view(batch, length, self.kv_heads, self.head_dim).transpose(1, 2))
        v = self.v(x).view(batch, length, self.kv_heads, self.head_dim).transpose(1, 2)
        repeat = self.heads // self.kv_heads
        k, v = k.repeat_interleave(repeat, 1), v.repeat_interleave(repeat, 1)
        positions = torch.arange(length, device=x.device)
        distance = positions[:, None] - positions[None, :]
        if getattr(self, "bidirectional", False):
            allowed = distance.abs() < self.window
        else:
            allowed = (distance >= 0) & (distance < self.window)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        output = torch.matmul(scores.softmax(-1), v).transpose(1, 2).contiguous().view(batch, length, -1)
        return self.out(output)


class EfficientBlock(nn.Module):
    def __init__(self, cfg: RunConfig):
        super().__init__()
        hidden = int(8 * cfg.d_model / 3)
        hidden += (-hidden) % 64
        self.norm1, self.norm2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.attention = GroupedQueryAttention(cfg.d_model, cfg.n_heads)
        self.gate = nn.Linear(cfg.d_model, hidden, bias=False)
        self.up = nn.Linear(cfg.d_model, hidden, bias=False)
        self.down = nn.Linear(hidden, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attention(self.norm1(x))
        y = self.norm2(x)
        return x + self.down(F.silu(self.gate(y)) * self.up(y))


class EfficientLM(nn.Module):
    def __init__(self, cfg: RunConfig, layers: int = 19):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(EfficientBlock(cfg) for _ in range(layers))
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.embedding.weight

    def logits(self, ids: torch.Tensor) -> torch.Tensor:
        x = self.embedding(ids)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm(x))

    def loss(self, batch: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        return F.cross_entropy(self.logits(batch[:, :-1]).flatten(0, 1), batch[:, 1:].flatten())

    def next_logits(self, source: torch.Tensor, generated: torch.Tensor, bos: int) -> torch.Tensor:
        del bos
        context = torch.cat((source, generated), dim=1)[:, -self.cfg.seq_len :]
        return self.logits(context)[:, -1]


def build_model(name: str, cfg: RunConfig) -> nn.Module:
    builders = {
        "decoder": DecoderOnlyLM,
        "encoder_decoder": EncoderDecoderLM,
        "hybrid": HybridLM,
        "efficient": EfficientLM,
    }
    return builders[name](cfg)


def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


def evaluate_loss(model: nn.Module, loader, bos: int, device: torch.device, batches: int = 20) -> float:
    model.eval()
    losses = []
    with torch.inference_mode():
        for index, batch in enumerate(loader):
            if index >= batches:
                break
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                losses.append(float(model.loss(batch, bos)))
    model.train()
    return sum(losses) / len(losses)


def train_model(
    model: nn.Module,
    loader,
    validation_loader,
    cfg: RunConfig,
    bos: int,
    device: torch.device,
    recovery_path: Path,
    start_step: int = 0,
    prior_losses: list[float] | None = None,
) -> tuple[list[float], float]:
    model.to(device).train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    warmup = max(1, cfg.steps // 20)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lambda step: (step + 1) / warmup
        if step < warmup
        else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, cfg.steps - warmup))),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    losses, iterator = list(prior_losses or []), iter(loader)
    optimizer.zero_grad(set_to_none=True)
    for step in range(start_step, cfg.steps):
        running = 0.0
        for _ in range(cfg.grad_accum):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)
            batch = batch.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                loss = model.loss(batch, bos) / cfg.grad_accum
            scaler.scale(loss).backward()
            running += float(loss.detach())
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(running)
        if step == 0 or (step + 1) % 10 == 0:
            print(f"step {step + 1:>3}/{cfg.steps}: loss={running:.4f}")
        if (step + 1) % 25 == 0 or step + 1 == cfg.steps:
            temporary = recovery_path.with_suffix(".tmp")
            torch.save(
                {
                    "model": model.state_dict(),
                    "model_name": recovery_path.stem.removesuffix("_recovery"),
                    "completed_steps": step + 1,
                    "config": asdict(cfg),
                    "train_loss": losses,
                },
                temporary,
            )
            os.replace(temporary, recovery_path)
            print(f"recovery checkpoint saved: {recovery_path.name}")
    return losses, evaluate_loss(model, validation_loader, bos, device)


def sample_next(logits: torch.Tensor, temperature: float = 0.2, top_k: int = 20) -> torch.Tensor:
    logits = logits / temperature
    values, indices = torch.topk(logits, min(top_k, logits.size(-1)), dim=-1)
    sampled = torch.multinomial(values.softmax(-1), 1)
    return indices.gather(1, sampled)


@torch.inference_mode()
def generate_predictions(model, tokenizer, test_data, cfg: RunConfig, device: torch.device) -> dict:
    model.eval()
    pad, bos, eos = (tokenizer.token_to_id(token) for token in SPECIAL[:3])
    items, predictions = list(test_data.items()), {}
    for start in range(0, len(items), cfg.generation_batch_size):
        batch = items[start : start + cfg.generation_batch_size]
        encoded = [tokenizer.encode(record["text"]).ids[-cfg.seq_len :] for _, record in batch]
        width = max(map(len, encoded))
        source = torch.full((len(encoded), width), pad, dtype=torch.long, device=device)
        for row, ids in enumerate(encoded):
            source[row, -len(ids) :] = torch.tensor(ids, device=device)
        generated = torch.empty((len(encoded), 0), dtype=torch.long, device=device)
        finished_text = [None] * len(encoded)
        # Encoder-decoder models reuse their source representation at every step.
        memory = model.encode(source) if isinstance(model, EncoderDecoderLM) else None
        for _ in range(cfg.max_generation_tokens):
            if memory is None:
                next_logits = model.next_logits(source, generated, bos)
            else:
                target = generated[:, -cfg.seq_len :]
                if target.size(1) == 0:
                    target = torch.full((source.size(0), 1), bos, dtype=torch.long, device=device)
                next_logits = model.decode(target, memory)[:, -1]
            next_id = sample_next(next_logits)
            next_id[next_id.eq(eos)] = bos
            generated = torch.cat((generated, next_id), dim=1)
            if generated.size(1) % 16 == 0:
                for row in range(len(encoded)):
                    words = tokenizer.decode(generated[row].tolist(), skip_special_tokens=True).split()
                    if len(words) >= cfg.target_words:
                        finished_text[row] = " ".join(words[: cfg.target_words])
                if all(text is not None for text in finished_text):
                    break
        for row, (example_id, _) in enumerate(batch):
            if finished_text[row] is None:
                words = tokenizer.decode(generated[row].tolist(), skip_special_tokens=True).split()
                if not words:
                    words = ["the"]
                words = (words * math.ceil(cfg.target_words / len(words)))[: cfg.target_words]
                finished_text[row] = " ".join(words)
            predictions[str(example_id)] = {"text": finished_text[row]}
        print(f"generated {min(start + len(batch), len(items))}/{len(items)}")
    return predictions


def validate_and_zip(predictions: dict, test_data: dict, output_dir: Path, name: str) -> Path:
    expected, actual = {str(key) for key in test_data}, set(predictions)
    if expected != actual:
        raise ValueError(f"ID mismatch: missing={expected - actual}, extra={actual - expected}")
    for example_id, record in predictions.items():
        if set(record) != {"text"} or len(record["text"].split()) != 128:
            raise ValueError(f"Invalid prediction for {example_id}")
    folder = output_dir / name
    folder.mkdir(parents=True, exist_ok=True)
    prediction_path = folder / "prediction.json"
    prediction_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    zip_path = output_dir / f"submission_{name}.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as archive:
        if archive.namelist() != ["prediction.json"]:
            raise ValueError(f"Bad archive layout: {archive.namelist()}")
    return zip_path


def run(args) -> None:
    seed_everything()
    cfg = RunConfig(
        steps=args.steps,
        train_chars=args.train_chars,
        train_tokens=args.train_tokens,
        seq_len=args.seq_len,
        batch_size=args.batch_size,
        grad_accum=args.grad_accum,
    )
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    tokenizer_path = output_dir / "phase1_bpe.json"
    if args.generate_only:
        if args.model == "all" or not args.resume_checkpoint:
            raise ValueError("--generate-only requires one --model and --resume-checkpoint")
        source_tokenizer = Path(args.tokenizer_path) if args.tokenizer_path else tokenizer_path
        tokenizer = load_tokenizer(source_tokenizer)
        cfg.vocab_size = tokenizer.get_vocab_size()
        model = build_model(args.model, cfg)
        saved = torch.load(args.resume_checkpoint, map_location="cpu")
        model.load_state_dict(saved["model"])
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        test_data = load_test_data()
        predictions = generate_predictions(model, tokenizer, test_data, cfg, device)
        zip_path = validate_and_zip(predictions, test_data, output_dir, args.model)
        print(f"created {zip_path}")
        return
    texts = collect_training_text(cfg)
    random.Random(SEED).shuffle(texts)
    if args.validation_prompts and args.validation_truth:
        validation_prompts = json.loads(Path(args.validation_prompts).read_text(encoding="utf-8"))
        validation_truth = json.loads(Path(args.validation_truth).read_text(encoding="utf-8"))
        texts = exclude_generation_validation(texts, validation_prompts, validation_truth)
    elif args.validation_prompts or args.validation_truth:
        raise ValueError("Provide both --validation-prompts and --validation-truth")
    else:
        texts, validation_prompts, validation_truth = make_generation_validation(texts)
    validation_prompts_path = output_dir / "validation_prompts.json"
    validation_truth_path = output_dir / "validation_truth.json"
    validation_prompts_path.write_text(
        json.dumps(validation_prompts, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    validation_truth_path.write_text(
        json.dumps(validation_truth, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if args.tokenizer_path:
        tokenizer = load_tokenizer(Path(args.tokenizer_path))
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer_texts, used = [], 0
        for text in texts:
            tokenizer_texts.append(text)
            used += len(text)
            if used >= cfg.tokenizer_chars:
                break
        tokenizer = train_tokenizer(tokenizer_texts, tokenizer_path, cfg.vocab_size)
    cfg.vocab_size = tokenizer.get_vocab_size()
    token_ids = tokenize_training_text(texts, tokenizer, cfg)
    split = int(0.95 * len(token_ids))
    train_ids, validation_ids = token_ids[:split], token_ids[split:]
    samples = cfg.steps * cfg.grad_accum * cfg.batch_size
    dataset = TokenBlocks(train_ids, cfg.seq_len + 1, samples, SEED)
    loader = torch.utils.data.DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False, pin_memory=True)
    validation = TokenBlocks(validation_ids, cfg.seq_len + 1, 20 * cfg.batch_size, SEED + 1)
    validation_loader = torch.utils.data.DataLoader(
        validation, batch_size=cfg.batch_size, shuffle=False, pin_memory=True
    )
    test_data = load_test_data()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    bos = tokenizer.token_to_id("<bos>")
    manifest_path = output_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    if args.models:
        selected = tuple(name.strip() for name in args.models.split(",") if name.strip())
        invalid = set(selected) - set(MODEL_NAMES)
        if invalid or not selected:
            raise ValueError(f"Invalid --models selection: {sorted(invalid)}")
    else:
        selected = MODEL_NAMES if args.model == "all" else (args.model,)
    for name in selected:
        seed_everything(SEED + MODEL_NAMES.index(name))
        print(f"\n{'=' * 20} {name} {'=' * 20}")
        model = build_model(name, cfg)
        previous_steps = 0
        recovery_path = output_dir / f"{name}_recovery.pt"
        recovery_step = 0
        recovery_losses = []
        if args.resume_checkpoint:
            if len(selected) != 1:
                raise ValueError("--resume-checkpoint requires one selected model")
            saved = torch.load(args.resume_checkpoint, map_location="cpu")
            saved_name = saved.get("model_name")
            if saved_name and saved_name != name:
                raise ValueError(f"Checkpoint model is {saved_name}, not {name}")
            model.load_state_dict(saved["model"])
            previous_steps = int(saved.get("total_steps", saved.get("config", {}).get("steps", 0)))
            print(f"resumed {name} after {previous_steps:,} prior steps")
        elif recovery_path.exists():
            saved = torch.load(recovery_path, map_location="cpu")
            model.load_state_dict(saved["model"])
            recovery_step = int(saved["completed_steps"])
            recovery_losses = list(saved.get("train_loss", []))
            print(f"automatically resumed {name} at step {recovery_step}/{cfg.steps}")
        parameters = parameter_count(model)
        if not 50_000_000 <= parameters <= 300_000_000:
            raise ValueError(f"{name} has {parameters:,} parameters; must be 50M-300M")
        print(f"parameters: {parameters:,}")
        started = time.perf_counter()
        losses, validation_loss = train_model(
            model,
            loader,
            validation_loader,
            cfg,
            bos,
            device,
            recovery_path,
            recovery_step,
            recovery_losses,
        )
        elapsed = time.perf_counter() - started
        log_path = output_dir / f"training_log_{name}.json"
        log_path.write_text(
            json.dumps(
                {
                    "config": asdict(cfg),
                    "train_loss": losses,
                    "validation_loss": validation_loss,
                    "training_seconds": elapsed,
                },
                indent=2,
            ),
            encoding="utf-8",
        )
        checkpoint = output_dir / f"{name}.pt"
        torch.save(
            {
                "model": model.state_dict(),
                "config": asdict(cfg),
                "parameters": parameters,
                "model_name": name,
                "total_steps": previous_steps + cfg.steps,
                "train_loss": losses,
                "validation_loss": validation_loss,
            },
            checkpoint,
        )
        validation_predictions = generate_predictions(
            model, tokenizer, validation_prompts, cfg, device
        )
        validation_prediction_path = output_dir / f"validation_prediction_{name}.json"
        validation_prediction_path.write_text(
            json.dumps(validation_predictions, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        predictions = generate_predictions(model, tokenizer, test_data, cfg, device)
        zip_path = validate_and_zip(predictions, test_data, output_dir, name)
        manifest[name] = {
            "category": {
                "decoder": "decoder-only",
                "encoder_decoder": "encoder-decoder",
                "hybrid": "encoder/hybrid",
                "efficient": "efficient/lightweight",
            }[name],
            "parameters": parameters,
            "steps_this_run": cfg.steps,
            "total_steps": previous_steps + cfg.steps,
            "final_loss": losses[-1],
            "validation_loss": validation_loss,
            "validation_perplexity": math.exp(min(validation_loss, 20.0)),
            "training_seconds": elapsed,
            "device": str(device),
            "checkpoint": str(checkpoint),
            "training_log": str(log_path),
            "validation_truth": str(validation_truth_path),
            "validation_prompts": str(validation_prompts_path),
            "validation_prediction": str(validation_prediction_path),
            "submission": str(zip_path),
        }
        manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        recovery_path.unlink(missing_ok=True)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    print(json.dumps(manifest, indent=2))


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=("all",) + MODEL_NAMES, default="all")
    parser.add_argument("--models", help="Comma-separated subset for multi-GPU orchestration")
    parser.add_argument("--steps", type=int, default=400)
    parser.add_argument("--train-chars", type=int, default=40_000_000)
    parser.add_argument("--train-tokens", type=int, default=12_000_000)
    parser.add_argument("--seq-len", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--tokenizer-path")
    parser.add_argument("--resume-checkpoint")
    parser.add_argument("--generate-only", action="store_true")
    parser.add_argument("--validation-prompts")
    parser.add_argument("--validation-truth")
    parser.add_argument("--output-dir", default="/kaggle/working/phase1")
    return parser.parse_args()


if __name__ == "__main__":
    run(parse_args())


In [ ]:
%%writefile /kaggle/working/phase1_foundation.py
"""Train the Phase-1 flagship legal encoder-decoder model from scratch.

This file deliberately reuses only the proven corpus/test download helpers from
phase1_pipeline.py.  The tokenizer, model, objective, checkpointing, validation,
generation, and submission path are implemented here.
"""

from __future__ import annotations

import argparse
import bisect
import contextlib
import json
import math
import os
import random
import time
from array import array
from dataclasses import asdict, dataclass
from datetime import timedelta
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

import phase1_pipeline as legacy


SEED = 781
SPECIAL = ("<pad>", "<bos>", "<eos>", "<unk>", "<mask>")


def init_distributed():
    distributed = "RANK" in os.environ and "WORLD_SIZE" in os.environ
    if not distributed:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        return False, 0, 1, 0, device
    if not torch.cuda.is_available():
        raise RuntimeError("Distributed training requires CUDA")
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    local_rank = int(os.environ["LOCAL_RANK"])
    if world_size > torch.cuda.device_count():
        raise RuntimeError(f"Requested {world_size} ranks but found {torch.cuda.device_count()} GPUs")
    torch.cuda.set_device(local_rank)
    dist.init_process_group("nccl", timeout=timedelta(minutes=30))
    return True, rank, world_size, local_rank, torch.device("cuda", local_rank)


def distributed_barrier(distributed):
    if distributed:
        dist.barrier()


def unwrap_model(model):
    return model.module if isinstance(model, DDP) else model


@dataclass
class Config:
    vocab_size: int = 24_000
    d_model: int = 640
    heads: int = 10
    encoder_layers: int = 12
    decoder_layers: int = 8
    d_ff: int = 1_728
    source_len: int = 1_024
    target_len: int = 256
    global_block: int = 32
    dropout: float = 0.1
    steps: int = 60_000
    batch_size: int = 1
    grad_accum: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_fraction: float = 0.02
    train_chars: int = 500_000_000
    token_pool: int = 100_000_000
    tokenizer_chars: int = 80_000_000
    validation_examples: int = 256
    validation_batches: int = 24
    checkpoint_every: int = 100
    mlm_probability: float = 0.25
    mlm_mask_fraction: float = 0.15
    mlm_weight: float = 0.35
    generation_batch_size: int = 8
    max_generation_tokens: int = 384
    target_words: int = 128
    use_global_memory: bool = True


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_tokenizer(texts: list[str], path: Path, cfg: Config) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=cfg.vocab_size,
        min_frequency=2,
        special_tokens=list(SPECIAL),
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )
    used = 0

    def iterator():
        nonlocal used
        for text in texts:
            if used >= cfg.tokenizer_chars:
                break
            used += len(text)
            yield text

    tokenizer.train_from_iterator(iterator(), trainer=trainer)
    tokenizer.save(str(path))
    return tokenizer


def make_validation(texts: list[str], count: int) -> tuple[list[str], list[str], dict, dict]:
    rng = random.Random(SEED + 11)
    order = list(range(len(texts)))
    rng.shuffle(order)
    used: set[int] = set()
    prompts, truth = {}, {}
    for index in order:
        words = texts[index].split()
        if len(words) < 640:
            continue
        start = rng.randrange(len(words) - 640 + 1)
        key = f"validation_{len(prompts):04d}"
        prompts[key] = {"text": " ".join(words[start : start + 512])}
        truth[key] = {"text": " ".join(words[start + 512 : start + 640])}
        used.add(index)
        if len(prompts) == count:
            break
    if len(prompts) < count:
        raise RuntimeError(f"Only {len(prompts)} documents could supply validation examples")
    training = [text for i, text in enumerate(texts) if i not in used]
    validation = [text for i, text in enumerate(texts) if i in used]
    return training, validation, prompts, truth


def tokenize_documents(
    texts: list[str],
    tokenizer: Tokenizer,
    cfg: Config,
    source_len: int | None = None,
    target_len: int | None = None,
):
    eos = tokenizer.token_to_id("<eos>")
    flat, spans = array("I"), []
    block = (source_len or cfg.source_len) + (target_len or cfg.target_len)
    for text in texts:
        ids = tokenizer.encode(text).ids
        if len(ids) < block:
            continue
        remaining = cfg.token_pool - len(flat)
        if remaining <= 0:
            break
        ids = ids[:remaining]
        if len(ids) < block:
            break
        start = len(flat)
        flat.extend(ids)
        flat.append(eos)
        spans.append((start, start + len(ids)))
    if not spans:
        raise RuntimeError("No documents were long enough for the configured source/target lengths")
    return flat, spans


class ContinuationDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        tokens,
        spans,
        source_len,
        target_len,
        samples,
        seed,
        sample_offset=0,
        rank=0,
        world_size=1,
        samples_per_step=1,
    ):
        self.tokens, self.spans = tokens, spans
        self.source_len, self.target_len = source_len, target_len
        self.samples, self.seed, self.sample_offset = samples, seed, sample_offset
        self.rank, self.world_size = rank, world_size
        self.samples_per_step = samples_per_step
        block = source_len + target_len
        self.cumulative, total = [], 0
        for start, end in spans:
            total += end - start - block + 1
            self.cumulative.append(total)
        self.total = total

    def __len__(self):
        return self.samples

    def __getitem__(self, index):
        step, within_step = divmod(index, self.samples_per_step)
        global_index = (
            self.sample_offset
            + step * self.samples_per_step * self.world_size
            + self.rank * self.samples_per_step
            + within_step
        )
        pick = random.Random(self.seed + global_index).randrange(self.total)
        doc = bisect.bisect_right(self.cumulative, pick)
        previous = self.cumulative[doc - 1] if doc else 0
        start = self.spans[doc][0] + pick - previous
        stop = start + self.source_len + self.target_len
        values = torch.tensor(self.tokens[start:stop], dtype=torch.long)
        return values[: self.source_len], values[self.source_len :]


class RMSNorm(nn.Module):
    def __init__(self, width: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(width))
        self.eps = eps

    def forward(self, x):
        scale = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt().to(x.dtype)
        return x * scale * self.weight


def rotate_half(x):
    left, right = x.chunk(2, dim=-1)
    return torch.cat((-right, left), dim=-1)


def apply_rope(q, k):
    length, width = q.size(-2), q.size(-1)
    positions = torch.arange(length, device=q.device, dtype=torch.float32)
    inverse = 1.0 / (10_000 ** (torch.arange(0, width, 2, device=q.device).float() / width))
    angles = torch.outer(positions, inverse)
    angles = torch.cat((angles, angles), dim=-1)[None, None]
    cos, sin = angles.cos().to(q.dtype), angles.sin().to(q.dtype)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


class Attention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.heads, self.head_dim = cfg.heads, cfg.d_model // cfg.heads
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, query, memory=None, key_mask=None, causal=False, rope=False):
        memory = query if memory is None else memory
        batch, query_len, _ = query.shape
        key_len = memory.size(1)

        def split(projection, values, length):
            return projection(values).view(batch, length, self.heads, self.head_dim).transpose(1, 2)

        q = split(self.q, query, query_len)
        k = split(self.k, memory, key_len)
        v = split(self.v, memory, key_len)
        if rope:
            q, k = apply_rope(q, k)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        if causal and mask is not None:
            causal_mask = torch.ones(query_len, key_len, device=query.device, dtype=torch.bool).tril()
            mask = mask & causal_mask[None, None]
            causal = False
        result = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=causal,
        )
        result = result.transpose(1, 2).contiguous().view(batch, query_len, -1)
        return self.out(result)


class SwiGLU(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.value = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.out = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x):
        return self.out(F.silu(self.gate(x)) * self.value(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.attn, self.ff = Attention(cfg), SwiGLU(cfg)
        self.attn_norm, self.ff_norm = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, mask=None):
        normalized = self.attn_norm(x)
        x = x + self.dropout(self.attn(normalized, key_mask=mask, rope=True))
        return x + self.dropout(self.ff(self.ff_norm(x)))


class DecoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.self_attn, self.cross_attn, self.ff = Attention(cfg), Attention(cfg), SwiGLU(cfg)
        self.self_norm = RMSNorm(cfg.d_model)
        self.cross_norm = RMSNorm(cfg.d_model)
        self.ff_norm = RMSNorm(cfg.d_model)
        self.memory_gate = nn.Linear(cfg.d_model, 1)
        self.dropout = nn.Dropout(cfg.dropout)
        self.use_global_memory = cfg.use_global_memory

    def forward(self, x, memory, global_memory, memory_mask=None, global_mask=None):
        normalized = self.self_norm(x)
        x = x + self.dropout(self.self_attn(normalized, causal=True, rope=True))
        normalized = self.cross_norm(x)
        token_context = self.cross_attn(normalized, memory, key_mask=memory_mask)
        if self.use_global_memory:
            global_context = self.cross_attn(normalized, global_memory, key_mask=global_mask)
            gate = torch.sigmoid(self.memory_gate(normalized))
            token_context = gate * token_context + (1.0 - gate) * global_context
        x = x + self.dropout(token_context)
        return x + self.dropout(self.ff(self.ff_norm(x)))


class LegalEncoderDecoder(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = nn.ModuleList(EncoderBlock(cfg) for _ in range(cfg.encoder_layers))
        self.decoder = nn.ModuleList(DecoderBlock(cfg) for _ in range(cfg.decoder_layers))
        self.encoder_norm = RMSNorm(cfg.d_model)
        self.decoder_norm = RMSNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.apply(self._init)

    def _init(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def encode(self, source, source_mask=None):
        x = self.dropout(self.embedding(source))
        for layer in self.encoder:
            if self.training:
                x = checkpoint(layer, x, source_mask, use_reentrant=False)
            else:
                x = layer(x, source_mask)
        return self.encoder_norm(x)

    def pool_memory(self, memory, mask=None):
        block = self.cfg.global_block
        batch, length, width = memory.shape
        padding = (-length) % block
        if padding:
            memory = F.pad(memory, (0, 0, 0, padding))
            if mask is not None:
                mask = F.pad(mask, (0, padding), value=False)
        memory = memory.view(batch, -1, block, width)
        if mask is None:
            return memory.mean(2), None
        mask = mask.view(batch, -1, block)
        counts = mask.sum(2, keepdim=True).clamp_min(1)
        pooled = (memory * mask[..., None]).sum(2) / counts
        return pooled, mask.any(2)

    def decode(self, decoder_input, memory, source_mask=None):
        x = self.dropout(self.embedding(decoder_input))
        global_memory, global_mask = self.pool_memory(memory, source_mask)
        for layer in self.decoder:
            if self.training:
                x = checkpoint(
                    layer, x, memory, global_memory, source_mask, global_mask, use_reentrant=False
                )
            else:
                x = layer(x, memory, global_memory, source_mask, global_mask)
        return self.decoder_norm(x)

    def logits(self, hidden):
        return F.linear(hidden, self.embedding.weight)

    def forward(self, source, decoder_input, source_mask=None):
        memory = self.encode(source, source_mask)
        return self.logits(self.decode(decoder_input, memory, source_mask)), memory


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


def corrupt_source(source, mask_id, pad_id, fraction):
    selected = torch.rand(source.shape, device=source.device) < fraction
    selected &= source.ne(pad_id)
    corrupted = source.clone()
    corrupted[selected] = mask_id
    return corrupted, selected


def cosine_multiplier(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


def atomic_save(payload, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def evaluate_loss(model, loader, bos, device, batches):
    model.eval()
    losses = []
    with torch.inference_mode():
        for number, (source, target) in enumerate(loader):
            if number == batches:
                break
            source, target = source.to(device), target.to(device)
            decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), 1)
            logits, _ = model(source, decoder_input)
            losses.append(F.cross_entropy(logits.flatten(0, 1), target.flatten()).item())
    model.train()
    return sum(losses) / len(losses)


def train(
    model,
    loader,
    validation_loader,
    tokenizer,
    cfg,
    output_dir,
    device,
    resume=None,
    run_steps=None,
    distributed=False,
    rank=0,
    world_size=1,
    max_hours=None,
    generation_reserve_hours=0.0,
    session_started=None,
    finalize_on_time_limit=False,
):
    if cfg.grad_accum % world_size:
        raise ValueError("--grad-accum must be divisible by the number of DDP processes")
    local_grad_accum = cfg.grad_accum // world_size
    raw_model = unwrap_model(model)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, betas=(0.9, 0.95), weight_decay=cfg.weight_decay
    )
    warmup = int(cfg.steps * cfg.warmup_fraction)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: cosine_multiplier(step, cfg.steps, warmup)
    )
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    amp_dtype = (
        torch.bfloat16
        if device.type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float16
    )
    start_step, losses = 0, []
    if resume:
        saved = torch.load(resume, map_location="cpu", weights_only=False)
        state = saved["model"]
        if state and all(key.startswith("module.") for key in state):
            state = {key.removeprefix("module."): value for key, value in state.items()}
        raw_model.load_state_dict(state)
        if "optimizer" in saved:
            optimizer.load_state_dict(saved["optimizer"])
            scheduler.load_state_dict(saved["scheduler"])
            scaler.load_state_dict(saved["scaler"])
        start_step = int(saved.get("completed_steps", saved.get("total_steps", 0)))
        losses = list(saved.get("train_loss", []))
        rng_states = saved.get("rng_states")
        if rng_states and rank < len(rng_states):
            rng = rng_states[rank]
            random.setstate(rng["python"])
            torch.set_rng_state(rng["torch"])
            if device.type == "cuda" and rng.get("cuda") is not None:
                torch.cuda.set_rng_state(rng["cuda"], device)
        else:
            seed_everything(SEED + rank + start_step)
        if rank == 0:
            print(f"resumed at step {start_step}/{cfg.steps}")
    else:
        seed_everything(SEED + rank)
    bos = tokenizer.token_to_id("<bos>")
    pad = tokenizer.token_to_id("<pad>")
    mask_id = tokenizer.token_to_id("<mask>")
    iterator = iter(loader)
    recovery = output_dir / "foundation_recovery.pt"
    model.train()
    optimizer.zero_grad(set_to_none=True)
    end_step = min(cfg.steps, start_step + run_steps) if run_steps else cfg.steps
    if max_hours and not 0 <= generation_reserve_hours < max_hours:
        raise ValueError("--generation-reserve-hours must be non-negative and below --max-hours")
    training_hours = max_hours - generation_reserve_hours if max_hours else None
    deadline = (
        (session_started if session_started is not None else time.perf_counter())
        + training_hours * 3600
        if training_hours
        else None
    )
    if rank == 0 and max_hours:
        print(
            f"total runtime budget: {max_hours:.2f}h; training/data budget: "
            f"{training_hours:.2f}h; generation reserve: {generation_reserve_hours:.2f}h",
            flush=True,
        )
    completed = start_step
    stopped_by_time = False

    def save_recovery():
        local_rng = {
            "python": random.getstate(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
        }
        if distributed:
            rng_states = [None] * world_size
            dist.all_gather_object(rng_states, local_rng)
        else:
            rng_states = [local_rng]
        if rank == 0:
            atomic_save(
                {
                    "model": raw_model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler": scaler.state_dict(),
                    "completed_steps": completed,
                    "config": asdict(cfg),
                    "parameters": parameter_count(raw_model),
                    "train_loss": losses,
                    "rng_states": rng_states,
                    "world_size": world_size,
                },
                recovery,
            )
            print(f"checkpoint saved: {recovery}", flush=True)
        distributed_barrier(distributed)

    for step in range(start_step, end_step):
        running = 0.0
        for micro_step in range(local_grad_accum):
            try:
                source, target = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                source, target = next(iterator)
            source, target = source.to(device), target.to(device)
            decoder_input = torch.cat((torch.full_like(target[:, :1], bos), target[:, :-1]), 1)
            use_mlm = random.random() < cfg.mlm_probability
            model_source, selected = (
                corrupt_source(source, mask_id, pad, cfg.mlm_mask_fraction)
                if use_mlm
                else (source, None)
            )
            synchronize = micro_step == local_grad_accum - 1
            sync_context = (
                contextlib.nullcontext()
                if synchronize or not distributed
                else model.no_sync()
            )
            with sync_context:
                with torch.amp.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                    logits, memory = model(model_source, decoder_input)
                    loss = F.cross_entropy(logits.flatten(0, 1), target.flatten())
                    if selected is not None and selected.any():
                        encoder_logits = raw_model.logits(memory[selected])
                        loss = loss + cfg.mlm_weight * F.cross_entropy(
                            encoder_logits, source[selected]
                        )
                    loss = loss / local_grad_accum
                scaler.scale(loss).backward()
            running += float(loss.detach())
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        if distributed:
            reduced = torch.tensor(running, dtype=torch.float32, device=device)
            dist.all_reduce(reduced, op=dist.ReduceOp.SUM)
            running = reduced.item() / world_size
        losses.append(running)
        completed = step + 1
        if rank == 0 and (completed == 1 or completed % 10 == 0):
            tokens = completed * cfg.batch_size * cfg.grad_accum * (cfg.source_len + cfg.target_len)
            elapsed = time.perf_counter() - (session_started or time.perf_counter())
            rate = elapsed / max(1, completed - start_step)
            print(
                f"step {completed}/{cfg.steps}: loss={running:.4f}, tokens={tokens:,}, "
                f"elapsed={elapsed / 3600:.2f}h, seconds/step={rate:.2f}",
                flush=True,
            )
        time_limit = bool(deadline and time.perf_counter() >= deadline)
        if distributed:
            stop = torch.tensor(int(time_limit), device=device)
            dist.broadcast(stop, src=0)
            time_limit = bool(stop.item())
        if completed % cfg.checkpoint_every == 0 or completed == end_step or time_limit:
            save_recovery()
        if time_limit:
            stopped_by_time = True
            break
    should_finalize = completed >= cfg.steps or (stopped_by_time and finalize_on_time_limit)
    if not should_finalize:
        if rank == 0:
            reason = "time limit" if stopped_by_time else "chunk limit"
            print(f"{reason} reached at {completed}/{cfg.steps}; resume from {recovery}")
        return recovery, None, False, completed, "time_limit" if stopped_by_time else "chunk_limit"
    distributed_barrier(distributed)
    validation_loss = None
    if rank == 0:
        validation_loss = evaluate_loss(
            raw_model, validation_loader, bos, device, cfg.validation_batches
        )
    if distributed:
        value = torch.tensor(
            float("nan") if validation_loss is None else validation_loss,
            dtype=torch.float64,
            device=device,
        )
        dist.broadcast(value, src=0)
        validation_loss = value.item()
    final = output_dir / "foundation.pt"
    if rank == 0:
        atomic_save(
            {
                "model": raw_model.state_dict(),
                "total_steps": completed,
                "target_steps": cfg.steps,
                "config": asdict(cfg),
                "parameters": parameter_count(raw_model),
                "train_loss": losses,
                "validation_loss": validation_loss,
            },
            final,
        )
    distributed_barrier(distributed)
    reason = "target_steps" if completed >= cfg.steps else "time_limit_finalized"
    return final, validation_loss, True, completed, reason


@torch.inference_mode()
def generate(model, tokenizer, records, cfg, device):
    model.eval()
    pad, bos = tokenizer.token_to_id("<pad>"), tokenizer.token_to_id("<bos>")
    items, predictions = list(records.items()), {}
    truncated = 0
    for start in range(0, len(items), cfg.generation_batch_size):
        batch = items[start : start + cfg.generation_batch_size]
        encoded = []
        for _, record in batch:
            ids = tokenizer.encode(record["text"]).ids
            truncated += int(len(ids) > cfg.source_len)
            encoded.append(ids[-cfg.source_len :])
        width = max(map(len, encoded))
        source = torch.full((len(batch), width), pad, dtype=torch.long, device=device)
        source_mask = torch.zeros((len(batch), width), dtype=torch.bool, device=device)
        for row, ids in enumerate(encoded):
            source[row, : len(ids)] = torch.tensor(ids, device=device)
            source_mask[row, : len(ids)] = True
        memory = model.encode(source, source_mask)
        generated = torch.full((len(batch), 1), bos, dtype=torch.long, device=device)
        finished = [None] * len(batch)
        for _ in range(cfg.max_generation_tokens):
            hidden = model.decode(generated, memory, source_mask)
            next_id = model.logits(hidden[:, -1:])[:, -1].argmax(-1, keepdim=True)
            generated = torch.cat((generated, next_id), 1)
            if generated.size(1) % 8 == 0:
                for row in range(len(batch)):
                    words = tokenizer.decode(generated[row, 1:].tolist(), skip_special_tokens=True).split()
                    if len(words) >= cfg.target_words and finished[row] is None:
                        finished[row] = " ".join(words[: cfg.target_words])
                if all(finished):
                    break
        for row, (key, _) in enumerate(batch):
            words = tokenizer.decode(generated[row, 1:].tolist(), skip_special_tokens=True).split()
            if not words:
                words = ["the"]
            if len(words) < cfg.target_words:
                words = (words * math.ceil(cfg.target_words / len(words)))[: cfg.target_words]
            predictions[str(key)] = {"text": finished[row] or " ".join(words[: cfg.target_words])}
        print(f"generated {min(start + len(batch), len(items))}/{len(items)}")
    print(f"prompts truncated to {cfg.source_len} tokens: {truncated}/{len(items)}")
    return predictions


def distributed_generate(model, tokenizer, records, cfg, device, distributed, rank, world_size):
    if not distributed:
        return generate(model, tokenizer, records, cfg, device)
    local_records = dict(list(records.items())[rank::world_size])
    local_predictions = generate(model, tokenizer, local_records, cfg, device)
    gathered = [None] * world_size
    dist.all_gather_object(gathered, local_predictions)
    if rank != 0:
        return None
    predictions = {}
    for shard in gathered:
        predictions.update(shard)
    return predictions


def build_legacy_config(cfg):
    result = legacy.RunConfig(
        steps=cfg.steps,
        train_chars=cfg.train_chars,
        train_tokens=cfg.token_pool,
        seq_len=cfg.source_len,
        batch_size=cfg.batch_size,
        grad_accum=cfg.grad_accum,
    )
    return result


def preparation_signature(cfg):
    return {
        "train_chars": cfg.train_chars,
        "token_pool": cfg.token_pool,
        "tokenizer_chars": cfg.tokenizer_chars,
        "validation_examples": cfg.validation_examples,
        "source_len": cfg.source_len,
        "target_len": cfg.target_len,
        "vocab_size": cfg.vocab_size,
    }


def prepare_data(cfg, output, tokenizer_path, tokenizer_source=None):
    prepared_path = output / "prepared_data.pt"
    expected = preparation_signature(cfg)
    if prepared_path.exists() and tokenizer_path.exists():
        prepared = torch.load(prepared_path, map_location="cpu", weights_only=False)
        if prepared.get("signature") != expected:
            raise ValueError(
                f"{prepared_path} was built with different data settings; use another output "
                "directory or remove that cache"
            )
        return prepared

    texts = legacy.collect_training_text(build_legacy_config(cfg))
    random.Random(SEED).shuffle(texts)
    texts, validation_texts, validation_prompts, validation_truth = make_validation(
        texts, cfg.validation_examples
    )
    if tokenizer_source:
        tokenizer = Tokenizer.from_file(tokenizer_source)
        if Path(tokenizer_source) != tokenizer_path:
            tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = train_tokenizer(texts, tokenizer_path, cfg)
    cfg.vocab_size = tokenizer.get_vocab_size()
    expected["vocab_size"] = cfg.vocab_size
    tokens, spans = tokenize_documents(texts, tokenizer, cfg)
    validation_source_len, validation_target_len = 512, 128
    validation_tokens, validation_spans = tokenize_documents(
        validation_texts, tokenizer, cfg, validation_source_len, validation_target_len
    )
    prepared = {
        "signature": expected,
        "tokens": tokens,
        "spans": spans,
        "validation_tokens": validation_tokens,
        "validation_spans": validation_spans,
        "validation_prompts": validation_prompts,
        "validation_truth": validation_truth,
    }
    atomic_save(prepared, prepared_path)
    (output / "validation_prompts.json").write_text(
        json.dumps(validation_prompts, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (output / "validation_truth.json").write_text(
        json.dumps(validation_truth, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return prepared


def main(args):
    session_started = time.perf_counter()
    distributed, rank, world_size, local_rank, device = init_distributed()
    try:
        seed_everything()
        cfg = Config(
            steps=args.steps,
            train_chars=args.train_chars,
            token_pool=args.token_pool,
            validation_examples=args.validation_examples,
            batch_size=args.batch_size,
            grad_accum=args.grad_accum,
            use_global_memory=not args.no_global_memory,
        )
        output = Path(args.output_dir)
        if rank == 0:
            output.mkdir(parents=True, exist_ok=True)
        distributed_barrier(distributed)
        tokenizer_path = output / "legal_bpe_24k.json"

        if args.generate_only:
            if not args.checkpoint or not args.tokenizer:
                raise ValueError("--generate-only requires --checkpoint and --tokenizer")
            tokenizer = Tokenizer.from_file(args.tokenizer)
            saved = torch.load(args.checkpoint, map_location="cpu", weights_only=False)
            cfg = Config(**saved["config"])
            model = LegalEncoderDecoder(cfg).to(device)
            model.load_state_dict(saved["model"])
            prepared_path = output / "prepared_data.pt"
            if args.validation_only and not prepared_path.exists():
                raise ValueError("--validation-only requires prepared_data.pt in --output-dir")
            if prepared_path.exists():
                prepared = torch.load(prepared_path, map_location="cpu", weights_only=False)
                validation_predictions = distributed_generate(
                    model,
                    tokenizer,
                    prepared["validation_prompts"],
                    cfg,
                    device,
                    distributed,
                    rank,
                    world_size,
                )
                if rank == 0:
                    (output / "validation_prediction_foundation.json").write_text(
                        json.dumps(validation_predictions, ensure_ascii=False, indent=2),
                        encoding="utf-8",
                    )
            if args.validation_only:
                if rank == 0:
                    print("created validation_prediction_foundation.json; skipped official test generation")
                return
            test_holder = [legacy.load_test_data() if rank == 0 else None]
            if distributed:
                dist.broadcast_object_list(test_holder, src=0)
            test = test_holder[0]
            predictions = distributed_generate(
                model, tokenizer, test, cfg, device, distributed, rank, world_size
            )
            if rank == 0:
                zip_path = legacy.validate_and_zip(predictions, test, output, "foundation")
                print(f"created {zip_path}")
            return

        if args.checkpoint and not args.tokenizer and not tokenizer_path.exists():
            raise ValueError("An external --checkpoint must be paired with its --tokenizer")
        tokenizer_source = args.tokenizer or (str(tokenizer_path) if tokenizer_path.exists() else None)
        if rank == 0:
            prepare_data(cfg, output, tokenizer_path, tokenizer_source)
        distributed_barrier(distributed)
        prepared = torch.load(
            output / "prepared_data.pt", map_location="cpu", weights_only=False
        )
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
        cfg.vocab_size = tokenizer.get_vocab_size()
        if prepared.get("signature") != preparation_signature(cfg):
            raise ValueError("Prepared-data cache does not match the requested configuration")
        validation_prompts = prepared["validation_prompts"]
        validation_truth = prepared["validation_truth"]

        resume = args.checkpoint
        automatic = output / "foundation_recovery.pt"
        if resume is None and automatic.exists():
            resume = str(automatic)
        completed_steps = 0
        if resume:
            saved = torch.load(resume, map_location="cpu", weights_only=False)
            completed_steps = int(saved.get("completed_steps", saved.get("total_steps", 0)))
            saved_cfg = saved.get("config", {})
            architecture_fields = (
                "vocab_size", "d_model", "heads", "encoder_layers", "decoder_layers",
                "d_ff", "source_len", "target_len", "global_block", "use_global_memory",
            )
            mismatched = [
                name for name in architecture_fields
                if name in saved_cfg and saved_cfg[name] != getattr(cfg, name)
            ]
            if mismatched:
                raise ValueError(f"Checkpoint architecture mismatch: {', '.join(mismatched)}")
            schedule_fields = ("steps", "batch_size", "grad_accum")
            changed_schedule = [
                name for name in schedule_fields
                if name in saved_cfg and saved_cfg[name] != getattr(cfg, name)
            ]
            if changed_schedule:
                raise ValueError(
                    "Checkpoint training schedule mismatch: "
                    f"{', '.join(changed_schedule)}. Resume with the original values."
                )
        if completed_steps >= cfg.steps:
            raise ValueError(
                f"Checkpoint is already at {completed_steps} steps, not below target {cfg.steps}"
            )
        if cfg.grad_accum % world_size:
            raise ValueError("--grad-accum must be divisible by the number of DDP processes")
        local_grad_accum = cfg.grad_accum // world_size
        sample_offset = completed_steps * cfg.batch_size * cfg.grad_accum
        local_samples = (
            (cfg.steps - completed_steps) * cfg.batch_size * local_grad_accum
        )
        samples_per_step = cfg.batch_size * local_grad_accum
        training = ContinuationDataset(
            prepared["tokens"],
            prepared["spans"],
            cfg.source_len,
            cfg.target_len,
            local_samples,
            SEED,
            sample_offset,
            rank,
            world_size,
            samples_per_step,
        )
        validation_source_len, validation_target_len = 512, 128
        validation = ContinuationDataset(
            prepared["validation_tokens"],
            prepared["validation_spans"],
            validation_source_len,
            validation_target_len,
            cfg.validation_batches * cfg.batch_size,
            SEED + 1_000_000,
        )
        loader = torch.utils.data.DataLoader(
            training, batch_size=cfg.batch_size, pin_memory=True
        )
        validation_loader = torch.utils.data.DataLoader(
            validation, batch_size=cfg.batch_size, pin_memory=True
        )
        model = LegalEncoderDecoder(cfg).to(device)
        parameters = parameter_count(model)
        if rank == 0:
            print(
                f"parameters: {parameters:,}; GPUs: {world_size}; "
                f"local accumulation: {local_grad_accum}; global accumulation: {cfg.grad_accum}"
            )
        if not 100_000_000 <= parameters <= 150_000_000:
            raise ValueError("Flagship model must remain inside the recommended 100M-150M range")
        if distributed:
            model = DDP(
                model,
                device_ids=[local_rank],
                output_device=local_rank,
                gradient_as_bucket_view=True,
            )
        checkpoint_path, validation_loss, complete, completed_steps, stop_reason = train(
            model,
            loader,
            validation_loader,
            tokenizer,
            cfg,
            output,
            device,
            resume,
            args.run_steps,
            distributed,
            rank,
            world_size,
            args.max_hours,
            args.generation_reserve_hours,
            session_started,
            args.finalize_on_time_limit,
        )
        model = unwrap_model(model)
        if not complete:
            if rank == 0:
                partial = {
                    "status": "training_incomplete",
                    "stop_reason": stop_reason,
                    "completed_steps": completed_steps,
                    "checkpoint": str(checkpoint_path),
                    "tokenizer": str(tokenizer_path),
                    "prepared_data": str(output / "prepared_data.pt"),
                    "target_steps": cfg.steps,
                    "world_size": world_size,
                }
                (output / "manifest_partial.json").write_text(
                    json.dumps(partial, indent=2), encoding="utf-8"
                )
                print(json.dumps(partial, indent=2))
            return
        validation_predictions = distributed_generate(
            model, tokenizer, validation_prompts, cfg, device, distributed, rank, world_size
        )
        if rank == 0:
            (output / "validation_prediction_foundation.json").write_text(
                json.dumps(validation_predictions, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )
        manifest = {
            "architecture": "12-layer bidirectional encoder + 8-layer causal decoder",
            "original_component": "gated token-level and pooled global cross-attention memory",
            "parameters": parameters,
            "completed_steps": completed_steps,
            "target_steps": cfg.steps,
            "stop_reason": stop_reason,
            "world_size": world_size,
            "training_tokens_processed": completed_steps * cfg.batch_size * cfg.grad_accum * (cfg.source_len + cfg.target_len),
            "validation_loss": validation_loss,
            "training_seconds": time.perf_counter() - session_started,
            "checkpoint": str(checkpoint_path),
            "recovery_checkpoint": str(output / "foundation_recovery.pt"),
            "tokenizer": str(tokenizer_path),
            "submission": None,
            "config": asdict(cfg),
        }
        if args.validation_only:
            if rank == 0:
                (output / "manifest.json").write_text(
                    json.dumps(manifest, indent=2), encoding="utf-8"
                )
                print(json.dumps(manifest, indent=2))
                print("validation-only run complete; skipped official test generation")
            return
        test_holder = [legacy.load_test_data() if rank == 0 else None]
        if distributed:
            dist.broadcast_object_list(test_holder, src=0)
        test = test_holder[0]
        predictions = distributed_generate(
            model, tokenizer, test, cfg, device, distributed, rank, world_size
        )
        if rank != 0:
            return
        zip_path = legacy.validate_and_zip(predictions, test, output, "foundation")
        manifest["submission"] = str(zip_path)
        (output / "manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )
        print(json.dumps(manifest, indent=2))
    finally:
        if distributed and dist.is_initialized():
            dist.destroy_process_group()


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--steps", type=int, default=60_000)
    parser.add_argument("--run-steps", type=int)
    parser.add_argument("--max-hours", type=float)
    parser.add_argument("--generation-reserve-hours", type=float, default=0.0)
    parser.add_argument("--finalize-on-time-limit", action="store_true")
    parser.add_argument("--train-chars", type=int, default=500_000_000)
    parser.add_argument("--token-pool", type=int, default=100_000_000)
    parser.add_argument("--validation-examples", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=1)
    parser.add_argument("--grad-accum", type=int, default=8)
    parser.add_argument("--output-dir", default="/kaggle/working/phase1_foundation")
    parser.add_argument("--tokenizer")
    parser.add_argument("--checkpoint")
    parser.add_argument("--generate-only", action="store_true")
    parser.add_argument("--validation-only", action="store_true")
    parser.add_argument("--no-global-memory", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    main(parse_args())


In [ ]:
%%writefile /kaggle/working/phase2_lner.py
"""CS781 Phase 2 legal NER training, validation, and submission pipeline."""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import zipfile
from collections import Counter
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

from phase1_pipeline import (
    DecoderOnlyLM,
    EfficientLM,
    EncoderDecoderLM,
    HybridLM,
    RunConfig,
    build_model,
    causal_mask,
    sinusoidal_positions,
)
from phase1_foundation import Config as FoundationConfig, LegalEncoderDecoder


REPO_ID = "Exploration-Lab/CS-781-Capstone"
DATA_FILES = (
    "Phase_2_LNER/Train/fold1-text.json",
    "Phase_2_LNER/Train/fold1-gold.json",
    "Phase_2_LNER/Train/fold2-text.json",
    "Phase_2_LNER/Train/fold2-gold.json",
    "Phase_2_LNER/Train/labels.txt",
    "Phase_2_LNER/Test/fold3-text.json",
)
EXPECTED_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO",
}
ARTIFACT_NAMES = {
    "foundation": None,
    "decoder": "phase1_improved_decoder_artifacts.zip",
    "encoder_decoder": "phase1_ed_baseline_artifacts.zip",
    "hybrid": "phase1_hybrid_baseline_artifacts.zip",
    "efficient": "phase1_improved_efficient_artifacts.zip",
}
DEFAULT_METHOD = {
    "foundation": "full",
    "decoder": "distill",
    "encoder_decoder": "full",
    "hybrid": "partial",
    "efficient": "lorma",
}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_unique(filename: str, explicit: str | None = None) -> Path:
    if explicit:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    roots = [Path("/kaggle/input"), Path(__file__).resolve().parent / "inputs"]
    matches = [path for root in roots if root.exists() for path in root.rglob(filename)]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {filename}; found {matches}")
    return matches[0]


def extract_phase1_artifact(archive: Path, model_name: str, destination: Path) -> tuple[Path, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    checkpoint_name = f"{model_name}.pt"
    tokenizer_name = "phase1_bpe.json"
    with zipfile.ZipFile(archive) as bundle:
        names = set(bundle.namelist())
        missing = {checkpoint_name, tokenizer_name} - names
        if missing:
            raise ValueError(f"{archive.name} is missing {sorted(missing)}")
        checkpoint = destination / checkpoint_name
        tokenizer = destination / tokenizer_name
        checkpoint.write_bytes(bundle.read(checkpoint_name))
        tokenizer.write_bytes(bundle.read(tokenizer_name))
    return checkpoint, tokenizer


def locate_phase1_files(model_name: str, artifact_zip: str | None, destination: Path) -> tuple[Path, Path, str]:
    """Accept either the original ZIP or Kaggle's automatically unpacked dataset."""
    archive_name = ARTIFACT_NAMES[model_name]
    try:
        archive = find_unique(archive_name, artifact_zip)
    except FileNotFoundError:
        if artifact_zip:
            raise
        checkpoint = find_unique(f"{model_name}.pt")
        tokenizers = list(checkpoint.parent.rglob("phase1_bpe.json"))
        if len(tokenizers) != 1:
            raise FileNotFoundError(
                f"Found {checkpoint}, but expected one phase1_bpe.json beside it; found {tokenizers}"
            )
        return checkpoint, tokenizers[0], f"{checkpoint.parent.name} (Kaggle-unpacked)"
    checkpoint, tokenizer = extract_phase1_artifact(archive, model_name, destination)
    return checkpoint, tokenizer, archive.name


def download_data(destination: Path) -> dict[str, Path]:
    from huggingface_hub import hf_hub_download

    destination.mkdir(parents=True, exist_ok=True)
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("HF_TOKEN is missing. Add a Kaggle secret named HF and load it first.")
    paths = {}
    for filename in DATA_FILES:
        downloaded = Path(hf_hub_download(
            repo_id=REPO_ID,
            filename=filename,
            repo_type="dataset",
            token=token,
        ))
        target = destination / Path(filename).name
        target.write_bytes(downloaded.read_bytes())
        paths[Path(filename).name] = target
    return paths


def read_json(path: Path):
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)


def normalize_texts(raw) -> dict[str, str]:
    if not isinstance(raw, dict):
        raise TypeError("Text JSON must be an object keyed by document ID")
    result = {}
    for document_id, value in raw.items():
        if isinstance(value, str):
            text = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            text = value["text"]
        else:
            raise TypeError(f"Unsupported text entry for {document_id}: {type(value).__name__}")
        result[str(document_id)] = text
    return result


def normalize_entity(raw) -> tuple[int, int, str, str | None]:
    if isinstance(raw, (list, tuple)) and len(raw) >= 3:
        start, end, label = raw[:3]
        entity_text = raw[3] if len(raw) > 3 and isinstance(raw[3], str) else None
    elif isinstance(raw, dict):
        start = raw.get("start", raw.get("start_offset"))
        end = raw.get("end", raw.get("end_offset"))
        label = raw.get("label", raw.get("type"))
        entity_text = raw.get("text", raw.get("entity_text"))
    else:
        raise TypeError(f"Unsupported entity annotation: {raw!r}")
    return int(start), int(end), str(label), entity_text


def normalize_gold(raw) -> dict[str, list[tuple[int, int, str]]]:
    if not isinstance(raw, dict):
        raise TypeError("Gold JSON must be an object keyed by document ID")
    result = {}
    for document_id, value in raw.items():
        if isinstance(value, dict):
            value = value.get("entities", value.get("spans", value.get("annotations")))
        if not isinstance(value, list):
            raise TypeError(f"Unsupported gold entry for {document_id}")
        result[str(document_id)] = [(s, e, label) for s, e, label, _ in map(normalize_entity, value)]
    return result


def load_labels(path: Path) -> list[str]:
    raw = path.read_text(encoding="utf-8")
    labels = []
    for line in raw.splitlines():
        for chunk in line.split(","):
            if chunk.strip():
                candidate = chunk.strip().split()[0]
                if candidate in EXPECTED_LABELS:
                    labels.append(candidate)
    if set(labels) != EXPECTED_LABELS or len(labels) != 12:
        raise ValueError(f"Unexpected labels: {labels}")
    return labels


def validate_gold(texts: dict[str, str], gold: dict[str, list[tuple[int, int, str]]], labels: list[str]) -> None:
    if set(texts) != set(gold):
        raise ValueError(f"Text/gold ID mismatch: text-only={set(texts)-set(gold)}, gold-only={set(gold)-set(texts)}")
    allowed = set(labels)
    overlaps = 0
    for document_id, text in texts.items():
        previous_end = -1
        for start, end, label in sorted(gold[document_id]):
            if not 0 <= start < end <= len(text):
                raise ValueError(f"Bad span {document_id}: {(start, end, label)} for length {len(text)}")
            if label not in allowed:
                raise ValueError(f"Unknown label {label!r} in {document_id}")
            overlaps += start < previous_end
            previous_end = max(previous_end, end)
    if overlaps:
        raise ValueError(f"BIO cannot represent the {overlaps} overlapping gold spans; use a span head")


def make_tags(labels: list[str]) -> tuple[list[str], dict[str, int]]:
    tags = ["O"] + [tag for label in labels for tag in (f"B-{label}", f"I-{label}")]
    return tags, {tag: index for index, tag in enumerate(tags)}


def encode_document(tokenizer, text: str):
    encoding = tokenizer.encode(text)
    if len(encoding.ids) != len(encoding.offsets):
        raise RuntimeError("Tokenizer returned inconsistent IDs and offsets")
    boundary = set(" \t\r\n.,;:!?()[]{}\\\"“”‘’")
    offsets = []
    for left, right in encoding.offsets:
        left, right = int(left), int(right)
        while left < right and text[left] in boundary:
            left += 1
        while right > left and text[right - 1] in boundary:
            right -= 1
        offsets.append((left, right))
    return list(encoding.ids), offsets


def align_tags(
    offsets: list[tuple[int, int]],
    entities: list[tuple[int, int, str]],
    tag_to_id: dict[str, int],
) -> list[int]:
    tags = [tag_to_id["O"]] * len(offsets)
    for start, end, label in sorted(entities):
        token_indexes = [i for i, (left, right) in enumerate(offsets) if right > start and left < end]
        if not token_indexes:
            raise ValueError(f"No tokens cover entity {(start, end, label)}")
        token_indexes = [index for index in token_indexes if tags[index] == tag_to_id["O"]]
        if not token_indexes:
            continue
        first = token_indexes[0]
        tags[first] = tag_to_id[f"B-{label}"]
        for index in token_indexes[1:]:
            tags[index] = tag_to_id[f"I-{label}"]
    return tags


class WindowDataset(Dataset):
    def __init__(
        self,
        texts: dict[str, str],
        tokenizer,
        max_length: int,
        stride: int,
        gold: dict[str, list[tuple[int, int, str]]] | None = None,
        tag_to_id: dict[str, int] | None = None,
    ):
        if not 0 <= stride < max_length:
            raise ValueError("stride must satisfy 0 <= stride < max_length")
        self.documents = {}
        self.windows = []
        step = max_length - stride
        for document_id, text in texts.items():
            ids, offsets = encode_document(tokenizer, text)
            labels = align_tags(offsets, gold[document_id], tag_to_id) if gold is not None else None
            self.documents[document_id] = {"text": text, "ids": ids, "offsets": offsets}
            if not ids:
                continue
            for start in range(0, len(ids), step):
                end = min(start + max_length, len(ids))
                self.windows.append({
                    "document_id": document_id,
                    "start": start,
                    "ids": ids[start:end],
                    "labels": labels[start:end] if labels is not None else None,
                })
                if end == len(ids):
                    break

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        return self.windows[index]


def make_collate(pad_id: int):
    def collate(batch):
        length = max(len(item["ids"]) for item in batch)
        ids = torch.full((len(batch), length), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), length), dtype=torch.bool)
        has_labels = batch[0]["labels"] is not None
        labels = torch.full((len(batch), length), -100, dtype=torch.long) if has_labels else None
        for row, item in enumerate(batch):
            size = len(item["ids"])
            ids[row, :size] = torch.tensor(item["ids"], dtype=torch.long)
            mask[row, :size] = True
            if has_labels:
                labels[row, :size] = torch.tensor(item["labels"], dtype=torch.long)
        return {
            "ids": ids,
            "mask": mask,
            "labels": labels,
            "document_ids": [item["document_id"] for item in batch],
            "starts": [item["start"] for item in batch],
        }
    return collate


class LoRMAPlusLinear(nn.Module):
    """LoRMA+ output-space transform: (I + scale * BA) W0 x."""

    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 8.0):
        super().__init__()
        self.base = base
        self.a = nn.Linear(base.out_features, rank, bias=False)
        self.b = nn.Linear(rank, base.out_features, bias=False)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.b.weight)
        for parameter in self.base.parameters():
            parameter.requires_grad = False

    def forward(self, inputs):
        frozen = self.base(inputs)
        return frozen + self.scale * self.b(self.a(frozen))


class LNERModel(nn.Module):
    def __init__(self, backbone: nn.Module, model_name: str, tag_count: int):
        super().__init__()
        self.backbone = backbone
        self.model_name = model_name
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(backbone.cfg.d_model, tag_count)

    def hidden(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        cfg = self.backbone.cfg
        if self.model_name == "foundation":
            return self.backbone.encode(ids, mask)
        if self.model_name in {"encoder_decoder", "hybrid"}:
            embedded = self.backbone.embedding(ids) + sinusoidal_positions(ids.size(1), cfg.d_model, ids.device)
            if self.model_name == "encoder_decoder":
                memory = self.backbone.encoder(embedded, src_key_padding_mask=~mask)
            else:
                lengths = mask.sum(1).cpu()
                packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
                memory, _ = self.backbone.recurrent(packed)
                memory, _ = pad_packed_sequence(memory, batch_first=True, total_length=ids.size(1))
            target = self.backbone.embedding(ids) + sinusoidal_positions(ids.size(1), cfg.d_model, ids.device)
            decoded = self.backbone.decoder(
                target,
                memory,
                tgt_mask=causal_mask(ids.size(1), ids.device),
                tgt_key_padding_mask=~mask,
                memory_key_padding_mask=~mask,
            )
            denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
            summary = (memory * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
            decoded = self.backbone.fusion(decoded, summary.expand_as(decoded))
            return self.backbone.norm(decoded)
        if self.model_name == "decoder":
            hidden = self.backbone.embedding(ids) + sinusoidal_positions(ids.size(1), cfg.d_model, ids.device)
            hidden = self.backbone.blocks(
                hidden,
                mask=causal_mask(ids.size(1), ids.device),
                src_key_padding_mask=~mask,
            )
            counts = mask.cumsum(1).clamp_min(1).unsqueeze(-1)
            summary = (hidden * mask.unsqueeze(-1)).cumsum(1) / counts
            return self.backbone.norm(self.backbone.fusion(hidden, summary))
        if self.model_name == "efficient":
            hidden = self.backbone.embedding(ids)
            for block in self.backbone.blocks:
                hidden = block(hidden)
            return self.backbone.norm(hidden)
        raise ValueError(self.model_name)

    def forward(self, ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.dropout(self.hidden(ids, mask)))


def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        raise RuntimeError("This notebook requires a PyTorch version supporting torch.load(..., weights_only=True)")


def load_phase1(checkpoint: Path, expected_name: str):
    saved = safe_torch_load(checkpoint)
    if saved.get("model_name") not in (None, expected_name):
        raise ValueError(f"Checkpoint is {saved.get('model_name')}, expected {expected_name}")
    cfg = RunConfig(**saved["config"])
    model = build_model(expected_name, cfg)
    model.load_state_dict(saved["model"], strict=True)
    return model, cfg, saved


def load_foundation(checkpoint: Path):
    saved = safe_torch_load(checkpoint)
    cfg = FoundationConfig(**saved["config"])
    model = LegalEncoderDecoder(cfg)
    model.load_state_dict(saved["model"], strict=True)
    return model, cfg, saved


def make_student(full_decoder: DecoderOnlyLM, layers: int = 8) -> DecoderOnlyLM:
    if layers >= len(full_decoder.blocks.layers):
        raise ValueError("Student must have fewer layers than teacher/base")
    student = DecoderOnlyLM(full_decoder.cfg, layers=layers)
    student.embedding.load_state_dict(full_decoder.embedding.state_dict())
    student.fusion.load_state_dict(full_decoder.fusion.state_dict())
    student.norm.load_state_dict(full_decoder.norm.state_dict())
    indexes = torch.linspace(0, len(full_decoder.blocks.layers) - 1, layers).round().int().tolist()
    for target, source_index in zip(student.blocks.layers, indexes):
        target.load_state_dict(full_decoder.blocks.layers[source_index].state_dict())
    student.head.weight = student.embedding.weight
    return student


def apply_method(model: LNERModel, method: str, rank: int) -> None:
    if method in {"full", "distill"}:
        return
    for parameter in model.backbone.parameters():
        parameter.requires_grad = False
    if method == "partial":
        if model.model_name != "hybrid":
            raise ValueError("The provided partial configuration is for the hybrid model")
        for name, parameter in model.backbone.recurrent.named_parameters():
            if "_l4" in name:
                parameter.requires_grad = True
        for layer in model.backbone.decoder.layers[-2:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
        for module in (model.backbone.fusion, model.backbone.norm):
            for parameter in module.parameters():
                parameter.requires_grad = True
    elif method == "lorma":
        if model.model_name == "efficient":
            for block in model.backbone.blocks:
                block.attention.q = LoRMAPlusLinear(block.attention.q, rank=rank, alpha=float(rank))
                block.attention.v = LoRMAPlusLinear(block.attention.v, rank=rank, alpha=float(rank))
            for parameter in model.backbone.norm.parameters():
                parameter.requires_grad = True
        elif model.model_name == "hybrid":
            for layer in model.backbone.decoder.layers:
                layer.linear1 = LoRMAPlusLinear(layer.linear1, rank=rank, alpha=float(rank))
                layer.linear2 = LoRMAPlusLinear(layer.linear2, rank=rank, alpha=float(rank))
            for module in (model.backbone.fusion, model.backbone.norm):
                for parameter in module.parameters():
                    parameter.requires_grad = True
        else:
            raise ValueError("LoRMA is configured for the efficient or hybrid model")
    else:
        raise ValueError(method)


def decode_tags(tags: list[int], offsets: list[tuple[int, int]], id_to_tag: list[str]):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close():
        nonlocal active_label, active_start, active_end
        if active_label is not None and active_start is not None and active_end is not None:
            spans.append([int(active_start), int(active_end), active_label])
        active_label = active_start = active_end = None

    for tag_id, (start, end) in zip(tags, offsets):
        tag = id_to_tag[tag_id]
        if tag == "O" or end <= start:
            close()
            continue
        prefix, label = tag.split("-", 1)
        if prefix == "B" or label != active_label:
            close()
            active_label, active_start, active_end = label, start, end
        else:
            active_end = end
    close()
    return spans


def decode_score_maps(dataset, totals, counts, id_to_tag, o_bias=0.0):
    predictions = {}
    for document_id, record in dataset.documents.items():
        if len(record["ids"]) == 0:
            predictions[document_id] = []
            continue
        if torch.any(counts[document_id] == 0):
            raise RuntimeError(f"Uncovered tokens in {document_id}")
        scores = totals[document_id] / counts[document_id].unsqueeze(1)
        scores[:, 0] -= o_bias
        tags = scores.argmax(1).tolist()
        predictions[document_id] = decode_tags(tags, record["offsets"], id_to_tag)
    return predictions


@torch.inference_mode()
def predict(
    model, dataset: WindowDataset, loader: DataLoader, id_to_tag: list[str],
    device: torch.device, o_bias: float = 0.0, return_scores: bool = False,
):
    model.eval()
    tag_count = len(id_to_tag)
    totals = {
        document_id: torch.zeros((len(record["ids"]), tag_count), dtype=torch.float32)
        for document_id, record in dataset.documents.items()
    }
    counts = {
        document_id: torch.zeros(len(record["ids"]), dtype=torch.float32)
        for document_id, record in dataset.documents.items()
    }
    for batch in loader:
        ids, mask = batch["ids"].to(device), batch["mask"].to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(ids, mask).float().cpu()
        for row, (document_id, start) in enumerate(zip(batch["document_ids"], batch["starts"])):
            size = int(batch["mask"][row].sum())
            totals[document_id][start:start + size] += logits[row, :size]
            counts[document_id][start:start + size] += 1
    if return_scores:
        return totals, counts
    return decode_score_maps(dataset, totals, counts, id_to_tag, o_bias)


def strict_macro_f1(predictions, gold, labels):
    per_label = {}
    for label in labels:
        predicted = {
            (document_id, int(span[0]), int(span[1]))
            for document_id, spans in predictions.items()
            for span in spans if span[2] == label
        }
        actual = {
            (document_id, int(start), int(end))
            for document_id, spans in gold.items()
            for start, end, entity_label in spans if entity_label == label
        }
        tp = len(predicted & actual)
        precision = tp / len(predicted) if predicted else 0.0
        recall = tp / len(actual) if actual else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {"precision": precision, "recall": recall, "f1": f1, "support": len(actual)}
    return sum(row["f1"] for row in per_label.values()) / len(labels), per_label


def class_weights(dataset: WindowDataset, tag_count: int) -> torch.Tensor:
    counts = Counter(tag for window in dataset.windows for tag in (window["labels"] or []))
    maximum = max(counts.values())
    weights = torch.ones(tag_count)
    for tag in range(tag_count):
        weights[tag] = min(8.0, math.sqrt(maximum / max(1, counts[tag])))
    weights[0] = 0.05
    return weights


def build_teacher(path: Path, tag_count: int, device: torch.device):
    saved = safe_torch_load(path)
    config = saved["run_config"]
    if config["method"] != "full":
        raise ValueError("Use a full-fine-tuning Phase 2 checkpoint as the distillation teacher")
    if config["model_name"] == "foundation":
        cfg = FoundationConfig(**saved["phase1_config"])
        backbone = LegalEncoderDecoder(cfg)
    else:
        cfg = RunConfig(**saved["phase1_config"])
        backbone = build_model(config["model_name"], cfg)
    teacher = LNERModel(backbone, config["model_name"], tag_count)
    teacher.load_state_dict(saved["model"], strict=True)
    teacher.to(device).eval()
    for parameter in teacher.parameters():
        parameter.requires_grad = False
    return teacher


def train(
    model,
    teacher,
    train_loader,
    validation_dataset,
    validation_loader,
    validation_gold,
    labels,
    id_to_tag,
    device,
    args,
    output_dir,
    phase1_config,
):
    weights = class_weights(train_loader.dataset, len(id_to_tag)).to(device)
    classifier = [parameter for parameter in model.classifier.parameters() if parameter.requires_grad]
    classifier_ids = {id(parameter) for parameter in classifier}
    backbone = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad and id(parameter) not in classifier_ids
    ]
    trainable = backbone + classifier
    optimizer = torch.optim.AdamW([
        {"params": backbone, "lr": args.learning_rate},
        {"params": classifier, "lr": max(5e-4, args.learning_rate * 25)},
    ], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    best_score = -1.0
    best_path = output_dir / "best.pt"
    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, batch in enumerate(train_loader, 1):
            ids = batch["ids"].to(device)
            mask = batch["mask"].to(device)
            targets = batch["labels"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids, mask)
                hard_loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten(), weight=weights, ignore_index=-100)
                if teacher is None:
                    loss = hard_loss
                else:
                    with torch.no_grad():
                        teacher_logits = teacher(ids, mask)
                    valid = targets != -100
                    temperature = args.temperature
                    soft_loss = F.kl_div(
                        F.log_softmax(logits[valid] / temperature, dim=-1),
                        F.softmax(teacher_logits[valid] / temperature, dim=-1),
                        reduction="batchmean",
                    ) * temperature * temperature
                    loss = 0.5 * hard_loss + 0.5 * soft_loss
                loss = loss / args.grad_accum
            scaler.scale(loss).backward()
            if step % args.grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss) * args.grad_accum)
        totals, counts = predict(
            model, validation_dataset, validation_loader, id_to_tag, device,
            return_scores=True,
        )
        score, per_label, selected_bias = -1.0, None, 0.0
        for candidate_bias in (0.0, 0.5, 1.0, 1.5, 2.0, 3.0):
            predictions = decode_score_maps(
                validation_dataset, totals, counts, id_to_tag,
                o_bias=candidate_bias,
            )
            candidate_score, candidate_per_label = strict_macro_f1(predictions, validation_gold, labels)
            if candidate_score > score:
                score, per_label, selected_bias = candidate_score, candidate_per_label, candidate_bias
        row = {
            "epoch": epoch,
            "train_loss": sum(losses) / len(losses),
            "strict_macro_f1": score,
            "o_bias": selected_bias,
        }
        history.append(row)
        print(json.dumps(row), flush=True)
        if score > best_score:
            best_score = score
            torch.save({
                "model": model.state_dict(),
                "phase1_config": asdict(phase1_config),
                "run_config": {
                    "model_name": args.model,
                    "method": args.method,
                    "max_length": args.max_length,
                    "stride": args.stride,
                    "rank": args.rank,
                },
                "labels": labels,
                "tags": id_to_tag,
                "score": score,
                "per_label": per_label,
                "o_bias": selected_bias,
            }, best_path)
    saved = safe_torch_load(best_path)
    model.load_state_dict(saved["model"], strict=True)
    (output_dir / "training_log.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    (output_dir / "validation_metrics.json").write_text(json.dumps({
        "strict_macro_f1": saved["score"], "per_label": saved["per_label"]
    }, indent=2), encoding="utf-8")
    return best_path


def validate_and_zip(predictions, test_texts, labels, output_dir):
    if set(predictions) != set(test_texts):
        raise ValueError("Prediction IDs do not exactly match test IDs")
    allowed = set(labels)
    cleaned = {}
    for document_id, spans in predictions.items():
        text_length = len(test_texts[document_id])
        unique = set()
        for span in spans:
            if len(span) != 3:
                raise ValueError(f"Bad prediction: {span}")
            start, end, label = int(span[0]), int(span[1]), str(span[2])
            if not 0 <= start < end <= text_length or label not in allowed:
                raise ValueError(f"Invalid prediction for {document_id}: {span}")
            unique.add((start, end, label))
        cleaned[document_id] = [list(span) for span in sorted(unique)]
    prediction_path = output_dir / "prediction.json"
    prediction_path.write_text(json.dumps(cleaned, ensure_ascii=False), encoding="utf-8")
    archive_path = output_dir / f"submission_{output_dir.name}.zip"
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(archive_path) as archive:
        if archive.namelist() != ["prediction.json"]:
            raise RuntimeError("Submission ZIP has the wrong layout")
        json.loads(archive.read("prediction.json"))
    return archive_path


def self_check() -> None:
    tags, tag_to_id = make_tags(["COURT"])
    decoded = decode_tags(
        [tag_to_id["B-COURT"], tag_to_id["I-COURT"], tag_to_id["O"]],
        [(0, 5), (5, 10), (11, 12)],
        tags,
    )
    assert decoded == [[0, 10, "COURT"]]
    score, _ = strict_macro_f1({"d": decoded}, {"d": [(0, 10, "COURT")]}, ["COURT"])
    assert score == 1.0
    assert align_tags([(136, 153)], [(137, 153, "COURT")], tag_to_id) == [tag_to_id["B-COURT"]]
    assert align_tags([(7439, 7501)], [(7439, 7500, "COURT")], tag_to_id) == [tag_to_id["B-COURT"]]
    print("Phase 2 helper self-check passed")


def run(args):
    seed_everything(args.seed)
    args.method = args.method or DEFAULT_METHOD[args.model]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_dir = Path(args.output_dir or f"/kaggle/working/phase2_{args.model}_{args.method}")
    output_dir.mkdir(parents=True, exist_ok=True)

    if args.model == "foundation":
        phase1_checkpoint = find_unique("foundation.pt", args.phase1_checkpoint)
        tokenizer_path = find_unique("legal_bpe_24k.json", args.phase1_tokenizer)
        artifact_label = phase1_checkpoint.name
    else:
        phase1_checkpoint, tokenizer_path, artifact_label = locate_phase1_files(
            args.model, args.artifact_zip, output_dir / "phase1"
        )
    data_paths = download_data(output_dir / "data")

    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    pad_id = tokenizer.token_to_id("<pad>")
    if pad_id is None:
        raise ValueError("Phase 1 tokenizer has no <pad> token")

    labels = load_labels(data_paths["labels.txt"])
    tags, tag_to_id = make_tags(labels)
    train_texts = normalize_texts(read_json(data_paths["fold1-text.json"]))
    train_gold = normalize_gold(read_json(data_paths["fold1-gold.json"]))
    validation_texts = normalize_texts(read_json(data_paths["fold2-text.json"]))
    validation_gold = normalize_gold(read_json(data_paths["fold2-gold.json"]))
    test_texts = normalize_texts(read_json(data_paths["fold3-text.json"]))
    validate_gold(train_texts, train_gold, labels)
    validate_gold(validation_texts, validation_gold, labels)

    train_dataset = WindowDataset(train_texts, tokenizer, args.max_length, args.stride, train_gold, tag_to_id)
    validation_dataset = WindowDataset(validation_texts, tokenizer, args.max_length, args.stride, validation_gold, tag_to_id)
    test_dataset = WindowDataset(test_texts, tokenizer, args.max_length, args.stride)
    collate = make_collate(pad_id)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)
    validation_loader = DataLoader(validation_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=collate, num_workers=2, pin_memory=True)

    if args.model == "foundation":
        backbone, phase1_config, phase1_saved = load_foundation(phase1_checkpoint)
    else:
        backbone, phase1_config, phase1_saved = load_phase1(phase1_checkpoint, args.model)
    if args.model == "efficient" and args.bidirectional_efficient:
        for block in backbone.blocks:
            block.attention.bidirectional = True
    if args.method == "distill":
        if args.model != "decoder":
            raise ValueError("The provided student recipe uses the decoder Phase 1 model")
        backbone = make_student(backbone, layers=args.student_layers)
    model = LNERModel(backbone, args.model, len(tags))
    if args.model == "foundation":
        for parameter in model.backbone.decoder.parameters():
            parameter.requires_grad = False
        for parameter in model.backbone.decoder_norm.parameters():
            parameter.requires_grad = False
    apply_method(model, args.method, args.rank)
    model.to(device)

    teacher = None
    if args.method == "distill":
        if not args.teacher_checkpoint:
            raise ValueError("--teacher-checkpoint is required for distillation")
        teacher = build_teacher(Path(args.teacher_checkpoint), len(tags), device)

    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(json.dumps({
        "artifact": artifact_label,
        "model": args.model,
        "method": args.method,
        "phase1_steps": phase1_saved.get("total_steps"),
        "parameters": total,
        "trainable_parameters": trainable,
        "train_windows": len(train_dataset),
        "validation_windows": len(validation_dataset),
        "test_windows": len(test_dataset),
        "device": str(device),
    }, indent=2))

    best_path = train(
        model, teacher, train_loader, validation_dataset, validation_loader,
        validation_gold, labels, tags, device, args, output_dir, phase1_config,
    )
    selected = safe_torch_load(best_path)
    predictions = predict(
        model, test_dataset, test_loader, tags, device,
        o_bias=float(selected.get("o_bias", 0.0)),
    )
    submission = validate_and_zip(predictions, test_texts, labels, output_dir)
    manifest = {
        "model": args.model,
        "method": args.method,
        "phase1_artifact": artifact_label,
        "phase1_steps": phase1_saved.get("total_steps"),
        "parameters": total,
        "trainable_parameters": trainable,
        "checkpoint": str(best_path),
        "submission": str(submission),
    }
    (output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(json.dumps(manifest, indent=2))


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=tuple(ARTIFACT_NAMES))
    parser.add_argument("--method", choices=("full", "partial", "lorma", "distill"))
    parser.add_argument("--artifact-zip")
    parser.add_argument("--phase1-checkpoint")
    parser.add_argument("--phase1-tokenizer")
    parser.add_argument("--teacher-checkpoint")
    parser.add_argument("--output-dir")
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--grad-accum", type=int, default=2)
    parser.add_argument("--learning-rate", type=float, default=2e-5)
    parser.add_argument("--max-length", type=int, default=256)
    parser.add_argument("--stride", type=int, default=64)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--student-layers", type=int, default=8)
    parser.add_argument("--temperature", type=float, default=2.0)
    parser.add_argument("--bidirectional-efficient", action="store_true")
    parser.add_argument("--seed", type=int, default=230338)
    parser.add_argument("--self-test", action="store_true")
    return parser.parse_args()


if __name__ == "__main__":
    arguments = parse_args()
    if arguments.self_test:
        self_check()
    elif not arguments.model:
        raise SystemExit("--model is required unless --self-test is used")
    else:
        run(arguments)


In [ ]:
%%writefile /kaggle/working/phase3_rr.py
"""CS781 Phase 3 - Rhetorical Role (RR) prediction from Phase 1 legal language models.

Every experiment starts from a Phase 1 checkpoint (decoder / encoder_decoder / hybrid / efficient).
A sentence-level head is added on top of a pooled representation of the *target* sentence; the
target can be presented alone (``--context none``) or after its neighbouring sentences
(``--context window``).  Adaptation techniques: full fine-tuning, LORMA+, LoRA, IA3 and
knowledge distillation (truncated decoder student trained on a teacher's soft labels).
"""

from __future__ import annotations

import argparse
import copy
import json
import math
import os
import random
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from phase2_lner import (
    ARTIFACT_NAMES,
    LNERModel,
    LoRMAPlusLinear,
    extract_phase1_artifact,
    load_phase1,
    make_student,
    safe_torch_load,
)


def walk_files(roots):
    """All files under the roots, following symlinks (Kaggle mounts inputs through symlinks)."""
    files = []
    for root in roots:
        if root and os.path.exists(root):
            for folder, _, names in os.walk(root, followlinks=True):
                files += [Path(folder) / name for name in names]
    return files


def show_inputs(root="/kaggle/input"):
    print(f"Files under {root}:")
    for path in walk_files([root])[:300]:
        try:
            print("  ", path, path.stat().st_size)
        except OSError:
            print("  ", path)


def locate_phase1_files(model_name: str, artifact_zip, destination: Path):
    """Find the Phase 1 checkpoint + tokenizer as a ZIP or as Kaggle's auto-extracted folder."""
    if artifact_zip:
        archive = Path(artifact_zip)
        checkpoint, tokenizer = extract_phase1_artifact(archive, model_name, destination)
        return checkpoint, tokenizer, archive.name
    files = walk_files(["/kaggle/input", str(Path(__file__).resolve().parent / "inputs")])
    archive_name = ARTIFACT_NAMES[model_name]
    zips = [f for f in files if f.name == archive_name]
    if not zips:  # tolerate renamed uploads: any zip that contains <model>.pt
        for f in files:
            if f.suffix == ".zip":
                try:
                    with zipfile.ZipFile(f) as bundle:
                        if f"{model_name}.pt" in bundle.namelist():
                            zips.append(f)
                except (zipfile.BadZipFile, OSError):
                    pass
    if zips:
        checkpoint, tokenizer = extract_phase1_artifact(zips[0], model_name, destination)
        return checkpoint, tokenizer, zips[0].name
    checkpoints = [f for f in files if f.name == f"{model_name}.pt"]
    for checkpoint in checkpoints:
        tokenizers = [f for f in files if f.name == "phase1_bpe.json" and f.parent == checkpoint.parent]
        if tokenizers:
            return checkpoint, tokenizers[0], f"{checkpoint.parent.name} (Kaggle-unpacked)"
    show_inputs()
    raise FileNotFoundError(
        f"Could not find {archive_name} or {model_name}.pt + phase1_bpe.json under /kaggle/input. "
        "Attach the dataset holding the Phase 1 artifact ZIP (right panel -> Add Input), then rerun this cell."
    )

REPO_ID = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = ("CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "label_vocab.json")
TEST_FILES = ("CL_test.json", "IT_test.json")
DOMAINS = ("CL", "IT")
POSITION_BUCKETS = 10
OFFICIAL_LABELS = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
    "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt",
    "RatioOfTheDecision", "RulingByPresentCourt", "None",
]


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ----------------------------------------------------------------------------- data

def obtain_data(destination: Path, data_dir: str | None) -> dict[str, Path]:
    """Use local copies (``--data-dir`` or an attached Kaggle dataset) or download from HF."""
    names = TRAIN_FILES + TEST_FILES
    roots = [Path(data_dir)] if data_dir else []
    roots.append(Path("/kaggle/input"))
    found = {}
    available = walk_files([str(root) for root in roots])
    for name in names:
        matches = sorted(f for f in available if f.name == name)
        if matches:
            found[name] = matches[0]
    missing = [name for name in names if name not in found]
    if missing:
        from huggingface_hub import hf_hub_download

        token = os.environ.get("HF_TOKEN")
        if not token:
            raise RuntimeError("HF_TOKEN is missing (add the Kaggle secret HF) and data was not found locally")
        destination.mkdir(parents=True, exist_ok=True)
        for name in missing:
            folder = "Train" if name in TRAIN_FILES else "Test"
            path = Path(hf_hub_download(
                repo_id=REPO_ID, filename=f"Phase_3_RR/{folder}/{name}", repo_type="dataset", token=token,
            ))
            target = destination / name
            target.write_bytes(path.read_bytes())
            found[name] = target
    return found


def read_json(path: Path):
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)


def _pick(record: dict, keys: tuple[str, ...]):
    for key in keys:
        if key in record:
            return record[key]
    return None


def normalize_documents(raw, domain: str, require_labels: bool) -> list[dict]:
    """Accept a list of documents or an id->document mapping; keep the original order."""
    if isinstance(raw, dict):
        for wrapper in ("data", "documents", "docs"):
            if wrapper in raw and isinstance(raw[wrapper], (list, dict)):
                raw = raw[wrapper]
                break
    if isinstance(raw, dict):
        items = []
        for key, value in raw.items():
            if isinstance(value, list):
                value = {"text": value}
            value = dict(value)
            value.setdefault("id", key)
            items.append(value)
    else:
        items = list(raw)
    documents = []
    for position, item in enumerate(items):
        doc_id = _pick(item, ("id", "doc_id", "document_id", "docid", "name"))
        if doc_id is None:
            raise ValueError(f"{domain} document #{position} has no id field: keys={list(item)}")
        sentences = _pick(item, ("text", "sentences", "sents", "sentence"))
        if not isinstance(sentences, list):
            raise ValueError(f"{domain}/{doc_id}: expected a list of sentences, got {type(sentences)}")
        sentences = [str(sentence) for sentence in sentences]
        labels = _pick(item, ("labels", "label", "roles", "rhetorical_roles"))
        if require_labels:
            if not isinstance(labels, list) or len(labels) != len(sentences):
                raise ValueError(f"{domain}/{doc_id}: labels missing or length mismatch")
            labels = [str(label) for label in labels]
        else:
            labels = None
        documents.append({"id": str(doc_id), "domain": domain, "sentences": sentences, "labels": labels})
    return documents


def load_label_vocab(path: Path) -> list[str]:
    raw = read_json(path)
    if isinstance(raw, dict):
        for wrapper in ("labels", "label_vocab", "vocab"):
            if wrapper in raw:
                raw = raw[wrapper]
                break
    if isinstance(raw, dict):
        if all(isinstance(value, int) for value in raw.values()):
            labels = [label for label, _ in sorted(raw.items(), key=lambda item: item[1])]
        else:
            labels = [raw[key] for key in sorted(raw, key=lambda key: int(key))]
    else:
        labels = list(raw)
    labels = [str(label) for label in labels]
    if len(set(labels)) != len(labels):
        raise ValueError("Duplicate labels in label_vocab.json")
    if set(labels) != set(OFFICIAL_LABELS):
        print("WARNING: label_vocab.json differs from the competition document list:",
              sorted(set(labels) ^ set(OFFICIAL_LABELS)), flush=True)
    return labels


def check_labels(documents: list[dict], labels: list[str]) -> None:
    allowed = set(labels)
    unknown = Counter(label for doc in documents for label in doc["labels"] if label not in allowed)
    if unknown:
        raise ValueError(f"Gold labels outside label_vocab.json: {dict(unknown)}")


# ----------------------------------------------------------------------------- examples

class SentenceDataset(Dataset):
    """One example per sentence; inputs are built from cached per-sentence token ids."""

    def __init__(self, documents, tokenizer, label_to_id, args, separator_id):
        self.documents = documents
        self.args = args
        self.separator_id = separator_id
        self.examples = []
        self.token_cache = []
        for doc_index, document in enumerate(documents):
            encodings = tokenizer.encode_batch(document["sentences"]) if document["sentences"] else []
            self.token_cache.append([list(encoding.ids) for encoding in encodings])
            total = len(document["sentences"])
            for sentence_index in range(total):
                label = None
                if document["labels"] is not None:
                    label = label_to_id[document["labels"][sentence_index]]
                self.examples.append({
                    "doc": doc_index,
                    "index": sentence_index,
                    "domain": DOMAINS.index(document["domain"]),
                    "bucket": min(POSITION_BUCKETS - 1, int(POSITION_BUCKETS * sentence_index / max(1, total))),
                    "label": label,
                    "key": f"{document['domain']}::{document['id']}::{sentence_index}",
                })

    def __len__(self):
        return len(self.examples)

    def build(self, doc_index: int, sentence_index: int) -> tuple[list[int], int]:
        """Layout: [previous context] SEP [next context] SEP target (target always last)."""
        args = self.args
        sentences = self.token_cache[doc_index]
        target = sentences[sentence_index][: args.target_tokens] or [self.separator_id]
        if args.context == "none":
            return target, len(target)
        budget = args.max_length - len(target) - 2
        previous, following = [], []
        for offset in range(1, args.context_sentences + 1):
            for side in ("prev", "next"):
                position = sentence_index - offset if side == "prev" else sentence_index + offset
                if not 0 <= position < len(sentences) or budget <= 0:
                    continue
                piece = sentences[position][: min(args.context_tokens, budget)]
                budget -= len(piece)
                if side == "prev":
                    previous.insert(0, piece)
                else:
                    following.append(piece)
        ids = [token for piece in previous for token in piece] + [self.separator_id]
        ids += [token for piece in following for token in piece] + [self.separator_id]
        ids += target
        return ids, len(target)

    def __getitem__(self, item):
        example = self.examples[item]
        ids, target_length = self.build(example["doc"], example["index"])
        return {**example, "ids": ids, "target_length": target_length, "row": item}


def make_collate(pad_id: int):
    def collate(batch):
        length = max(len(item["ids"]) for item in batch)
        ids = torch.full((len(batch), length), pad_id, dtype=torch.long)
        mask = torch.zeros((len(batch), length), dtype=torch.bool)
        target = torch.zeros((len(batch), length), dtype=torch.bool)
        for row, item in enumerate(batch):
            size = len(item["ids"])
            ids[row, :size] = torch.tensor(item["ids"], dtype=torch.long)
            mask[row, :size] = True
            target[row, size - item["target_length"]: size] = True
        labels = None
        if batch[0]["label"] is not None:
            labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        return {
            "ids": ids,
            "mask": mask,
            "target": target,
            "last": torch.tensor([len(item["ids"]) - 1 for item in batch], dtype=torch.long),
            "domain": torch.tensor([item["domain"] for item in batch], dtype=torch.long),
            "bucket": torch.tensor([item["bucket"] for item in batch], dtype=torch.long),
            "labels": labels,
            "rows": [item["row"] for item in batch],
            "keys": [item["key"] for item in batch],
        }
    return collate


# ----------------------------------------------------------------------------- model

class RRModel(nn.Module):
    """Phase 1 backbone -> [mean(target tokens); last target token; position; domain] -> MLP."""

    def __init__(self, backbone: nn.Module, model_name: str, label_count: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = LNERModel(backbone, model_name, 1)
        del self.encoder.classifier
        dim = backbone.cfg.d_model
        self.position = nn.Embedding(POSITION_BUCKETS, 32)
        self.domain = nn.Embedding(len(DOMAINS), 16)
        self.head = nn.Sequential(
            nn.LayerNorm(2 * dim + 48),
            nn.Dropout(dropout),
            nn.Linear(2 * dim + 48, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, label_count),
        )

    def head_parameters(self):
        return [*self.position.parameters(), *self.domain.parameters(), *self.head.parameters()]

    def forward(self, batch):
        hidden = self.encoder.hidden(batch["ids"], batch["mask"]).float()
        weights = batch["target"].unsqueeze(-1).float()
        mean = (hidden * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        last = hidden[torch.arange(hidden.size(0), device=hidden.device), batch["last"]]
        features = torch.cat((mean, last, self.position(batch["bucket"]), self.domain(batch["domain"])), dim=-1)
        return self.head(features)


class LoRALinear(nn.Module):
    """LoRA: W0 x + (alpha / r) B A x, with W0 frozen."""

    def __init__(self, base: nn.Linear, rank: int = 8, alpha: float = 16.0, dropout: float = 0.05):
        super().__init__()
        self.base = base
        self.a = nn.Linear(base.in_features, rank, bias=False)
        self.b = nn.Linear(rank, base.out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.a.weight, a=math.sqrt(5))
        nn.init.zeros_(self.b.weight)
        for parameter in self.base.parameters():
            parameter.requires_grad = False

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, inputs):
        return self.base(inputs) + self.scale * self.b(self.a(self.dropout(inputs)))


class IA3Linear(nn.Module):
    """IA3: learned element-wise rescaling l * (W0 x) of a frozen projection."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.scale = nn.Parameter(torch.ones(base.out_features))
        for parameter in self.base.parameters():
            parameter.requires_grad = False

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, inputs):
        return self.base(inputs) * self.scale


class LoRMAWrapped(LoRMAPlusLinear):
    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias


def transformer_layers(backbone: nn.Module, model_name: str):
    if model_name == "decoder":
        return list(backbone.blocks.layers)
    if model_name == "encoder_decoder":
        return list(backbone.encoder.layers) + list(backbone.decoder.layers)
    if model_name == "hybrid":
        return list(backbone.decoder.layers)
    raise ValueError(model_name)


def apply_adapter(model: RRModel, model_name: str, method: str, rank: int) -> None:
    if method in {"full", "distill"}:
        return
    backbone = model.encoder.backbone
    for parameter in backbone.parameters():
        parameter.requires_grad = False

    def wrap(linear: nn.Linear, role: str) -> nn.Module:
        if method == "lorma":
            return LoRMAWrapped(linear, rank=rank, alpha=float(rank))
        if method == "lora":
            return LoRALinear(linear, rank=rank, alpha=2.0 * rank)
        if method == "ia3":
            return IA3Linear(linear)
        raise ValueError(method)

    if model_name == "efficient":
        for block in backbone.blocks:
            attention = block.attention
            if method == "ia3":       # IA3 rescales keys, values and the FFN intermediate activation
                attention.k, attention.v, block.up = wrap(attention.k, "k"), wrap(attention.v, "v"), wrap(block.up, "ffn")
            else:
                attention.q, attention.v = wrap(attention.q, "q"), wrap(attention.v, "v")
                block.down = wrap(block.down, "ffn")
    else:
        for layer in transformer_layers(backbone, model_name):
            layer.linear1 = wrap(layer.linear1, "ffn")
            if method != "ia3":
                layer.linear2 = wrap(layer.linear2, "ffn")
    for module_name in ("norm", "fusion"):
        module = getattr(backbone, module_name, None)
        if module is not None:
            for parameter in module.parameters():
                parameter.requires_grad = True


# ----------------------------------------------------------------------------- metrics

def macro_scores(gold: list[str], predicted: list[str]) -> dict:
    """sklearn-style macro P/R/F1 over labels appearing in gold or predictions."""
    labels = sorted(set(gold) | set(predicted))
    per_label = {}
    for label in labels:
        tp = sum(1 for g, p in zip(gold, predicted) if g == label and p == label)
        fp = sum(1 for g, p in zip(gold, predicted) if g != label and p == label)
        fn = sum(1 for g, p in zip(gold, predicted) if g == label and p != label)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {"precision": precision, "recall": recall, "f1": f1, "support": tp + fn}
    count = max(1, len(labels))
    return {
        "macro_f1": sum(row["f1"] for row in per_label.values()) / count,
        "macro_precision": sum(row["precision"] for row in per_label.values()) / count,
        "macro_recall": sum(row["recall"] for row in per_label.values()) / count,
        "accuracy": sum(g == p for g, p in zip(gold, predicted)) / max(1, len(gold)),
        "per_label": per_label,
    }


@torch.no_grad()
def predict_logits(model, loader, device) -> torch.Tensor:
    model.eval()
    outputs = torch.zeros(len(loader.dataset), model.head[-1].out_features)
    for batch in loader:
        tensors = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(tensors)
        outputs[batch["rows"]] = logits.float().cpu()
    return outputs


def evaluate(dataset: SentenceDataset, logits: torch.Tensor, labels: list[str]) -> dict:
    predicted = [labels[index] for index in logits.argmax(-1).tolist()]
    gold = [labels[example["label"]] for example in dataset.examples]
    result = macro_scores(gold, predicted)
    for domain_index, domain in enumerate(DOMAINS):
        rows = [row for row, example in enumerate(dataset.examples) if example["domain"] == domain_index]
        if rows:
            scores = macro_scores([gold[row] for row in rows], [predicted[row] for row in rows])
            result[f"{domain}_macro_f1"] = scores["macro_f1"]
    confusion = defaultdict(Counter)
    for g, p in zip(gold, predicted):
        confusion[g][p] += 1
    result["confusion"] = {g: dict(row) for g, row in confusion.items()}
    return result


def class_weights(dataset: SentenceDataset, count: int, power: float) -> torch.Tensor:
    frequency = Counter(example["label"] for example in dataset.examples)
    largest = max(frequency.values())
    weights = torch.ones(count)
    for index in range(count):
        weights[index] = min(10.0, (largest / max(1, frequency[index])) ** power)
    return weights / weights.mean()


# ----------------------------------------------------------------------------- training

def train(model, train_loader, dev_loader, labels, device, args, soft_targets=None):
    weights = class_weights(train_loader.dataset, len(labels), args.class_weight_power).to(device)
    head = [p for p in model.head_parameters() if p.requires_grad]
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head, "lr": args.head_lr}]
    if body:
        groups.append({"params": body, "lr": args.learning_rate})
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    steps_total = max(1, args.epochs * math.ceil(len(train_loader) / args.grad_accum))
    warmup = max(1, int(0.06 * steps_total))
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: min((step + 1) / warmup, max(0.05, (steps_total - step) / max(1, steps_total - warmup))),
    )
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    trainable = [p for p in model.parameters() if p.requires_grad]
    start = time.time()
    budget = args.time_budget_min * 60
    best = {"macro_f1": -1.0}
    best_state, history, out_of_time = None, [], False

    def checkpoint(epoch, step, mean_loss):
        nonlocal best, best_state
        logits = predict_logits(model, dev_loader, device)
        scores = evaluate(dev_loader.dataset, logits, labels)
        row = {
            "epoch": epoch, "step": step, "train_loss": mean_loss,
            "dev_macro_f1": scores["macro_f1"], "dev_macro_precision": scores["macro_precision"],
            "dev_macro_recall": scores["macro_recall"], "dev_accuracy": scores["accuracy"],
            "CL_macro_f1": scores.get("CL_macro_f1"), "IT_macro_f1": scores.get("IT_macro_f1"),
            "minutes": (time.time() - start) / 60,
        }
        history.append(row)
        print(json.dumps(row), flush=True)
        if scores["macro_f1"] > best["macro_f1"]:
            best = scores
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        model.train()

    evals_per_epoch = max(1, args.evals_per_epoch)
    eval_every = max(1, len(train_loader) // evals_per_epoch)
    for epoch in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, batch in enumerate(train_loader, 1):
            tensors = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(tensors).float()
            hard = F.cross_entropy(logits, tensors["labels"], weight=weights, label_smoothing=args.label_smoothing)
            if soft_targets is None:
                loss = hard
            else:
                teacher = soft_targets[batch["rows"]].to(device)
                t = args.temperature
                soft = F.kl_div(
                    F.log_softmax(logits / t, dim=-1), F.softmax(teacher / t, dim=-1), reduction="batchmean",
                ) * t * t
                loss = args.kd_alpha * hard + (1.0 - args.kd_alpha) * soft
            scaler.scale(loss / args.grad_accum).backward()
            if step % args.grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            losses.append(float(loss))
            if step % 100 == 0:
                print(f"epoch {epoch} step {step}/{len(train_loader)} loss {sum(losses[-100:]) / len(losses[-100:]):.4f} "
                      f"elapsed {(time.time() - start) / 60:.1f} min", flush=True)
            if time.time() - start > budget:
                out_of_time = True
                print("Time budget reached; evaluating and stopping.", flush=True)
                break
            if evals_per_epoch > 1 and step % eval_every == 0 and step < len(train_loader):
                checkpoint(epoch, step, sum(losses) / len(losses))
        checkpoint(epoch, len(losses), sum(losses) / max(1, len(losses)))
        if out_of_time:
            break
    model.load_state_dict(best_state)
    return best, history, time.time() - start


# ----------------------------------------------------------------------------- outputs

def write_submission(test_datasets, test_logits, labels, output_dir: Path) -> Path:
    combined, per_domain = {}, {}
    collisions = []
    for domain, dataset in test_datasets.items():
        logits = test_logits[domain]
        domain_predictions = {}
        for document in dataset.documents:
            domain_predictions[document["id"]] = [None] * len(document["sentences"])
        for row, example in enumerate(dataset.examples):
            document = dataset.documents[example["doc"]]
            domain_predictions[document["id"]][example["index"]] = labels[int(logits[row].argmax())]
        for document in dataset.documents:
            predictions = domain_predictions[document["id"]]
            if len(predictions) != len(document["sentences"]) or any(p not in labels for p in predictions):
                raise RuntimeError(f"Invalid prediction list for {domain}/{document['id']}")
            if document["id"] in combined:
                collisions.append(document["id"])
            combined[document["id"]] = predictions
        per_domain[domain] = domain_predictions
        (output_dir / f"prediction_{domain}.json").write_text(json.dumps(domain_predictions), encoding="utf-8")
    if collisions:
        raise RuntimeError(f"CL and IT share document ids {collisions[:5]}; check the Codabench format")
    expected = sum(len(dataset.documents) for dataset in test_datasets.values())
    if len(combined) != expected:
        raise RuntimeError("Document count mismatch")
    path = output_dir / "prediction.json"
    path.write_text(json.dumps(combined, ensure_ascii=False), encoding="utf-8")
    archive = output_dir / f"submission_{output_dir.name}.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(path, arcname="prediction.json")
    with zipfile.ZipFile(archive) as bundle:
        assert bundle.namelist() == ["prediction.json"]
        loaded = json.loads(bundle.read("prediction.json"))
    print(f"Wrote {archive} with {len(loaded)} documents / {sum(map(len, loaded.values()))} sentences", flush=True)
    return archive


# ----------------------------------------------------------------------------- main

def run(args):
    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    name = args.run_name or f"phase3_{args.model}_{args.method}_{args.context}"
    output_dir = Path(args.output_dir or f"/kaggle/working/{name}")
    output_dir.mkdir(parents=True, exist_ok=True)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    checkpoint, tokenizer_path, artifact = locate_phase1_files(args.model, args.artifact_zip, output_dir / "phase1")
    files = obtain_data(output_dir / "data", args.data_dir)
    labels = load_label_vocab(files["label_vocab.json"])
    label_to_id = {label: index for index, label in enumerate(labels)}

    def documents(split, require_labels):
        docs = []
        for domain in DOMAINS:
            docs += normalize_documents(read_json(files[f"{domain}_{split}.json"]), domain, require_labels)
        if args.limit_docs:
            docs = [d for domain in DOMAINS for d in [x for x in docs if x["domain"] == domain][: args.limit_docs]]
        return docs

    train_docs, dev_docs = documents("train", True), documents("dev", True)
    check_labels(train_docs, labels)
    check_labels(dev_docs, labels)
    test_docs = {domain: normalize_documents(read_json(files[f"{domain}_test.json"]), domain, False) for domain in DOMAINS}
    shared = {d["id"] for d in test_docs["CL"]} & {d["id"] for d in test_docs["IT"]}
    if shared:  # fail fast, before any training time is spent
        raise RuntimeError(f"CL and IT test sets share document ids {sorted(shared)[:5]}; one prediction.json cannot hold both")
    print({d: f"{len(v)} test docs" for d, v in test_docs.items()}, "| train docs", len(train_docs), "| dev docs", len(dev_docs), flush=True)
    print("Train label distribution:", dict(Counter(l for d in train_docs for l in d["labels"]).most_common()), flush=True)

    from tokenizers import Tokenizer

    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    pad_id = tokenizer.token_to_id("<pad>")
    separator = tokenizer.token_to_id("<eos>")
    if pad_id is None or separator is None:
        raise ValueError("Phase 1 tokenizer must contain <pad> and <eos>")

    train_set = SentenceDataset(train_docs, tokenizer, label_to_id, args, separator)
    dev_set = SentenceDataset(dev_docs, tokenizer, label_to_id, args, separator)
    test_sets = {d: SentenceDataset(test_docs[d], tokenizer, label_to_id, args, separator) for d in DOMAINS}
    collate = make_collate(pad_id)
    loader = lambda ds, shuffle, bs: DataLoader(ds, batch_size=bs, shuffle=shuffle, collate_fn=collate, num_workers=2)
    train_loader = loader(train_set, True, args.batch_size)
    dev_loader = loader(dev_set, False, args.batch_size * 2)

    backbone, phase1_config, phase1_saved = load_phase1(checkpoint, args.model)
    phase1_parameters = sum(p.numel() for p in backbone.parameters())
    if args.method == "distill":
        if args.model != "decoder":
            raise ValueError("The distillation student recipe uses the decoder Phase 1 model")
        backbone = make_student(backbone, layers=args.student_layers)
    model = RRModel(backbone, args.model, len(labels))
    apply_adapter(model, args.model, args.method, args.rank)
    model.to(device)

    soft_targets = None
    if args.method == "distill":
        if not args.teacher_soft_labels:
            raise ValueError("--teacher-soft-labels is required for distillation")
        saved = torch.load(args.teacher_soft_labels, map_location="cpu")
        if saved["labels"] != labels:
            raise ValueError("Teacher label order differs")
        index = {key: row for row, key in enumerate(saved["keys"])}
        missing = [e["key"] for e in train_set.examples if e["key"] not in index]
        if missing:
            raise ValueError(f"Teacher soft labels miss {len(missing)} training sentences, e.g. {missing[:3]}")
        soft_targets = saved["logits"][[index[e["key"]] for e in train_set.examples]]
        print(f"Loaded teacher soft labels from {saved.get('teacher')} ({len(index)} sentences)", flush=True)

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    summary = {
        "run": name, "phase1_artifact": artifact, "model": args.model, "method": args.method,
        "context": args.context, "phase1_parameters": phase1_parameters, "parameters": total,
        "trainable_parameters": trainable, "trainable_fraction": trainable / total,
        "train_sentences": len(train_set), "dev_sentences": len(dev_set),
        "test_sentences": {d: len(s) for d, s in test_sets.items()}, "device": str(device),
        "config": vars(args),
    }
    print(json.dumps(summary, indent=2, default=str), flush=True)

    best, history, seconds = train(model, train_loader, dev_loader, labels, device, args, soft_targets)
    print(json.dumps({k: best[k] for k in ("macro_f1", "macro_precision", "macro_recall", "accuracy")}), flush=True)

    test_logits = {d: predict_logits(model, loader(s, False, args.batch_size * 2), device) for d, s in test_sets.items()}
    archive = write_submission(test_sets, test_logits, labels, output_dir)

    if args.export_soft_labels:
        train_eval_loader = loader(train_set, False, args.batch_size * 2)
        logits = predict_logits(model, train_eval_loader, device)
        torch.save({
            "keys": [e["key"] for e in train_set.examples], "logits": logits, "labels": labels, "teacher": name,
        }, output_dir / "teacher_soft_labels.pt")
        print("Saved", output_dir / "teacher_soft_labels.pt", flush=True)

    if args.save_checkpoint:
        torch.save({"model": model.state_dict(), "phase1_config": asdict(phase1_config),
                    "run_config": vars(args), "labels": labels}, output_dir / "best.pt")
    (output_dir / "training_log.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    (output_dir / "validation_metrics.json").write_text(json.dumps(best, indent=2), encoding="utf-8")
    manifest = {
        **summary, "best_dev_macro_f1": best["macro_f1"], "best_dev_macro_precision": best["macro_precision"],
        "best_dev_macro_recall": best["macro_recall"], "dev_CL_macro_f1": best.get("CL_macro_f1"),
        "dev_IT_macro_f1": best.get("IT_macro_f1"), "train_minutes": seconds / 60,
        "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None,
        "phase1_steps": phase1_saved.get("total_steps"), "submission": str(archive),
    }
    (output_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    print(json.dumps({k: manifest[k] for k in ("run", "best_dev_macro_f1", "dev_CL_macro_f1", "dev_IT_macro_f1",
                                                "trainable_parameters", "train_minutes", "submission")}, indent=2))


def parse_args(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True, choices=("decoder", "encoder_decoder", "hybrid", "efficient"))
    parser.add_argument("--method", required=True, choices=("full", "lorma", "lora", "ia3", "distill"))
    parser.add_argument("--context", default="window", choices=("none", "window"))
    parser.add_argument("--context-sentences", type=int, default=2)
    parser.add_argument("--context-tokens", type=int, default=48)
    parser.add_argument("--target-tokens", type=int, default=96)
    parser.add_argument("--max-length", type=int, default=256)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--grad-accum", type=int, default=1)
    parser.add_argument("--learning-rate", type=float, default=3e-5)
    parser.add_argument("--head-lr", type=float, default=1e-3)
    parser.add_argument("--rank", type=int, default=8)
    parser.add_argument("--class-weight-power", type=float, default=0.5)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--student-layers", type=int, default=8)
    parser.add_argument("--teacher-soft-labels")
    parser.add_argument("--kd-alpha", type=float, default=0.5)
    parser.add_argument("--temperature", type=float, default=2.0)
    parser.add_argument("--export-soft-labels", action="store_true")
    parser.add_argument("--save-checkpoint", action="store_true")
    parser.add_argument("--time-budget-min", type=float, default=30.0)
    parser.add_argument("--evals-per-epoch", type=int, default=1)
    parser.add_argument("--artifact-zip")
    parser.add_argument("--data-dir")
    parser.add_argument("--output-dir")
    parser.add_argument("--run-name")
    parser.add_argument("--limit-docs", type=int, default=0)
    parser.add_argument("--seed", type=int, default=230338)
    return parser.parse_args(argv)


if __name__ == "__main__":
    run(parse_args())


In [ ]:
%%writefile /kaggle/working/phase4_cjpe.py
"""CS781 Phase 4 - Court Judgment Prediction (CJPE) from Phase 1 legal language models.

Every experiment starts from a Phase 1 checkpoint (decoder / encoder_decoder / hybrid / efficient);
no other pretrained weights are used.  Training uses only the official Phase 4 CJPE train/dev data.

Long-document strategy (hierarchical, section-aware):
  * the case is tokenised with the Phase 1 tokenizer and cut into chunks of 256 tokens (the Phase 1
    context length);
  * ``--selection headtail`` keeps the first chunk (parties / facts / background) plus the last
    K-1 chunks (arguments, analysis and the operative part, where the outcome is decided);
    ``--selection tail`` keeps the last K chunks;
  * the Phase 1 backbone encodes every chunk -> [masked mean ; last token] chunk vector
    + learned chunk-slot embedding;
  * attention pooling across chunks (+ mean of chunks) -> MLP -> 2 logits.
Adaptation: full fine-tuning, LORMA+, LoRA, IA3, partial (top-layer) fine-tuning, BitFit and
knowledge distillation (truncated 8-layer decoder student trained on a teacher's soft labels).
Post-processing: the decision threshold on P(label=1) is tuned on dev for Macro-F1.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import time
import zipfile
from collections import Counter
from dataclasses import asdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from phase2_lner import LNERModel, load_phase1, make_student
from phase3_rr import apply_adapter, locate_phase1_files, seed_everything, transformer_layers

REPO_ID = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_4_CJPE"


# ----------------------------------------------------------------------------- data

def obtain_data(destination: Path, data_dir: str | None) -> dict[str, Path]:
    """train.json / dev.json / test.json from --data-dir or the official HF dataset (always the latest version)."""
    names = ("train.json", "dev.json", "test.json")
    if data_dir:
        found = {}
        for folder, _, files in os.walk(data_dir, followlinks=True):
            for name in files:
                if name in names and name not in found:
                    found[name] = Path(folder) / name
        if len(found) == 3:
            return found
        raise FileNotFoundError(f"--data-dir lacks {set(names) - set(found)}")
    from huggingface_hub import hf_hub_download, list_repo_files

    token = os.environ.get("HF_TOKEN")
    if not token:
        raise RuntimeError("HF_TOKEN is missing: add the Kaggle secret HF and enable it")
    repo_files = [f for f in list_repo_files(REPO_ID, repo_type="dataset", token=token) if f.startswith(PHASE_DIR + "/")]
    print("Phase 4 files on HF:", repo_files, flush=True)
    found = {}
    for name in names:
        matches = [f for f in repo_files if f.lower().endswith("/" + name)]
        if not matches:
            raise FileNotFoundError(f"{name} not found under {PHASE_DIR} on HF: {repo_files}")
        path = Path(hf_hub_download(repo_id=REPO_ID, filename=matches[0], repo_type="dataset", token=token,
                                    force_download=True))  # never reuse a cached (pre-fix, leaked) copy
        found[name] = path
    return found


def _pick(record: dict, keys):
    for key in keys:
        if key in record and record[key] is not None:
            return record[key]
    return None


def load_cases(path: Path, require_labels: bool) -> list[dict]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(raw, dict):
        for wrapper in ("data", "cases", "documents", "examples"):
            if isinstance(raw.get(wrapper), (list, dict)):
                raw = raw[wrapper]
                break
    if isinstance(raw, dict):
        raw = [{**(v if isinstance(v, dict) else {"text": v}), "id": v.get("id", k) if isinstance(v, dict) else k}
               for k, v in raw.items()]
    cases = []
    for position, item in enumerate(raw):
        case_id = _pick(item, ("id", "case_id", "uid", "name", "doc_id"))
        text = _pick(item, ("text", "case_text", "document", "raw_text", "content"))
        if case_id is None or text is None:
            raise ValueError(f"{path.name} record #{position}: cannot find id/text, keys={list(item)}")
        if isinstance(text, list):
            text = " ".join(map(str, text))
        label = _pick(item, ("label", "labels", "decision", "outcome", "judgment", "pred"))
        if require_labels:
            if isinstance(label, list):
                label = label[0]
            label = int(label)
            if label not in (0, 1):
                raise ValueError(f"{path.name}/{case_id}: label {label} is not 0/1")
        else:
            label = None
        cases.append({"id": case_id, "text": str(text), "label": label})
    print(f"{path.name}: {len(cases)} cases; keys of first record: {list(raw[0]) if raw else []}", flush=True)
    return cases


def text_key(text: str) -> str:
    norm = " ".join(text.split()).lower()
    return hashlib.sha1(norm.encode("utf-8")).hexdigest()


def leakage_report(train, dev, test) -> dict:
    """Exact-duplicate check (by id and by whitespace/case-normalised text). Overlaps are removed from
    train/dev so that nothing that also appears in the test set is ever trained on or used for selection."""
    test_ids = {c["id"] for c in test}
    test_keys = {text_key(c["text"]) for c in test}
    dev_keys = {text_key(c["text"]) for c in dev}
    report = {
        "train_test_id_overlap": sum(c["id"] in test_ids for c in train),
        "dev_test_id_overlap": sum(c["id"] in test_ids for c in dev),
        "train_test_text_overlap": sum(text_key(c["text"]) in test_keys for c in train),
        "dev_test_text_overlap": sum(text_key(c["text"]) in test_keys for c in dev),
        "train_dev_text_overlap": sum(text_key(c["text"]) in dev_keys for c in train),
    }
    train[:] = [c for c in train if c["id"] not in test_ids and text_key(c["text"]) not in test_keys
                and text_key(c["text"]) not in dev_keys]
    dev[:] = [c for c in dev if c["id"] not in test_ids and text_key(c["text"]) not in test_keys]
    report["train_after_filter"], report["dev_after_filter"] = len(train), len(dev)
    print("LEAKAGE CHECK:", json.dumps(report), flush=True)
    return report


def encode_cases(cases, tokenizer, args):
    """-> list of chunk lists (token ids). Only the needed head/tail words are tokenised (speed)."""
    L, K = args.chunk_len, args.max_chunks
    tail_words = int(K * L * 1.1) + 50
    head_words = int(L * 1.1) + 20
    heads, tails = [], []
    for case in cases:
        words = case["text"].split()
        heads.append(" ".join(words[:head_words]))
        tails.append(" ".join(words[-tail_words:]))
    tail_ids = [e.ids for e in tokenizer.encode_batch(tails)]
    head_ids = [e.ids for e in tokenizer.encode_batch(heads)] if args.selection == "headtail" else None
    out = []
    for i, case in enumerate(cases):
        ids = tail_ids[i]
        n_words = len(case["text"].split())
        if args.selection == "headtail" and n_words > tail_words:
            tail = ids[-(K - 1) * L:]
            chunks = [head_ids[i][:L]] + [tail[j:j + L] for j in range(0, len(tail), L)]
        else:
            ids = ids[-K * L:]
            chunks = [ids[j:j + L] for j in range(0, len(ids), L)]
        chunks = [c for c in chunks if c] or [[args.eos_id]]
        out.append(chunks[:K])
    return out


def make_batch(chunk_lists, pad_id, device):
    flat = [c for chunks in chunk_lists for c in chunks]
    length = max(len(c) for c in flat)
    ids = torch.full((len(flat), length), pad_id, dtype=torch.long)
    mask = torch.zeros((len(flat), length), dtype=torch.bool)
    for r, c in enumerate(flat):
        ids[r, :len(c)] = torch.tensor(c)
        mask[r, :len(c)] = True
    K = max(len(chunks) for chunks in chunk_lists)
    slot_valid = torch.zeros((len(chunk_lists), K), dtype=torch.bool)
    for i, chunks in enumerate(chunk_lists):
        slot_valid[i, :len(chunks)] = True
    return ids.to(device), mask.to(device), slot_valid.to(device)


# ----------------------------------------------------------------------------- model

class CJPEModel(nn.Module):
    def __init__(self, backbone, model_name, max_chunks, dropout=0.1):
        super().__init__()
        self.encoder = LNERModel(backbone, model_name, 1)
        del self.encoder.classifier
        d = backbone.cfg.d_model
        self.chunk_proj = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d), nn.GELU())
        self.slot = nn.Embedding(max_chunks, d)
        self.attn = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
        self.head = nn.Sequential(nn.LayerNorm(2 * d), nn.Dropout(dropout), nn.Linear(2 * d, 256), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(256, 2))

    def head_parameters(self):
        return [*self.chunk_proj.parameters(), *self.slot.parameters(), *self.attn.parameters(), *self.head.parameters()]

    def forward(self, ids, mask, slot_valid):
        hidden = self.encoder.hidden(ids, mask).float()
        w = mask.unsqueeze(-1).float()
        mean = (hidden * w).sum(1) / w.sum(1).clamp_min(1)
        last = hidden[torch.arange(hidden.size(0), device=hidden.device), mask.sum(1) - 1]
        vectors = self.chunk_proj(torch.cat((mean, last), -1))
        B, K = slot_valid.shape
        chunks = torch.zeros(B, K, vectors.size(-1), device=vectors.device, dtype=vectors.dtype)
        chunks[slot_valid] = vectors
        chunks = chunks + self.slot(torch.arange(K, device=ids.device))[None]
        scores = self.attn(chunks).squeeze(-1).masked_fill(~slot_valid, -1e4)
        alpha = scores.softmax(-1)
        pooled = (alpha.unsqueeze(-1) * chunks).sum(1)
        mean_chunks = (chunks * slot_valid.unsqueeze(-1)).sum(1) / slot_valid.sum(1, keepdim=True).clamp_min(1)
        return self.head(torch.cat((pooled, mean_chunks), -1)), alpha


def apply_method(model: CJPEModel, model_name: str, method: str, rank: int, unfreeze: int):
    backbone = model.encoder.backbone
    if method in ("full", "distill"):
        return
    if method in ("lorma", "lora", "ia3"):
        apply_adapter(model, model_name, method, rank)
        return
    for p in backbone.parameters():
        p.requires_grad = False
    if method == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith("bias"):
                p.requires_grad = True
    elif method == "partial":
        if model_name == "efficient":
            groups = [list(backbone.blocks)]
        elif model_name == "encoder_decoder":
            groups = [list(backbone.encoder.layers), list(backbone.decoder.layers)]
        else:
            groups = [transformer_layers(backbone, model_name)]
        for layers in groups:
            for layer in layers[-unfreeze:]:
                for p in layer.parameters():
                    p.requires_grad = True
    else:
        raise ValueError(method)
    for module_name in ("norm", "fusion"):
        module = getattr(backbone, module_name, None)
        if module is not None:
            for p in module.parameters():
                p.requires_grad = True


# ----------------------------------------------------------------------------- metrics / inference

def binary_scores(gold, pred) -> dict:
    out = {}
    for c in (0, 1):
        tp = sum(g == c and p == c for g, p in zip(gold, pred))
        fp = sum(g != c and p == c for g, p in zip(gold, pred))
        fn = sum(g == c and p != c for g, p in zip(gold, pred))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        out[c] = (pr, rc, 2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    tp1 = sum(g == 1 and p == 1 for g, p in zip(gold, pred))
    return {
        "macro_f1": (out[0][2] + out[1][2]) / 2, "macro_precision": (out[0][0] + out[1][0]) / 2,
        "macro_recall": (out[0][1] + out[1][1]) / 2, "accuracy": sum(g == p for g, p in zip(gold, pred)) / max(1, len(gold)),
        "f1_class0": out[0][2], "f1_class1": out[1][2], "precision_class1": out[1][0], "recall_class1": out[1][1],
        "false_positives": sum(g == 0 and p == 1 for g, p in zip(gold, pred)),
        "false_negatives": sum(g == 1 and p == 0 for g, p in zip(gold, pred)), "true_positives": tp1,
    }


@torch.no_grad()
def predict(model, chunk_lists, pad_id, device, batch_docs):
    model.eval()
    logits, attentions = [], []
    order = sorted(range(len(chunk_lists)), key=lambda i: -sum(map(len, chunk_lists[i])))
    result = [None] * len(chunk_lists)
    for s in range(0, len(order), batch_docs):
        rows = order[s:s + batch_docs]
        ids, mask, valid = make_batch([chunk_lists[r] for r in rows], pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            out, alpha = model(ids, mask, valid)
        out, alpha = out.float().cpu(), alpha.float().cpu()
        for j, r in enumerate(rows):
            result[r] = (out[j], alpha[j, :len(chunk_lists[r])])
    return torch.stack([r[0] for r in result]), [r[1] for r in result]


def tune_threshold(probs, gold):
    best_t, best_f = 0.5, binary_scores(gold, [int(p >= 0.5) for p in probs])["macro_f1"]
    for t in [x / 100 for x in range(10, 91)]:
        f = binary_scores(gold, [int(p >= t) for p in probs])["macro_f1"]
        if f > best_f + 1e-9:
            best_t, best_f = t, f
    return best_t, best_f


# ----------------------------------------------------------------------------- training

def train(model, train_chunks, train_labels, dev_chunks, dev_labels, args, device, pad_id, soft=None):
    counts = Counter(train_labels)
    largest = max(counts.values())
    weights = torch.tensor([(largest / max(1, counts[c])) ** args.class_weight_power for c in (0, 1)])
    weights = (weights / weights.mean()).to(device)
    print("train label counts", dict(counts), "class weights", weights.tolist(), flush=True)
    head_ids = {id(p) for p in model.head_parameters()}
    head = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head, "lr": args.head_lr}] + ([{"params": body, "lr": args.learning_rate}] if body else [])
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    budget = args.time_budget_min * 60
    # schedule over the number of steps the time budget is expected to allow (re-estimated after warm-up)
    est_steps = max(50, args.epochs * math.ceil(len(train_chunks) / (args.batch_docs * args.grad_accum)))
    warmup = 30
    sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min((s + 1) / warmup, max(0.1, 1 - s / est_steps)))
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
    trainable = [p for p in model.parameters() if p.requires_grad]
    start, last_eval = time.time(), time.time()
    best, best_state, history = {"macro_f1": -1}, None, []
    step, opt_steps, seen = 0, 0, 0

    def evaluate(tag):
        nonlocal best, best_state, last_eval
        logits, _ = predict(model, dev_chunks, pad_id, device, args.batch_docs * 2)
        pred = logits.argmax(-1).tolist()
        scores = binary_scores(dev_labels, pred)
        row = {"tag": tag, "docs_seen": seen, "opt_steps": opt_steps, "minutes": round((time.time() - start) / 60, 2),
               **{k: scores[k] for k in ("macro_f1", "macro_precision", "macro_recall", "accuracy", "f1_class0", "f1_class1")}}
        history.append(row)
        print(json.dumps(row), flush=True)
        if scores["macro_f1"] > best["macro_f1"]:
            best = scores
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        model.train()
        last_eval = time.time()

    done = False
    for epoch in range(1, args.epochs + 1):
        order = list(range(len(train_chunks)))
        random.Random(args.seed + epoch).shuffle(order)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for s in range(0, len(order), args.batch_docs):
            rows = order[s:s + args.batch_docs]
            ids, mask, valid = make_batch([train_chunks[r] for r in rows], pad_id, device)
            y = torch.tensor([train_labels[r] for r in rows], device=device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits, _ = model(ids, mask, valid)
            logits = logits.float()
            loss = F.cross_entropy(logits, y, weight=weights, label_smoothing=args.label_smoothing)
            if soft is not None:
                t = args.temperature
                teacher = soft[rows].to(device)
                kd = F.kl_div(F.log_softmax(logits / t, -1), F.softmax(teacher / t, -1), reduction="batchmean") * t * t
                loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
            scaler.scale(loss / args.grad_accum).backward()
            step += 1
            seen += len(rows)
            if step % args.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                sched.step()
                opt_steps += 1
                if opt_steps == 40:  # re-estimate how many optimiser steps fit into the budget
                    rate = opt_steps / (time.time() - start)
                    est_steps = max(60, int(rate * budget * 0.92))
                    print(f"~{rate:.2f} opt steps/s -> schedule over {est_steps} steps "
                          f"(~{est_steps * args.batch_docs * args.grad_accum} docs)", flush=True)
            losses.append(float(loss))
            if step % 50 == 0:
                print(f"epoch {epoch} docs {seen} loss {sum(losses[-50:]) / len(losses[-50:]):.4f} "
                      f"elapsed {(time.time() - start) / 60:.1f} min", flush=True)
            if time.time() - start > budget:
                print("Time budget reached.", flush=True)
                done = True
                break
            if time.time() - last_eval > args.eval_every_min * 60:
                evaluate(f"epoch{epoch}")
        if done:
            break
        evaluate(f"end_epoch{epoch}")
    evaluate("final")
    model.load_state_dict(best_state)
    return best, history, time.time() - start


# ----------------------------------------------------------------------------- main

def run(args):
    t0 = time.time()
    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    name = args.run_name or f"p4_{args.model}_{args.method}"
    out = Path(args.output_dir or f"/kaggle/working/{name}")
    out.mkdir(parents=True, exist_ok=True)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    files = obtain_data(out / "data", args.data_dir)
    train_cases = load_cases(files["train.json"], True)
    dev_cases = load_cases(files["dev.json"], True)
    test_cases = load_cases(files["test.json"], False)
    if len({c["id"] for c in test_cases}) != len(test_cases):
        raise RuntimeError("Duplicate ids in test.json")
    leak = leakage_report(train_cases, dev_cases, test_cases)
    if args.limit_train:
        train_cases = train_cases[:args.limit_train]
    print({"train": len(train_cases), "dev": len(dev_cases), "test": len(test_cases),
           "train_labels": dict(Counter(c["label"] for c in train_cases)),
           "dev_labels": dict(Counter(c["label"] for c in dev_cases))}, flush=True)

    checkpoint, tokenizer_path, artifact = locate_phase1_files(args.model, args.artifact_zip, out / "phase1")
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    pad_id, args.eos_id = tokenizer.token_to_id("<pad>"), tokenizer.token_to_id("<eos>")
    if pad_id is None or args.eos_id is None:
        raise ValueError("Phase 1 tokenizer must contain <pad> and <eos>")

    soft = None
    if args.method == "distill":
        saved = torch.load(args.teacher_soft_labels, map_location="cpu")
        index = {cid: r for r, cid in enumerate(saved["ids"])}
        before = len(train_cases)
        train_cases = [c for c in train_cases if c["id"] in index]
        soft = saved["logits"][[index[c["id"]] for c in train_cases]]
        print(f"KD: teacher {saved['teacher']} soft labels for {len(train_cases)}/{before} train cases", flush=True)

    enc_t = time.time()
    train_chunks = encode_cases(train_cases, tokenizer, args)
    dev_chunks = encode_cases(dev_cases, tokenizer, args)
    test_chunks = encode_cases(test_cases, tokenizer, args)
    chunk_stats = Counter(len(c) for c in train_chunks)
    print(f"tokenised in {time.time() - enc_t:.0f}s; chunks/doc histogram {dict(sorted(chunk_stats.items()))}", flush=True)
    train_labels = [c["label"] for c in train_cases]
    dev_labels = [c["label"] for c in dev_cases]

    backbone, phase1_cfg, phase1_saved = load_phase1(checkpoint, args.model)
    phase1_params = sum(p.numel() for p in backbone.parameters())
    if args.method == "distill":
        if args.model != "decoder":
            raise ValueError("The distillation student recipe uses the decoder Phase 1 model")
        backbone = make_student(backbone, layers=args.student_layers)
    model = CJPEModel(backbone, args.model, args.max_chunks)
    apply_method(model, args.model, args.method, args.rank, args.unfreeze_layers)
    model.to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    summary = {"run": name, "model": args.model, "method": args.method, "phase1_artifact": artifact,
               "phase1_parameters": phase1_params, "parameters": total, "trainable_parameters": trainable,
               "trainable_fraction": trainable / total, "leakage_check": leak, "device": str(device),
               "config": vars(args)}
    print(json.dumps(summary, indent=2, default=str), flush=True)

    best, history, seconds = train(model, train_chunks, train_labels, dev_chunks, dev_labels, args, device, pad_id, soft)

    inf_t = time.time()
    dev_logits, _ = predict(model, dev_chunks, pad_id, device, args.batch_docs * 2)
    dev_probs = dev_logits.softmax(-1)[:, 1].tolist()
    argmax_scores = binary_scores(dev_labels, dev_logits.argmax(-1).tolist())
    threshold, tuned_f1 = tune_threshold(dev_probs, dev_labels) if args.tune_threshold else (0.5, None)
    tuned = binary_scores(dev_labels, [int(p >= threshold) for p in dev_probs])
    test_logits, test_alpha = predict(model, test_chunks, pad_id, device, args.batch_docs * 2)
    inference_seconds = time.time() - inf_t
    test_probs = test_logits.softmax(-1)[:, 1].tolist()
    print(f"dev macro-F1 argmax {argmax_scores['macro_f1']:.4f} | threshold {threshold:.2f} -> {tuned['macro_f1']:.4f}", flush=True)

    predictions = [{"id": c["id"], "pred": int(p >= threshold)} for c, p in zip(test_cases, test_probs)]
    assert len(predictions) == len(test_cases) and all(p["pred"] in (0, 1) for p in predictions)
    path = out / "prediction.json"
    path.write_text(json.dumps(predictions), encoding="utf-8")
    archive = out / f"submission_{name}.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        bundle.write(path, arcname="prediction.json")
    with zipfile.ZipFile(archive) as bundle:
        assert bundle.namelist() == ["prediction.json"]
        assert len(json.loads(bundle.read("prediction.json"))) == len(test_cases)

    # error analysis material: dev predictions with probabilities + attention over chunks
    _, dev_alpha = predict(model, dev_chunks, pad_id, device, args.batch_docs * 2)
    (out / "dev_predictions.json").write_text(json.dumps([
        {"id": c["id"], "gold": c["label"], "prob1": round(p, 4), "pred": int(p >= threshold),
         "words": len(c["text"].split()), "chunk_attention": [round(a, 3) for a in al.tolist()]}
        for c, p, al in zip(dev_cases, dev_probs, dev_alpha)]), encoding="utf-8")
    slot_attention = torch.zeros(args.max_chunks)
    slot_counts = torch.zeros(args.max_chunks)
    for al in dev_alpha:
        slot_attention[:len(al)] += al
        slot_counts[:len(al)] += 1
    mean_slot_attention = (slot_attention / slot_counts.clamp_min(1)).tolist()

    if args.export_soft_labels:
        ex_t = time.time()
        order = list(range(len(train_cases)))
        random.Random(args.seed).shuffle(order)
        ids, logits = [], []
        for s in range(0, len(order), 256):
            rows = order[s:s + 256]
            lg, _ = predict(model, [train_chunks[r] for r in rows], pad_id, device, args.batch_docs * 2)
            ids += [train_cases[r]["id"] for r in rows]
            logits.append(lg)
            if time.time() - ex_t > args.export_budget_min * 60:
                break
        torch.save({"ids": ids, "logits": torch.cat(logits), "teacher": name}, out / "teacher_soft_labels.pt")
        print(f"Saved teacher soft labels for {len(ids)} train cases in {(time.time() - ex_t) / 60:.1f} min", flush=True)

    if args.save_checkpoint:
        torch.save({"model": model.state_dict(), "phase1_config": asdict(phase1_cfg), "run_config": vars(args)},
                   out / "best.pt")
    (out / "training_log.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    (out / "validation_metrics.json").write_text(json.dumps(
        {"selected_checkpoint": best, "argmax": argmax_scores, "threshold": threshold, "tuned": tuned}, indent=2),
        encoding="utf-8")
    manifest = {**summary, "best_dev_macro_f1_argmax": argmax_scores["macro_f1"], "threshold": threshold,
                "dev_macro_f1_tuned": tuned["macro_f1"], "dev_macro_precision": tuned["macro_precision"],
                "dev_macro_recall": tuned["macro_recall"], "dev_false_positives": tuned["false_positives"],
                "dev_false_negatives": tuned["false_negatives"], "mean_chunk_attention_by_slot": mean_slot_attention,
                "test_pred_distribution": dict(Counter(p["pred"] for p in predictions)),
                "train_minutes": seconds / 60, "inference_seconds_dev_plus_test": inference_seconds,
                "total_minutes": (time.time() - t0) / 60,
                "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None,
                "phase1_steps": phase1_saved.get("total_steps"), "submission": str(archive)}
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    print(json.dumps({k: manifest[k] for k in ("run", "best_dev_macro_f1_argmax", "threshold", "dev_macro_f1_tuned",
                                                "trainable_parameters", "test_pred_distribution", "total_minutes")}, indent=2))
    print("SUBMISSION ZIP:", archive, flush=True)


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True, choices=("decoder", "encoder_decoder", "hybrid", "efficient"))
    p.add_argument("--method", required=True, choices=("full", "lorma", "lora", "ia3", "partial", "bitfit", "distill"))
    p.add_argument("--selection", default="headtail", choices=("headtail", "tail"))
    p.add_argument("--chunk-len", type=int, default=256)
    p.add_argument("--max-chunks", type=int, default=6)
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch-docs", type=int, default=4)
    p.add_argument("--grad-accum", type=int, default=2)
    p.add_argument("--learning-rate", type=float, default=3e-5)
    p.add_argument("--head-lr", type=float, default=1e-3)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--unfreeze-layers", type=int, default=2)
    p.add_argument("--class-weight-power", type=float, default=0.5)
    p.add_argument("--label-smoothing", type=float, default=0.05)
    p.add_argument("--student-layers", type=int, default=8)
    p.add_argument("--teacher-soft-labels")
    p.add_argument("--kd-alpha", type=float, default=0.5)
    p.add_argument("--temperature", type=float, default=2.0)
    p.add_argument("--export-soft-labels", action="store_true")
    p.add_argument("--export-budget-min", type=float, default=6.0)
    p.add_argument("--save-checkpoint", action="store_true")
    p.add_argument("--tune-threshold", type=int, default=1)
    p.add_argument("--time-budget-min", type=float, default=25.0)
    p.add_argument("--eval-every-min", type=float, default=6.0)
    p.add_argument("--limit-train", type=int, default=0)
    p.add_argument("--artifact-zip")
    p.add_argument("--data-dir")
    p.add_argument("--output-dir")
    p.add_argument("--run-name")
    p.add_argument("--seed", type=int, default=230338)
    return p.parse_args(argv)


if __name__ == "__main__":
    run(parse_args())


In [ ]:
%%writefile /kaggle/working/phase7_pcr.py
"""CS781 Phase 7 - Prior Case Retrieval (PCR) starting from Phase 1 legal language models.

For every query case, rank the candidate cases of the SAME split's pool and return exactly 7 unique ids.
Metric: Micro-F1@7 (micro precision/recall over the top 7 of every query vs the complete gold set).

Pipeline
  1. Lexical stage (no pretrained weights, built from the split's own pool):
       BM25 on word unigrams / bigrams / trigrams (hashed, so rare quoted phrases are kept) + TF-IDF cosine
       (uni, bi).  k1 / b are chosen on dev.  Candidate generation = reciprocal-rank fusion of these systems
       -> top-N per query (the query itself / exact duplicates of it are never returned).
     Extra legal signals: section / article number overlap, judgment-year difference (prior cases must be
     older), how often the candidate is cited by train queries, how central the candidate is in the pool.
  2. Dense retriever fine-tuned from a Phase 1 checkpoint (the required part of every experiment):
       * long documents -> up to K windows of 256 tokens (the Phase 1 context); queries use the windows
         around the masked citation markers (that is where a prior case is discussed) + the opening window;
         candidates use windows spread over the whole judgment;
       * window vector = masked mean of the Phase 1 hidden states -> projection (separate query / candidate
         heads); document = attention + mean pooling over windows (``attn``) or late interaction
         (``maxsim``: mean over query windows of the max cosine over candidate windows);
       * InfoNCE (learned temperature) over 1 positive + H BM25 hard negatives per query + in-batch
         candidates, with other gold candidates of the same query masked out (no false negatives);
       * adaptation: full FT, LORMA+, LoRA, IA3, partial (top layers), BitFit, knowledge distillation
         (student = decoder cut to 8 layers, listwise KL to the teacher's score distribution + InfoNCE).
  3. Fusion: per-query normalised lexical + dense + legal features -> LambdaMART (LightGBM) / logistic
     regression, trained on dev only (the dense model never saw dev).  The method is chosen by 5-fold
     query-grouped cross-fitting on dev (honest estimate) and only among methods that use the Phase 1
     retriever; the lexical-only number is logged for comparison.  Refit on all of dev -> test top 7.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import pickle
import random
import re
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO_ID = "Exploration-Lab/CS-781-Capstone"
SPLITS = ("train", "dev", "test")
TOP_K = 7
CITE_RES = [re.compile(r"<\s*/?\s*cit(?:ation|ed)?[^>]{0,20}>", re.I), re.compile(r"\[\s*citation\s*\]", re.I),
            re.compile(r"\bCITATION\b"), re.compile(r"\{\s*citation\s*\}", re.I)]
YEAR_RE = re.compile(r"\b(18[5-9]\d|19\d\d|20[0-2]\d)\b")
SECTION_RE = re.compile(r"\b(?:section|sec\.|sections|u/s|u/ss)\s*(\d+[a-z]?)", re.I)
ARTICLE_RE = re.compile(r"\b(?:article|art\.|articles)\s*(\d+[a-z]?)", re.I)


def log(*a):
    print(*a, flush=True)


# ============================================================================= data

def obtain_data(data_dir: str | None) -> dict:
    """{split: {"q": path, "c": path or None}} from --data-dir or the official HF dataset (re-downloaded)."""
    want = {s: (f"{s}.json", f"{s}_candidates.json") for s in SPLITS}
    if data_dir:
        files = [Path(f) / n for f, _, ns in os.walk(data_dir, followlinks=True) for n in ns]
    else:
        from huggingface_hub import hf_hub_download, list_repo_files
        token = os.environ.get("HF_TOKEN")
        if not token:
            raise RuntimeError("HF_TOKEN is missing: add the Kaggle secret HF and enable it")
        repo = list_repo_files(REPO_ID, repo_type="dataset", token=token)
        p7 = [f for f in repo if ("phase_7" in f.lower() or "pcr" in f.lower()) and f.lower().endswith((".json", ".jsonl"))]
        log("Phase 7 files on HF:", p7)
        if not p7:
            raise FileNotFoundError(f"No Phase 7 / PCR json files on HF yet. Repo has: {sorted({f.split('/')[0] for f in repo})}")
        files = []
        for f in p7:
            if Path(f).name.lower().replace(".jsonl", ".json") in {n for pair in want.values() for n in pair}:
                files.append(Path(hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=token,
                                                  force_download=True)))
    found = {}
    for s, (qn, cn) in want.items():
        q = [f for f in files if f.name.lower().replace(".jsonl", ".json") == qn]
        c = [f for f in files if f.name.lower().replace(".jsonl", ".json") == cn]
        found[s] = {"q": q[0] if q else None, "c": c[0] if c else None}
    if not found["train"]["q"] or not found["train"]["c"] or not found["test"]["q"] or not found["test"]["c"]:
        raise FileNotFoundError(f"Missing Phase 7 files: {found}")
    return found


ID_KEYS = ("id", "query_id", "qid", "case_id", "doc_id", "candidate_id", "cid", "uid", "name", "file", "filename")
TEXT_KEYS = ("text", "sentences", "content", "document", "case_text", "raw_text", "doc", "body", "judgment")
REL_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "relevant_cases", "positives", "positive",
            "labels", "label", "gold", "citations", "cited", "relevant_candidate_ids", "targets")


def canon(x) -> str:
    s = str(x).strip()
    for ext in (".txt", ".json"):
        if s.lower().endswith(ext):
            s = s[: -len(ext)]
    return (s.lstrip("0") or "0") if s.isdigit() else s


def to_sentences(t) -> list[str]:
    if t is None:
        return []
    if isinstance(t, str):
        parts = [p for p in re.split(r"\n+|(?<=[.?!])\s+(?=[A-Z(\"'])", t) if p.strip()]
        return parts or [t]
    if isinstance(t, dict):
        t = list(t.values())
    out = []
    for x in t:
        if isinstance(x, (list, tuple, dict)):
            out += to_sentences(x)
        elif x is not None and str(x).strip():
            out.append(str(x))
    return out


def _pick(rec, keys):
    for k in keys:
        if k in rec and rec[k] is not None:
            return rec[k]
    return None


def load_records(path: Path, with_rel: bool) -> list[dict]:
    content = Path(path).read_text(encoding="utf-8")
    try:
        raw = json.loads(content)
    except json.JSONDecodeError:  # JSON lines
        raw = [json.loads(line) for line in content.splitlines() if line.strip()]
    if isinstance(raw, dict):
        for w in ("data", "queries", "candidates", "documents", "cases", "examples", "items"):
            if isinstance(raw.get(w), (list, dict)) and len(raw) <= 3:
                raw = raw[w]
                break
    if isinstance(raw, dict):
        items = []
        for k, v in raw.items():
            if isinstance(v, dict):
                items.append({**v, "id": _pick(v, ID_KEYS) or k})
            else:
                items.append({"id": k, "text": v})
        raw = items
    out = []
    for i, r in enumerate(raw):
        if not isinstance(r, dict):
            raise ValueError(f"{Path(path).name} record #{i} is {type(r).__name__}, expected an object")
        rid = _pick(r, ID_KEYS)
        text = _pick(r, TEXT_KEYS)
        if rid is None or text is None:
            raise ValueError(f"{Path(path).name} record #{i}: cannot find id/text, keys={list(r)}")
        rel = None
        if with_rel:
            rel = _pick(r, REL_KEYS)
            if rel is None:
                raise ValueError(f"{Path(path).name}/{rid}: no relevance field among {REL_KEYS}; keys={list(r)}")
            if isinstance(rel, dict):
                rel = [k for k, v in rel.items() if v not in (0, False, None)]
            if not isinstance(rel, (list, tuple)):
                rel = [rel]
            rel = [(_pick(x, ID_KEYS) if isinstance(x, dict) else x) for x in rel]
        sents = to_sentences(text)
        out.append({"id": rid, "key": canon(rid), "sents": sents, "text": " ".join(sents),
                    "rel": None if rel is None else sorted({canon(x) for x in rel})})
    log(f"{Path(path).name}: {len(out)} records; keys of first record: {list(raw[0]) if raw else []}")
    return out


def text_key(text: str) -> str:
    return hashlib.md5(" ".join(text.lower().split()).encode()).hexdigest()


def load_all(data_dir, seed):
    files = obtain_data(data_dir)
    data = {}
    for s in SPLITS:
        f = files[s]
        if f["q"] is None:
            continue
        data[s] = {"q": load_records(f["q"], s != "test"), "c": load_records(f["c"], False)}
    if "dev" not in data:  # fallback: hold out 20% of the train queries (same train pool)
        rng = random.Random(seed)
        qs = data["train"]["q"][:]
        rng.shuffle(qs)
        cut = max(20, len(qs) // 5)
        data["dev"] = {"q": qs[:cut], "c": data["train"]["c"]}
        data["train"] = {"q": qs[cut:], "c": data["train"]["c"]}
        log(f"NO dev.json: held out {cut} train queries as dev (same candidate pool)")
    # leakage: never train on / select with a query that is also a test query
    test_keys = {q["key"] for q in data["test"]["q"]}
    test_text = {text_key(q["text"]) for q in data["test"]["q"]}
    leak = {}
    for s in ("train", "dev"):
        before = len(data[s]["q"])
        data[s]["q"] = [q for q in data[s]["q"] if q["key"] not in test_keys and text_key(q["text"]) not in test_text]
        leak[f"{s}_queries_removed_as_test_duplicates"] = before - len(data[s]["q"])
    dev_keys = {q["key"] for q in data["dev"]["q"]}
    before = len(data["train"]["q"])
    data["train"]["q"] = [q for q in data["train"]["q"] if q["key"] not in dev_keys]
    leak["train_queries_removed_as_dev_duplicates"] = before - len(data["train"]["q"])
    for s in SPLITS:
        pool = {c["key"] for c in data[s]["c"]}
        if len(pool) != len(data[s]["c"]):
            log(f"WARNING {s}: duplicate candidate ids in pool ({len(data[s]['c'])} records, {len(pool)} unique)")
            seen, keep = set(), []
            for c in data[s]["c"]:
                if c["key"] not in seen:
                    seen.add(c["key"])
                    keep.append(c)
            data[s]["c"] = keep
        leak[f"{s}_queries_inside_own_pool"] = sum(q["key"] in pool for q in data[s]["q"])
    pools = {s: {c["key"] for c in data[s]["c"]} for s in SPLITS}
    leak["train_test_pool_overlap"] = len(pools["train"] & pools["test"])
    leak["dev_test_pool_overlap"] = len(pools["dev"] & pools["test"])
    log("LEAKAGE CHECK:", json.dumps(leak))
    stats = {}
    for s in SPLITS:
        pool = pools[s]
        qs = data[s]["q"]
        words = sorted(len(q["text"].split()) for q in qs)
        cw = sorted(len(c["text"].split()) for c in data[s]["c"])
        st = {"queries": len(qs), "pool": len(pool),
              "query_words_p50_p90": [words[len(words) // 2], words[int(len(words) * .9)]] if words else None,
              "cand_words_p50_p90": [cw[len(cw) // 2], cw[int(len(cw) * .9)]] if cw else None,
              "queries_with_citation_markers": sum(any(r.search(q["text"]) for r in CITE_RES) for q in qs)}
        if s != "test":
            g = [len(q["rel"]) for q in qs]
            inpool = [sum(x in pool for x in q["rel"]) for q in qs]
            st.update({"gold_mean": round(sum(g) / max(1, len(g)), 3), "gold_hist": dict(sorted(Counter(min(x, 15) for x in g).items())),
                       "gold_in_pool_fraction": round(sum(inpool) / max(1, sum(g)), 4),
                       "queries_without_gold_in_pool": sum(x == 0 for x in inpool),
                       "oracle_micro_f1@7": round(oracle_f1(qs, pool), 4)})
        stats[s] = st
        log(f"DATA [{s}]", json.dumps(st))
    return data, leak, stats


def oracle_f1(queries, pool):
    tp = sum(min(TOP_K, sum(x in pool for x in q["rel"])) for q in queries)
    return micro_from(tp, TOP_K * len(queries), sum(len(q["rel"]) for q in queries))["f1"]


# ============================================================================= metrics

def micro_from(tp, n_pred, n_gold):
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    return {"f1": 2 * p * r / (p + r) if p + r else 0.0, "precision": p, "recall": r}


def micro_at_k(ranked: list[list[str]], gold: list[list[str]], k=TOP_K):
    tp = sum(len(set(r[:k]) & set(g)) for r, g in zip(ranked, gold))
    return micro_from(tp, k * len(ranked), sum(len(g) for g in gold))


# ============================================================================= lexical stage

def clean_for_lexical(text: str) -> str:
    for r in CITE_RES:
        text = r.sub(" ", text)
    return text


def hashed_tf(texts, n, n_features, n_jobs=4):
    from joblib import Parallel, delayed
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, HashingVectorizer
    vec = HashingVectorizer(n_features=n_features, ngram_range=(n, n), alternate_sign=False, norm=None,
                            lowercase=True, stop_words=list(ENGLISH_STOP_WORDS), token_pattern=r"(?u)\b\w\w+\b",
                            dtype=np.float32)
    chunks = [texts[i:i + 200] for i in range(0, len(texts), 200)]
    mats = Parallel(n_jobs=n_jobs)(delayed(vec.transform)(c) for c in chunks)
    return sp.vstack(mats).tocsr()


def idf_of(D):
    N = D.shape[0]
    df = np.bincount(D.indices, minlength=D.shape[1]).astype(np.float32)
    return np.log1p((N - df + 0.5) / (df + 0.5)).astype(np.float32), df


def bm25_scores(Qtf, Dtf, k1, b):
    idf, _ = idf_of(Dtf)
    D = Dtf.copy().astype(np.float32)
    dl = np.asarray(Dtf.sum(1)).ravel()
    avg = dl.mean() if dl.mean() > 0 else 1.0
    rows = np.repeat(np.arange(D.shape[0]), np.diff(D.indptr))
    denom = D.data + k1 * (1 - b + b * dl[rows] / avg)
    D.data = D.data * (k1 + 1) / denom * idf[D.indices]
    Q = Qtf.copy().astype(np.float32)
    Q.data = 1 + np.log(Q.data)
    return (Q @ D.T).toarray().astype(np.float32)


def tfidf_cos(Qtf, Dtf):
    idf, _ = idf_of(Dtf)

    def w(M):
        M = M.copy().astype(np.float32)
        M.data = (1 + np.log(M.data)) * idf[M.indices]
        norm = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()) + 1e-9
        return sp.diags(1 / norm) @ M
    return (w(Qtf) @ w(Dtf).T).toarray().astype(np.float32)


def rank_of(S):
    """rank (0 = best) of every column within each row."""
    order = np.argsort(-S, axis=1, kind="stable")
    ranks = np.empty_like(order)
    ranks[np.arange(S.shape[0])[:, None], order] = np.arange(S.shape[1])[None, :]
    return ranks


def legal_sets(text):
    return ({m.lower() for m in SECTION_RE.findall(text)}, {m.lower() for m in ARTICLE_RE.findall(text)})


def doc_year(text):
    ys = [int(y) for y in YEAR_RE.findall(text)]
    ys = [y for y in ys if y <= 2026]
    return max(ys) if ys else -1


def exclusion_mask(queries, cands):
    ckeys = {c["key"]: j for j, c in enumerate(cands)}
    ctext = defaultdict(list)
    for j, c in enumerate(cands):
        ctext[text_key(c["text"])].append(j)
    mask = np.zeros((len(queries), len(cands)), dtype=bool)
    for i, q in enumerate(queries):
        if q["key"] in ckeys:
            mask[i, ckeys[q["key"]]] = True
        for j in ctext.get(text_key(q["text"]), []):
            mask[i, j] = True
    return mask


def lexical_split(split, queries, cands, args, params=None, train_popularity=None):
    """-> dict with candidate lists (top-N), feature tensor and the per-system score matrices' summaries."""
    t = time.time()
    qt = [clean_for_lexical(q["text"]) for q in queries]
    ct = [clean_for_lexical(c["text"]) for c in cands]
    excl = exclusion_mask(queries, cands)
    tfs = {}
    for n in ((1,) if split == "train" else (1, 2, 3)):
        both = hashed_tf(qt + ct, n, 2 ** (20 if n == 1 else 22), args.cpu_jobs)
        tfs[n] = (both[:len(qt)], both[len(qt):])
    log(f"[{split}] hashed n-gram tf in {time.time() - t:.0f}s")
    systems = {}
    gold = [q["rel"] for q in queries] if split != "test" else None
    tuned = {}
    for n in ((1,) if split == "train" else (1, 2, 3)):
        Q, D = tfs[n]
        if params is None and gold is not None and n in (1, 2):
            best = None
            for k1 in (0.9, 1.5, 3.0):
                for b in (0.5, 0.75, 1.0):
                    S = bm25_scores(Q, D, k1, b)
                    S[excl] = -1e9
                    f = micro_at_k(top_ids(S, cands, TOP_K), gold)["f1"]
                    if best is None or f > best[0]:
                        best = (f, k1, b)
            tuned[f"bm25_{n}"] = {"k1": best[1], "b": best[2], "dev_f1@7": round(best[0], 4)}
        p = (params or {}).get(f"bm25_{n}") or tuned.get(f"bm25_{n}") or {"k1": 1.5, "b": 0.75}
        systems[f"bm25_{n}"] = bm25_scores(Q, D, p["k1"], p["b"])
        if n in (1, 2) and split != "train":
            systems[f"tfidf_{n}"] = tfidf_cos(Q, D)
    for S in systems.values():
        S[excl] = -1e9
    if tuned:
        log(f"[{split}] BM25 grid (dev F1@7):", json.dumps(tuned))
    used_params = params or {k: tuned.get(k, {"k1": 1.5, "b": 0.75}) for k in ("bm25_1", "bm25_2", "bm25_3")}
    report = {}
    if gold is not None:
        for name, S in systems.items():
            report[name] = {k: round(v, 4) for k, v in micro_at_k(top_ids(S, cands, TOP_K), gold).items()}
    # RRF over the lexical systems -> candidate list
    rrf = np.zeros_like(next(iter(systems.values())))
    for S in systems.values():
        rrf += 1.0 / (60.0 + rank_of(S))
    rrf[excl] = -1e9
    N = min(args.top_n if split != "train" else args.hn_pool + 10, len(cands) - 1)
    order = np.argsort(-rrf, axis=1, kind="stable")
    cand_lists = order[:, :N]
    if gold is not None:
        report["rrf_lexical"] = {k: round(v, 4) for k, v in micro_at_k(top_ids(rrf, cands, TOP_K), gold).items()}
        tp = sum(len({cands[j]["key"] for j in row} & set(g)) for row, g in zip(cand_lists, gold))
        report[f"recall@{N}"] = round(tp / max(1, sum(len(g) for g in gold)), 4)
        log(f"[{split}] lexical dev/train metrics:", json.dumps(report))
    out = {"split": split, "query_ids": [q["id"] for q in queries], "query_keys": [q["key"] for q in queries],
           "cand_ids": [c["id"] for c in cands], "cand_keys": [c["key"] for c in cands], "lists": cand_lists,
           "gold": gold, "report": report, "params": used_params, "rrf_full_order": order[:, :max(N, 50)]}
    if split == "train":
        return out
    # ---------------- features for fusion (nq, N, F)
    names = list(systems)
    feats, fnames = [], []
    rows = np.arange(len(queries))[:, None]
    for name in names + ["rrf"]:
        S = systems[name] if name != "rrf" else rrf
        R = rank_of(S)
        v = S[rows, cand_lists]
        mu, sd = v.mean(1, keepdims=True), v.std(1, keepdims=True) + 1e-6
        top1 = v.max(1, keepdims=True)
        feats += [(v - mu) / sd, v / (np.abs(top1) + 1e-6), 1.0 / (1.0 + R[rows, cand_lists])]
        fnames += [f"{name}_z", f"{name}_rel_top", f"{name}_rr"]
    # legal features
    qsets = [legal_sets(q["text"]) for q in queries]
    csets = [legal_sets(c["text"]) for c in cands]
    qy = np.array([doc_year(q["text"]) for q in queries])
    cy = np.array([doc_year(c["text"]) for c in cands])
    clen = np.log1p(np.array([len(c["text"].split()) for c in cands], dtype=np.float32))
    pop = np.array([train_popularity.get(c["key"], 0) for c in cands], dtype=np.float32) if train_popularity else np.zeros(len(cands), np.float32)
    central = np.zeros(len(cands), np.float32)
    for row in order[:, :20]:
        central[row] += 1
    central /= max(1, len(queries))
    sec_j = np.zeros(cand_lists.shape, np.float32)
    sec_n = np.zeros(cand_lists.shape, np.float32)
    art_j = np.zeros(cand_lists.shape, np.float32)
    for i, row in enumerate(cand_lists):
        qs, qa = qsets[i]
        for r, j in enumerate(row):
            cs, ca = csets[j]
            inter = len(qs & cs)
            sec_n[i, r] = inter
            sec_j[i, r] = inter / max(1, len(qs | cs))
            art_j[i, r] = len(qa & ca) / max(1, len(qa | ca))
    ydiff = np.where((qy[:, None] > 0) & (cy[cand_lists] > 0), np.clip(qy[:, None] - cy[cand_lists], -30, 80), 0)
    feats += [sec_j, np.log1p(sec_n), art_j, ydiff.astype(np.float32),
              ((qy[:, None] > 0) & (cy[cand_lists] > qy[:, None])).astype(np.float32),
              ((qy[:, None] <= 0) | (cy[cand_lists] <= 0)).astype(np.float32),
              np.broadcast_to(clen[cand_lists], cand_lists.shape), np.log1p(pop[cand_lists]), central[cand_lists]]
    fnames += ["section_jaccard", "section_overlap_log", "article_jaccard", "year_diff", "cand_after_query",
               "year_missing", "cand_log_words", "train_cited_log", "pool_centrality"]
    out["features"] = np.stack(feats, -1).astype(np.float32)
    out["feature_names"] = fnames
    log(f"[{split}] lexical stage done in {time.time() - t:.0f}s; features {out['features'].shape}")
    return out


def top_ids(S, cands, k):
    order = np.argsort(-S, axis=1, kind="stable")[:, :k]
    return [[cands[j]["key"] for j in row] for row in order]


# ============================================================================= windows for the dense model

def select_spread(ids, L, K):
    n = len(ids)
    if n <= L:
        return [ids]
    starts = list(range(0, n - L + 1, L))
    if starts[-1] != n - L:
        starts.append(n - L)
    if len(starts) > K:
        picks = sorted({round(i * (len(starts) - 1) / (K - 1)) for i in range(K)}) if K > 1 else [0]
        starts = [starts[i] for i in picks]
    return [ids[s:s + L] for s in starts]


def citation_contexts(sents, before):
    out = []
    for i, s in enumerate(sents):
        if any(r.search(s) for r in CITE_RES):
            ctx = " ".join(sents[max(0, i - before): i + 1])
            out.append(clean_for_lexical(ctx))
    return out


def build_windows(docs, tokenizer, args, is_query):
    L, K = args.chunk_len, args.max_windows
    texts = [clean_for_lexical(" ".join(d["text"].split()[:args.max_words])) for d in docs]
    result = [None] * len(docs)
    marker_docs = 0
    ctx_jobs = []
    for s in range(0, len(docs), 256):
        for i, enc in zip(range(s, min(len(docs), s + 256)), tokenizer.encode_batch(texts[s:s + 256])):
            ids = np.asarray(enc.ids or [args.eos_id], dtype=np.int32)
            result[i] = select_spread(ids, L, K)
            if is_query and args.query_windows == "citation":
                ctx = citation_contexts(docs[i]["sents"], args.context_sentences)
                if ctx:
                    marker_docs += 1
                    keep = K - args.query_head_windows
                    if len(ctx) > keep:
                        picks = sorted({round(t * (len(ctx) - 1) / (keep - 1)) for t in range(keep)}) if keep > 1 else [0]
                        ctx = [ctx[p] for p in picks]
                    ctx_jobs.append((i, ids, ctx))
    for i, ids, ctx in ctx_jobs:
        encs = tokenizer.encode_batch(ctx)
        wins = [np.asarray(e.ids[-L:], dtype=np.int32) for e in encs if e.ids]
        head = [ids[h * L:(h + 1) * L] for h in range(args.query_head_windows) if len(ids) > h * L]
        result[i] = (head + wins)[:K] or result[i]
    if is_query:
        log(f"  {marker_docs}/{len(docs)} queries use citation-context windows")
    return result


def make_batch(window_lists, pad_id, device):
    flat = [w for ws in window_lists for w in ws]
    length = max(len(w) for w in flat)
    ids = torch.full((len(flat), length), pad_id, dtype=torch.long)
    mask = torch.zeros((len(flat), length), dtype=torch.bool)
    for r, w in enumerate(flat):
        ids[r, :len(w)] = torch.from_numpy(w.astype(np.int64))
        mask[r, :len(w)] = True
    K = max(len(ws) for ws in window_lists)
    valid = torch.zeros((len(window_lists), K), dtype=torch.bool)
    for i, ws in enumerate(window_lists):
        valid[i, :len(ws)] = True
    return ids.to(device), mask.to(device), valid.to(device)


# ============================================================================= model

class PCRModel(nn.Module):
    def __init__(self, backbone, model_name, proj_dim=256, pooling="attn", grad_ckpt=False):
        super().__init__()
        from phase2_lner import LNERModel
        self.encoder = LNERModel(backbone, model_name, 1)
        del self.encoder.classifier
        d = backbone.cfg.d_model
        self.model_name, self.pooling, self.grad_ckpt = model_name, pooling, grad_ckpt
        self.proj = nn.ModuleDict({s: nn.Sequential(nn.LayerNorm(d), nn.Linear(d, proj_dim)) for s in ("q", "c")})
        self.attn = nn.ModuleDict({s: nn.Linear(proj_dim, 1) for s in ("q", "c")})
        self.logit_scale = nn.Parameter(torch.tensor(math.log(20.0)))

    def head_parameters(self):
        return [p for n, p in self.named_parameters() if not n.startswith("encoder.")]

    def hidden(self, ids, mask):
        if self.model_name == "efficient" and self.grad_ckpt and self.training:
            from torch.utils.checkpoint import checkpoint
            bb = self.encoder.backbone
            h = bb.embedding(ids)
            for block in bb.blocks:
                h = checkpoint(block, h, use_reentrant=False)
            return bb.norm(h)
        return self.encoder.hidden(ids, mask)

    def embed(self, ids, mask, valid, side):
        h = self.hidden(ids, mask)
        w = mask.unsqueeze(-1).to(h.dtype)
        v = (h * w).sum(1) / w.sum(1).clamp_min(1)
        v = self.proj[side](v).float()
        B, K = valid.shape
        chunks = v.new_zeros(B, K, v.size(-1))
        chunks[valid] = v
        if self.pooling == "maxsim":
            return F.normalize(chunks, dim=-1) * valid.unsqueeze(-1)
        s = self.attn[side](chunks).squeeze(-1).masked_fill(~valid, -1e4)
        a = s.softmax(-1)
        doc = (a.unsqueeze(-1) * chunks).sum(1) + (chunks * valid.unsqueeze(-1)).sum(1) / valid.sum(1, keepdim=True).clamp_min(1)
        return F.normalize(doc, dim=-1)

    def sim(self, q, c, qvalid=None, cvalid=None):
        if self.pooling != "maxsim":
            return q @ c.T
        s = torch.einsum("ikp,jlp->ijkl", q, c)
        s = s.masked_fill(~cvalid[None, :, None, :], -1e4).max(-1).values
        qv = qvalid[:, None, :].float()
        return (s * qv).sum(-1) / qv.sum(-1).clamp_min(1)

    def scale(self):
        return self.logit_scale.exp().clamp(max=100.0)


@torch.no_grad()
def encode_docs(model, window_lists, side, pad_id, device, batch_windows=192):
    """-> (doc embeddings on CPU (fp16), valid mask) ; attn: (N, p) ; maxsim: (N, K, p)."""
    model.eval()
    embs, valids = [None] * len(window_lists), [None] * len(window_lists)
    order = sorted(range(len(window_lists)), key=lambda i: -len(window_lists[i]))
    s = 0
    while s < len(order):
        rows, total = [], 0
        while s < len(order) and (not rows or total + len(window_lists[order[s]]) <= batch_windows):
            rows.append(order[s])
            total += len(window_lists[order[s]])
            s += 1
        ids, mask, valid = make_batch([window_lists[r] for r in rows], pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            e = model.embed(ids, mask, valid, side)
        e = e.float().cpu()
        for j, r in enumerate(rows):
            embs[r], valids[r] = e[j], valid[j].cpu()
    if model.pooling == "maxsim":
        K = max(x.size(0) for x in embs)
        E = torch.zeros(len(embs), K, embs[0].size(-1))
        V = torch.zeros(len(embs), K, dtype=torch.bool)
        for i, (x, v) in enumerate(zip(embs, valids)):
            E[i, :x.size(0)] = x
            V[i, :v.size(0)] = v
        return E.half(), V
    return torch.stack(embs).half(), None


@torch.no_grad()
def dense_list_scores(model, q_windows, c_windows, lists, pad_id, device, q_rows=None):
    """Dense scores (cosine, before the temperature) for every (query, candidate in its list)."""
    q_rows = list(range(len(q_windows))) if q_rows is None else q_rows
    needed = sorted({int(j) for i in q_rows for j in lists[i]})
    Ec, Vc = encode_docs(model, [c_windows[j] for j in needed], "c", pad_id, device)
    Eq, Vq = encode_docs(model, [q_windows[i] for i in q_rows], "q", pad_id, device)
    pos = {j: r for r, j in enumerate(needed)}
    out = []
    for a, i in enumerate(q_rows):
        idx = torch.tensor([pos[int(j)] for j in lists[i]])
        q = Eq[a:a + 1].to(device).float()
        c = Ec[idx].to(device).float()
        if model.pooling == "maxsim":
            s = model.sim(q, c, Vq[a:a + 1].to(device), Vc[idx].to(device))
        else:
            s = model.sim(q, c)
        out.append(s[0].float().cpu().numpy())
    return np.stack(out)


# ============================================================================= training


def train(model, tr, dev, args, device, pad_id, teacher=None):
    """tr: {"queries": [{"key", "pos": [cand idx], "hn": [cand idx]}], "q_windows", "c_windows", "cand_keys"}
    dev: {"q_windows", "c_windows", "lists", "cand_keys", "gold"} (dense rerank of the lexical top-N)."""
    head_ids = {id(p) for p in model.head_parameters()}
    head = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    body = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head, "lr": args.head_lr}] + ([{"params": body, "lr": args.learning_rate}] if body else [])
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    budget = args.time_budget_min * 60
    est = [max(200, args.epochs * len(tr["queries"]) // args.batch_queries)]
    warm = 50
    sched = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min((s + 1) / warm, max(0.03, 1 - s / est[0])))
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    trainable = [p for p in model.parameters() if p.requires_grad]
    rng = random.Random(args.seed)
    eval_rows = sorted(rng.sample(range(len(dev["q_windows"])), min(args.eval_queries, len(dev["q_windows"]))))
    start, last_eval = time.time(), time.time()
    best, best_state, history = {"f1": -1.0}, None, []
    step, ooms, kd_used = 0, 0, 0
    queries = tr["queries"]

    def evaluate(tag):
        nonlocal best, best_state, last_eval
        S = dense_list_scores(model, dev["q_windows"], dev["c_windows"], dev["lists"], pad_id, device, eval_rows)
        ranked = [[dev["cand_keys"][dev["lists"][i][j]] for j in np.argsort(-S[a], kind="stable")[:TOP_K]]
                  for a, i in enumerate(eval_rows)]
        sc = micro_at_k(ranked, [dev["gold"][i] for i in eval_rows])
        row = {"tag": tag, "opt_steps": step, "minutes": round((time.time() - start) / 60, 2),
               "dense_rerank_f1@7": round(sc["f1"], 4), "logit_scale": round(float(model.scale()), 2),
               "loss": round(float(np.mean(losses[-100:])), 4) if losses else None,
               "kd": round(float(np.mean(kd_losses[-100:])), 4) if kd_losses else None}
        history.append(row)
        log(json.dumps(row))
        if sc["f1"] > best["f1"]:
            best = {**sc, "tag": tag}
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        model.train()
        last_eval = time.time()

    losses, kd_losses = [], []
    evaluate("start (no training)")
    epoch, done = 0, False
    while not done:
        epoch += 1
        order = list(range(len(queries)))
        random.Random(args.seed + epoch).shuffle(order)
        model.train()
        for s in range(0, len(order) - args.batch_queries + 1, args.batch_queries):
            rows = order[s:s + args.batch_queries]
            qw, cw, col_cand, targets, own, tlogits = [], [], [], [], [], []
            for i in rows:
                q = queries[i]
                ws = tr["q_windows"][i]
                pick = sorted(rng.sample(range(len(ws)), min(args.train_q_windows, len(ws))))
                qw.append([ws[t] for t in pick])
                pos = rng.choice(q["pos"])
                pool = [j for j in q["hn"][:args.hn_pool] if j not in q["pos_set"]]
                negs = rng.sample(pool, min(args.hard_negatives, len(pool)))
                while len(negs) < args.hard_negatives:
                    r = rng.randrange(len(tr["c_windows"]))
                    if r not in q["pos_set"]:
                        negs.append(r)
                lst = [pos] + negs
                targets.append(len(col_cand))
                own.append(list(range(len(col_cand), len(col_cand) + len(lst))))
                col_cand += lst
                for j in lst:
                    ws = tr["c_windows"][j]
                    pick = sorted(rng.sample(range(len(ws)), min(args.train_c_windows, len(ws))))
                    cw.append([ws[t] for t in pick])
                if teacher is not None:
                    tq = teacher.get(q["key"])
                    vals = None if tq is None else [tq.get(tr["cand_keys"][j]) for j in lst]
                    tlogits.append(vals if vals is not None and all(v is not None for v in vals) else None)
            try:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                    ids, mask, qvalid = make_batch(qw, pad_id, device)
                    Q = model.embed(ids, mask, qvalid, "q")
                    ids, mask, cvalid = make_batch(cw, pad_id, device)
                    C = model.embed(ids, mask, cvalid, "c")
                logits = model.sim(Q.float(), C.float(), qvalid, cvalid) * model.scale()
                false_neg = torch.zeros_like(logits, dtype=torch.bool)
                for r, i in enumerate(rows):
                    ps = queries[i]["pos_set"]
                    for col, j in enumerate(col_cand):
                        if col != targets[r] and j in ps:
                            false_neg[r, col] = True
                logits = logits.masked_fill(false_neg, -1e4)
                loss = F.cross_entropy(logits, torch.tensor(targets, device=device))
                if teacher is not None:
                    kd_terms = []
                    for r, tl in enumerate(tlogits):
                        if tl is None:
                            continue
                        st = logits[r, own[r]] / args.kd_temperature
                        tt = torch.tensor(tl, device=device, dtype=torch.float32) / args.kd_temperature
                        kd_terms.append(F.kl_div(F.log_softmax(st, -1), F.softmax(tt, -1), reduction="sum"))
                    if kd_terms:
                        kd = torch.stack(kd_terms).mean() * args.kd_temperature ** 2
                        kd_losses.append(float(kd))
                        kd_used += 1
                        loss = args.kd_alpha * loss + (1 - args.kd_alpha) * kd
                scaler.scale(loss / args.grad_accum).backward()
            except torch.cuda.OutOfMemoryError:
                ooms += 1
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                log(f"OOM #{ooms}: batch skipped")
                if ooms > 15:
                    raise RuntimeError("Too many OOMs: append --batch-queries 2 --train-c-windows 2 to the command")
                continue
            losses.append(float(loss))
            step += 1
            if step % args.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                sched.step()
            if step == 60:
                rate = step / (time.time() - start)
                est[0] = max(100, min(est[0] * 10, int(rate * budget * 0.95 / args.grad_accum)))
                log(f"~{rate:.2f} steps/s -> LR schedule over {est[0]} optimizer steps "
                    f"(~{est[0] * args.grad_accum * args.batch_queries / max(1, len(queries)):.1f} epochs over train queries)")
            if step % 100 == 0:
                log(f"epoch {epoch} step {step} loss {np.mean(losses[-100:]):.4f}"
                    + (f" kd {np.mean(kd_losses[-100:]):.4f}" if kd_losses else "")
                    + f" elapsed {(time.time() - start) / 60:.1f} min")
            if time.time() - start > budget:
                log("Time budget reached.")
                done = True
                break
            if time.time() - last_eval > args.eval_every_min * 60:
                evaluate(f"epoch{epoch}")
        if epoch >= args.epochs:
            done = True
    evaluate("final")
    model.load_state_dict(best_state)
    log(f"best checkpoint: {best.get('tag')} dense rerank F1@7 {best['f1']:.4f} (KD batches used: {kd_used})")
    return best, history, time.time() - start


# ============================================================================= fusion

def per_query_transform(S):
    mu, sd = S.mean(1, keepdims=True), S.std(1, keepdims=True) + 1e-6
    R = rank_of(S)
    return [(S - mu) / sd, S - S.max(1, keepdims=True), 1.0 / (1.0 + R)]


def fusion_matrix(lex, dense_list):
    """lex: (nq, N, F) ; dense_list: [(nq, N)] -> (nq, N, F') and feature names suffix."""
    parts = [lex]
    for S in dense_list:
        parts += [x[..., None] for x in per_query_transform(S)]
    if len(dense_list) > 1:
        Z = np.mean([per_query_transform(S)[0] for S in dense_list], 0)
        parts.append(Z[..., None])
    return np.concatenate(parts, -1).astype(np.float32)


def labels_for(split_lex):
    gold = split_lex["gold"]
    keys = split_lex["cand_keys"]
    return np.array([[keys[j] in set(g) for j in row] for row, g in zip(split_lex["lists"], gold)], dtype=np.int32)


def make_learner(kind, seed):
    if kind == "lgbm":
        import lightgbm as lgb
        return lgb.LGBMRanker(objective="lambdarank", n_estimators=400, learning_rate=0.03, num_leaves=15,
                              min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              reg_lambda=1.0, random_state=seed, verbose=-1, lambdarank_truncation_level=20)
    if kind == "hgb":
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=seed)
    if kind == "lr":
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000))
    raise ValueError(kind)


def fit_predict(kind, X, y, Xte, seed):
    nq, N, Fd = X.shape
    m = make_learner(kind, seed)
    if kind == "lgbm":
        m.fit(X.reshape(-1, Fd), y.reshape(-1), group=[N] * nq)
        return m.predict(Xte.reshape(-1, Fd)).reshape(Xte.shape[0], N), m
    m.fit(X.reshape(-1, Fd), y.reshape(-1))
    return m.predict_proba(Xte.reshape(-1, Fd))[:, 1].reshape(Xte.shape[0], N), m


def lgbm_available():
    try:
        import lightgbm  # noqa: F401
        return True
    except Exception:  # noqa: BLE001
        return False


def ranked_from(scores, split_lex, rows=None):
    rows = range(scores.shape[0]) if rows is None else rows
    keys = split_lex["cand_keys"]
    out = []
    for a, i in enumerate(rows):
        order = np.argsort(-scores[a], kind="stable")
        out.append([keys[split_lex["lists"][i][j]] for j in order[:TOP_K]])
    return out


def choose_fusion(dev_lex, test_lex, dense_dev, dense_test, args, n_dense_cols):
    """5-fold query-grouped cross-fit on dev -> pick a method among those that use the Phase 1 retriever."""
    y = labels_for(dev_lex)
    Xdev = fusion_matrix(dev_lex["features"], dense_dev)
    Xte = fusion_matrix(test_lex["features"], dense_test)
    nlex = dev_lex["features"].shape[-1]
    nq = Xdev.shape[0]
    gold = dev_lex["gold"]
    rng = np.random.RandomState(args.seed)
    perm = rng.permutation(nq)
    folds = [perm[f::5] for f in range(5)]
    kinds = (["lgbm"] if lgbm_available() else ["hgb"]) + ["lr"]
    methods = {}
    for k in kinds:
        methods[f"{k}_all"] = (k, slice(None))
        methods[f"{k}_lexical_only"] = (k, slice(0, nlex))
    report = {}
    oof = {}
    for name, (kind, cols) in methods.items():
        pred = np.zeros(y.shape, np.float32)
        for f in range(5):
            te = folds[f]
            tr_ = np.concatenate([folds[g] for g in range(5) if g != f])
            if y[tr_].sum() == 0:
                continue
            p, _ = fit_predict(kind, Xdev[tr_][..., cols], y[tr_], Xdev[te][..., cols], args.seed)
            pred[te] = p
        oof[name] = pred
        report[name] = {k: round(v, 4) for k, v in micro_at_k(ranked_from(pred, dev_lex), gold).items()}
    # simple no-learning baselines
    zd = np.mean([per_query_transform(S)[0] for S in dense_dev], 0)
    rrf_col = dev_lex["feature_names"].index("rrf_rr")
    report["dense_only"] = {k: round(v, 4) for k, v in micro_at_k(ranked_from(zd, dev_lex), gold).items()}
    rrf_dense = dev_lex["features"][..., rrf_col] + np.mean([per_query_transform(S)[2] for S in dense_dev], 0)
    report["rrf_lexical+dense"] = {k: round(v, 4) for k, v in micro_at_k(ranked_from(rrf_dense, dev_lex), gold).items()}
    report["rrf_lexical_only"] = {k: round(v, 4) for k, v in micro_at_k(
        ranked_from(dev_lex["features"][..., rrf_col], dev_lex), gold).items()}
    candidates = [m for m in report if m not in ("rrf_lexical_only",) and not m.endswith("lexical_only")]
    chosen = max(candidates, key=lambda m: report[m]["f1"])
    log("FUSION (5-fold cross-fitted dev micro F1/P/R@7):", json.dumps(report, indent=1))
    log(f"-> chosen: {chosen} (dev F1@7 {report[chosen]['f1']:.4f}; lexical-only best "
        f"{max(report[m]['f1'] for m in report if 'lexical' in m and 'dense' not in m):.4f})")
    importances = None
    if chosen in methods:
        kind, cols = methods[chosen]
        test_scores, model = fit_predict(kind, Xdev[..., cols], y, Xte[..., cols], args.seed)
        names = dev_lex["feature_names"] + [f"dense{d}_{t}" for d in range(len(dense_dev)) for t in ("z", "minus_top", "rr")] \
            + (["dense_mean_z"] if len(dense_dev) > 1 else [])
        try:
            if kind == "lr":
                coef = model[-1].coef_[0]
            else:
                coef = getattr(model, "feature_importances_", None)
            if coef is not None:
                importances = sorted(zip(names, map(float, coef)), key=lambda t: -abs(t[1]))[:25]
        except Exception:  # noqa: BLE001
            pass
        dev_scores = oof[chosen]
    elif chosen == "dense_only":
        test_scores = np.mean([per_query_transform(S)[0] for S in dense_test], 0)
        dev_scores = zd
    else:
        test_scores = test_lex["features"][..., rrf_col] + np.mean([per_query_transform(S)[2] for S in dense_test], 0)
        dev_scores = rrf_dense
    return chosen, report, test_scores, dev_scores, importances


def write_submission(test_lex, test_scores, test_queries_ids, out, name):
    keys = test_lex["cand_keys"]
    key_to_id = dict(zip(keys, test_lex["cand_ids"]))
    pool = set(keys)
    pred = {}
    for i, qid in enumerate(test_queries_ids):
        order = np.argsort(-test_scores[i], kind="stable")
        picked = []
        for j in order:
            k = keys[test_lex["lists"][i][j]]
            if k not in picked:
                picked.append(k)
            if len(picked) == TOP_K:
                break
        for j in test_lex["rrf_full_order"][i]:  # pad (never needed when top-N >= 7)
            if len(picked) == TOP_K:
                break
            if keys[j] not in picked and keys[j] != canon(qid):
                picked.append(keys[j])
        pred[str(qid)] = [str(key_to_id[k]) for k in picked]
    assert len(pred) == len(test_queries_ids)
    assert all(len(v) == TOP_K and len(set(v)) == TOP_K for v in pred.values())
    assert all(canon(c) in pool for v in pred.values() for c in v)
    path = out / "prediction.json"
    path.write_text(json.dumps(pred, indent=1), encoding="utf-8")
    archive = out / f"submission_{name}.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(archive) as z:
        assert z.namelist() == ["prediction.json"]
        assert len(json.loads(z.read("prediction.json"))) == len(test_queries_ids)
    log(f"VALIDATED prediction.json: {len(pred)} test queries x {TOP_K} unique candidates from the test pool")
    return archive


def analysis(dev_lex, dev_scores, dev_queries, report):
    gold = dev_lex["gold"]
    ranked = ranked_from(dev_scores, dev_lex)
    per = []
    for q, r, g in zip(dev_queries, ranked, gold):
        per.append({"id": q["id"], "gold": g, "top7": r, "hits": len(set(r) & set(g)), "words": q["words"],
                    "has_citation_markers": q["has_markers"]})
    words = np.array([p["words"] for p in per])
    cuts = np.quantile(words, [0.33, 0.66]).tolist() if len(words) > 3 else [0, 0]
    by_len = {}
    for name, sel in (("short", words <= cuts[0]), ("medium", (words > cuts[0]) & (words <= cuts[1])), ("long", words > cuts[1])):
        idx = np.nonzero(sel)[0]
        if len(idx):
            by_len[name] = {"queries": int(len(idx)), **{k: round(v, 4) for k, v in micro_at_k(
                [ranked[i] for i in idx], [gold[i] for i in idx]).items()}}
    by_gold = {}
    for b in sorted({min(len(g), 10) for g in gold}):
        idx = [i for i, g in enumerate(gold) if min(len(g), 10) == b]
        by_gold[f"{b}{'+' if b == 10 else ''}"] = {"queries": len(idx), **{k: round(v, 4) for k, v in micro_at_k(
            [ranked[i] for i in idx], [gold[i] for i in idx]).items()}}
    hits = Counter(p["hits"] for p in per)
    return per, {"f1_by_query_length": by_len, "length_cut_words": cuts, "f1_by_gold_count": by_gold,
                 "hits_in_top7_hist": dict(sorted(hits.items())), "queries_with_zero_hits": hits.get(0, 0)}


# ============================================================================= preparation (shared cache)

def prepare(args, tokenizer_path):
    key = hashlib.md5(json.dumps({"v": 3, "top_n": args.top_n, "hn_pool": args.hn_pool, "L": args.chunk_len,
                                  "K": args.max_windows, "qw": args.query_windows, "ctx": args.context_sentences,
                                  "head": args.query_head_windows, "words": args.max_words, "data": args.data_dir,
                                  "tok": hashlib.md5(Path(tokenizer_path).read_bytes()).hexdigest(),
                                  "limit": args.limit_queries}).encode()).hexdigest()[:10]
    cache_dir = Path(args.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"p7_cache_{key}.pkl"
    lock = cache_dir / f"p7_cache_{key}.lock"
    for _ in range(240):  # another run is building the same cache -> wait for it
        if path.exists() or not lock.exists():
            break
        time.sleep(15)
    if path.exists():
        log(f"Using cached lexical stage + windows: {path}")
        with open(path, "rb") as f:
            return pickle.load(f)
    lock.write_text(str(os.getpid()))
    try:
        t = time.time()
        data, leak, stats = load_all(args.data_dir, args.seed)
        if args.limit_queries:
            for s in ("train", "dev"):
                data[s]["q"] = data[s]["q"][:args.limit_queries]
        popularity = Counter(k for q in data["train"]["q"] for k in q["rel"])
        lex = {}
        lex["dev"] = lexical_split("dev", data["dev"]["q"], data["dev"]["c"], args, None, popularity)
        lex["test"] = lexical_split("test", data["test"]["q"], data["test"]["c"], args, lex["dev"]["params"], popularity)
        lex["train"] = lexical_split("train", data["train"]["q"], data["train"]["c"], args, lex["dev"]["params"])
        from tokenizers import Tokenizer
        tok = Tokenizer.from_file(str(tokenizer_path))
        args.eos_id = tok.token_to_id("<eos>")
        windows = {}
        for s in SPLITS:
            tw = time.time()
            windows[s] = {"q": build_windows(data[s]["q"], tok, args, True), "c": build_windows(data[s]["c"], tok, args, False)}
            log(f"[{s}] windows built in {time.time() - tw:.0f}s; windows/query "
                f"{np.mean([len(w) for w in windows[s]['q']]):.1f}, windows/candidate {np.mean([len(w) for w in windows[s]['c']]):.1f}")
        slim = {s: {"q": [{"id": q["id"], "key": q["key"], "rel": q["rel"], "words": len(q["text"].split()),
                           "has_markers": any(r.search(q["text"]) for r in CITE_RES)} for q in data[s]["q"]],
                    "c": [{"id": c["id"], "key": c["key"]} for c in data[s]["c"]]} for s in SPLITS}
        bundle = {"lex": lex, "windows": windows, "data": slim, "leak": leak, "stats": stats, "key": key,
                  "minutes": (time.time() - t) / 60}
        with open(str(path) + ".tmp", "wb") as f:
            pickle.dump(bundle, f, protocol=4)
        os.replace(str(path) + ".tmp", path)
        log(f"Prepared cache {path} in {(time.time() - t) / 60:.1f} min")
        return bundle
    finally:
        if lock.exists():
            lock.unlink()


# ============================================================================= main

def run(args):
    t0 = time.time()
    from phase3_rr import locate_phase1_files, seed_everything
    seed_everything(args.seed)
    np.random.seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    name = args.run_name or f"p7_{args.model}_{args.method}"
    out = Path(args.output_dir or f"/kaggle/working/{name}")
    out.mkdir(parents=True, exist_ok=True)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    checkpoint, tokenizer_path, artifact = locate_phase1_files(args.model, args.artifact_zip, out / "phase1")
    bundle = prepare(args, tokenizer_path)
    if args.prepare_only:
        log("prepare-only: done")
        return
    lex, win, data = bundle["lex"], bundle["windows"], bundle["data"]
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(str(tokenizer_path))
    pad_id, args.eos_id = tok.token_to_id("<pad>"), tok.token_to_id("<eos>")

    # ---------------- train structures
    tkeys = lex["train"]["cand_keys"]
    tindex = {k: j for j, k in enumerate(tkeys)}
    tr_queries, tr_qwin = [], []
    for i, q in enumerate(data["train"]["q"]):
        pos = [tindex[k] for k in q["rel"] if k in tindex and k != q["key"]]
        if not pos:
            continue
        ps = set(pos)
        hn = [int(j) for j in lex["train"]["lists"][i] if int(j) not in ps]
        tr_queries.append({"key": q["key"], "pos": pos, "pos_set": ps, "hn": hn})
        tr_qwin.append(win["train"]["q"][i])
    tr = {"queries": tr_queries, "q_windows": tr_qwin, "c_windows": win["train"]["c"], "cand_keys": tkeys}
    dev = {"q_windows": win["dev"]["q"], "c_windows": win["dev"]["c"], "lists": lex["dev"]["lists"],
           "cand_keys": lex["dev"]["cand_keys"], "gold": lex["dev"]["gold"]}
    log(f"train queries with >=1 positive in the train pool: {len(tr_queries)}; positives "
        f"{sum(len(q['pos']) for q in tr_queries)}; dev queries {len(dev['q_windows'])}")

    teacher = None
    if args.method == "distill":
        teacher = torch.load(args.teacher_scores, map_location="cpu", weights_only=False)
        log(f"KD: teacher {teacher.get('_teacher')} scores for {len(teacher) - 1} train queries")
        teacher = {k: v for k, v in teacher.items() if not k.startswith("_")}

    # ---------------- model
    from phase2_lner import load_phase1, make_student
    from phase4_cjpe import apply_method
    backbone, phase1_cfg, phase1_saved = load_phase1(checkpoint, args.model)
    phase1_params = sum(p.numel() for p in backbone.parameters())
    if args.method == "distill":
        if args.model != "decoder":
            raise ValueError("The distillation student recipe uses the decoder Phase 1 model (cut to fewer layers)")
        backbone = make_student(backbone, layers=args.student_layers)
    model = PCRModel(backbone, args.model, args.proj_dim, args.pooling, args.grad_ckpt)
    apply_method(model, args.model, args.method, args.rank, args.unfreeze_layers)
    model.to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    summary = {"run": name, "model": args.model, "method": args.method, "pooling": args.pooling,
               "phase1_artifact": artifact, "phase1_parameters": phase1_params, "parameters": total,
               "trainable_parameters": trainable, "trainable_fraction": trainable / total,
               "leakage_check": bundle["leak"], "data_stats": bundle["stats"], "lexical_dev": lex["dev"]["report"],
               "bm25_params": lex["dev"]["params"], "device": str(device), "config": vars(args)}
    log(json.dumps({k: v for k, v in summary.items() if k not in ("data_stats", "config")}, indent=1, default=str))

    best, history, seconds = train(model, tr, dev, args, device, pad_id, teacher)
    (out / "training_log.json").write_text(json.dumps(history, indent=1), encoding="utf-8")

    # ---------------- dense scores for the full dev / test candidate lists
    inf_t = time.time()
    dense_dev = dense_list_scores(model, win["dev"]["q"], win["dev"]["c"], lex["dev"]["lists"], pad_id, device)
    dev_sec = time.time() - inf_t
    inf_t = time.time()
    dense_test = dense_list_scores(model, win["test"]["q"], win["test"]["c"], lex["test"]["lists"], pad_id, device)
    test_sec = time.time() - inf_t
    np.savez_compressed(out / "dense_scores.npz", dev=dense_dev, test=dense_test, cache_key=bundle["key"],
                        dev_lists=lex["dev"]["lists"], test_lists=lex["test"]["lists"])

    if args.export_teacher:
        ex = time.time()
        lists = []
        for q in tr["queries"]:
            lists.append(list(dict.fromkeys(q["pos"] + q["hn"][:args.hn_pool])))
        maxlen = max(len(x) for x in lists)
        padded = [x + [x[0]] * (maxlen - len(x)) for x in lists]
        S = dense_list_scores(model, tr["q_windows"], tr["c_windows"], padded, pad_id, device) * float(model.scale())
        scores = {"_teacher": name}
        for q, row, x in zip(tr["queries"], S, lists):
            scores[q["key"]] = {tkeys[j]: float(v) for j, v in zip(x, row[:len(x)])}
        torch.save(scores, out / "teacher_scores.pt")
        log(f"Saved teacher scores for {len(lists)} train queries in {(time.time() - ex) / 60:.1f} min")

    chosen, report, test_scores, dev_scores, importances = choose_fusion(lex["dev"], lex["test"], [dense_dev], [dense_test], args, 1)
    archive = write_submission(lex["test"], test_scores, [q["id"] for q in data["test"]["q"]], out, name)
    try:
        per, details = analysis(lex["dev"], dev_scores, data["dev"]["q"], report)
        (out / "dev_predictions.json").write_text(json.dumps(per), encoding="utf-8")
    except Exception as error:  # noqa: BLE001
        import traceback
        traceback.print_exc()
        details = {"analysis_error": repr(error)}
    if args.save_checkpoint:
        torch.save({"model": {k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()},
                    "phase1_config": asdict(phase1_cfg), "run_config": vars(args)}, out / "best_fp16.pt")
    manifest = {**summary, "selected_checkpoint": best.get("tag"), "dense_rerank_dev_f1@7_eval_subset": best["f1"],
                "fusion_chosen": chosen, "dev_micro_f1@7_crossfit": report[chosen]["f1"],
                "dev_micro_precision@7_crossfit": report[chosen]["precision"],
                "dev_micro_recall@7_crossfit": report[chosen]["recall"], "fusion_report": report,
                "fusion_top_features": importances, "analysis": details, "train_minutes": seconds / 60,
                "dev_inference_seconds": dev_sec, "test_inference_seconds": test_sec,
                "test_queries_per_second": len(data["test"]["q"]) / max(1e-6, test_sec),
                "total_minutes": (time.time() - t0) / 60,
                "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None,
                "phase1_steps": phase1_saved.get("total_steps"), "submission": str(archive)}
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1, default=str), encoding="utf-8")
    log(json.dumps({k: manifest[k] for k in ("run", "method", "trainable_parameters", "fusion_chosen",
                                             "dev_micro_f1@7_crossfit", "dense_rerank_dev_f1@7_eval_subset",
                                             "total_minutes")}, indent=1))
    log("SUBMISSION ZIP:", archive)


def fuse_runs(args):
    """Ensemble: lexical features + the dense scores of several finished runs -> one fusion model."""
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    caches = sorted(Path(args.cache_dir).glob("p7_cache_*.pkl"))
    runs = [Path(r) for r in args.fuse_runs if (Path(r) / "dense_scores.npz").exists()]
    if len(runs) < 2:
        raise RuntimeError(f"Need >= 2 finished runs with dense_scores.npz, found {runs}")
    loaded = [np.load(r / "dense_scores.npz") for r in runs]
    key = str(loaded[0]["cache_key"])
    cache = [c for c in caches if key in c.name]
    if not cache:
        raise FileNotFoundError(f"cache {key} not found in {args.cache_dir}")
    with open(cache[0], "rb") as f:
        bundle = pickle.load(f)
    lex, data = bundle["lex"], bundle["data"]
    for r, d in zip(runs, loaded):
        if str(d["cache_key"]) != key:
            raise RuntimeError(f"{r} used a different candidate cache; cannot fuse")
    chosen, report, test_scores, dev_scores, imp = choose_fusion(lex["dev"], lex["test"], [d["dev"] for d in loaded],
                                                                 [d["test"] for d in loaded], args, len(loaded))
    name = args.run_name or "p7_ENSEMBLE"
    archive = write_submission(lex["test"], test_scores, [q["id"] for q in data["test"]["q"]], out, name)
    (out / "manifest.json").write_text(json.dumps({"run": name, "method": "ensemble", "submission": str(archive), "members": [str(r) for r in runs], "fusion_chosen": chosen,
                                                   "dev_micro_f1@7_crossfit": report[chosen]["f1"], "fusion_report": report,
                                                   "fusion_top_features": imp}, indent=1, default=str), encoding="utf-8")
    log("SUBMISSION ZIP:", archive)


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="efficient", choices=("decoder", "encoder_decoder", "hybrid", "efficient"))
    p.add_argument("--method", default="full", choices=("full", "lorma", "lora", "ia3", "partial", "bitfit", "distill"))
    p.add_argument("--pooling", default="attn", choices=("attn", "maxsim"))
    p.add_argument("--proj-dim", type=int, default=256)
    p.add_argument("--chunk-len", type=int, default=256)
    p.add_argument("--max-windows", type=int, default=8)
    p.add_argument("--max-words", type=int, default=15000)
    p.add_argument("--query-windows", default="citation", choices=("citation", "spread"))
    p.add_argument("--context-sentences", type=int, default=2)
    p.add_argument("--query-head-windows", type=int, default=1)
    p.add_argument("--train-q-windows", type=int, default=4)
    p.add_argument("--train-c-windows", type=int, default=3)
    p.add_argument("--batch-queries", type=int, default=4)
    p.add_argument("--hard-negatives", type=int, default=3)
    p.add_argument("--hn-pool", type=int, default=50)
    p.add_argument("--top-n", type=int, default=100)
    p.add_argument("--grad-accum", type=int, default=2)
    p.add_argument("--epochs", type=int, default=100)
    p.add_argument("--learning-rate", type=float, default=3e-5)
    p.add_argument("--head-lr", type=float, default=1e-3)
    p.add_argument("--rank", type=int, default=16)
    p.add_argument("--unfreeze-layers", type=int, default=4)
    p.add_argument("--student-layers", type=int, default=8)
    p.add_argument("--teacher-scores")
    p.add_argument("--kd-alpha", type=float, default=0.5)
    p.add_argument("--kd-temperature", type=float, default=2.0)
    p.add_argument("--export-teacher", action="store_true")
    p.add_argument("--grad-ckpt", action="store_true")
    p.add_argument("--save-checkpoint", action="store_true")
    p.add_argument("--time-budget-min", type=float, default=60.0)
    p.add_argument("--eval-every-min", type=float, default=15.0)
    p.add_argument("--eval-queries", type=int, default=250)
    p.add_argument("--limit-queries", type=int, default=0)
    p.add_argument("--cpu-jobs", type=int, default=4)
    p.add_argument("--cache-dir", default="/kaggle/working/p7_cache")
    p.add_argument("--prepare-only", action="store_true")
    p.add_argument("--fuse-runs", nargs="*")
    p.add_argument("--artifact-zip")
    p.add_argument("--data-dir")
    p.add_argument("--output-dir")
    p.add_argument("--run-name")
    p.add_argument("--seed", type=int, default=230338)
    return p.parse_args(argv)


if __name__ == "__main__":
    a = parse_args()
    if a.fuse_runs:
        fuse_runs(a)
    else:
        run(a)


## 1. Data + lexical stage + Phase 1 windows (CPU, shared by both runs)
Look for `Phase 7 files on HF`, `LEAKAGE CHECK`, `DATA [train]/[dev]/[test]`, the BM25 grid and `lexical dev/train metrics` (the lexical baseline).


In [ ]:
!cd /kaggle/working && python phase7_pcr.py --prepare-only --model efficient --output-dir /kaggle/working/p7_prep 2>&1 | tee /kaggle/working/p7_prepare.log | grep -v '^\s*$' | tail -n 40


## 2. Two runs in parallel (one per GPU)


In [ ]:
%%bash
set +e
cd /kaggle/working
NGPU=$(nvidia-smi -L | wc -l); echo "GPUs: $NGPU"
if [ "$NGPU" -ge 2 ]; then
  ( CUDA_VISIBLE_DEVICES=0 python phase7_pcr.py --model efficient --method lorma --rank 32 --learning-rate 5e-4 --run-name p7_B_efficient_lorma --grad-ckpt --batch-queries 8 --hard-negatives 3 --train-q-windows 4 --train-c-windows 3 --eval-every-min 15 --save-checkpoint --time-budget-min 135 --output-dir /kaggle/working/p7_B_efficient_lorma > p7_B_efficient_lorma.log 2>&1 ) &
  sleep 60
  ( CUDA_VISIBLE_DEVICES=1 python phase7_pcr.py --model efficient --method ia3 --learning-rate 3e-3 --run-name p7_F_efficient_ia3 --grad-ckpt --batch-queries 8 --hard-negatives 3 --train-q-windows 4 --train-c-windows 3 --eval-every-min 15 --save-checkpoint --time-budget-min 135 --output-dir /kaggle/working/p7_F_efficient_ia3 > p7_F_efficient_ia3.log 2>&1 ) &
  wait
else
  echo 'Only one GPU: running everything one after the other at ~45% budget'
  CUDA_VISIBLE_DEVICES=0 python phase7_pcr.py --model efficient --method lorma --rank 32 --learning-rate 5e-4 --run-name p7_B_efficient_lorma --grad-ckpt --batch-queries 8 --hard-negatives 3 --train-q-windows 4 --train-c-windows 3 --eval-every-min 15 --save-checkpoint --time-budget-min 61 --output-dir /kaggle/working/p7_B_efficient_lorma > p7_B_efficient_lorma.log 2>&1 ; CUDA_VISIBLE_DEVICES=0 python phase7_pcr.py --model efficient --method ia3 --learning-rate 3e-3 --run-name p7_F_efficient_ia3 --grad-ckpt --batch-queries 8 --hard-negatives 3 --train-q-windows 4 --train-c-windows 3 --eval-every-min 15 --save-checkpoint --time-budget-min 61 --output-dir /kaggle/working/p7_F_efficient_ia3 > p7_F_efficient_ia3.log 2>&1
fi
for f in p7_B_efficient_lorma.log p7_F_efficient_ia3.log; do echo "===== $f ====="; grep -E 'LEAKAGE|DATA \[|steps/s|best checkpoint|chosen|VALIDATED|SUBMISSION|Error|error|OOM' $f | tail -n 20; tail -n 4 $f; done
exit 0


## 3. Ensemble of this notebook's runs (dense scores of all members in one fusion ranker)


In [ ]:
!cd /kaggle/working && python phase7_pcr.py --fuse-runs /kaggle/working/p7_B_efficient_lorma /kaggle/working/p7_F_efficient_ia3 --output-dir /kaggle/working/p7_ENS_NB2 --run-name p7_ENS_NB2 2>&1 | tee p7_ENS_NB2.log | tail -n 30


## 4. Results + submission ZIPs
Upload order is in the README. `dev F1@7 (crossfit)` is the honest dev estimate; `dense-only` is the Phase 1 retriever alone (rerank of the lexical top-N).


In [ ]:
import json, shutil, glob
from pathlib import Path
from IPython.display import FileLink, display
Path('/kaggle/working/SUBMIT').mkdir(exist_ok=True)
rows = []
for m in sorted(glob.glob('/kaggle/working/p7_*/manifest.json')):
    m = json.load(open(m))
    z = Path(m.get('submission', '/nonexistent'))
    if z.exists():
        shutil.copy(z, '/kaggle/working/SUBMIT/')
    rows.append((m['run'], m.get('method'), m.get('fusion_chosen'), m.get('dev_micro_f1@7_crossfit'),
                 m.get('dense_rerank_dev_f1@7_eval_subset'), m.get('trainable_parameters')))
print(f"{'run':28s} {'method':8s} {'fusion':18s} {'dev F1@7 (crossfit)':>20s} {'dense-only':>11s} {'trainable':>12s}")
for r in sorted(rows, key=lambda r: -(r[3] or 0)):
    print(f"{r[0]:28s} {str(r[1]):8s} {str(r[2]):18s} {r[3] or 0:20.4f} {r[4] or 0:11.4f} {r[5] or 0:12,}")
for z in sorted(glob.glob('/kaggle/working/SUBMIT/*.zip')):
    display(FileLink(z.replace('/kaggle/working/', '')))
missing = [l for l in glob.glob('/kaggle/working/p7_*.log') if 'SUBMISSION ZIP' not in open(l).read()]
print('runs WITHOUT a submission (read their log):', missing or 'none')


In [ ]:
!cd /kaggle/working && zip -qr phase7_outputs.zip p7_* *.log -x '*/phase1/*' '*/best_fp16.pt' 'p7_cache/*' && ls -la phase7_outputs.zip SUBMIT
!ls -la /kaggle/working/p7_*/best_fp16.pt
